In [1]:
# Cell 1: Initialize 30D Corsi research branch and load frozen Excel baseline benchmark
#
# Objective:
#   Start a clean 30D Corsi/HAR research notebook.
#
# Scope:
#   - Create new vrp_30d_corsi_v1 folders
#   - Load the frozen 30D Excel-style baseline signal tape
#   - Load baseline validation/performance/year/worst-trade outputs
#   - Define the benchmark every Corsi denominator must beat
#   - Save benchmark and success criteria into the new audit folder
#
# Explicitly NOT doing in this cell:
#   - No Corsi/HAR feature construction
#   - No forecast target construction
#   - No model fitting
#   - No threshold optimization
#   - No cross-tenor work
#   - No Secondary signal
#   - No sizing

from pathlib import Path
import json
import numpy as np
import pandas as pd


# =============================================================================
# 0. Paths and branch setup
# =============================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

BASELINE_BRANCH = "vrp_30d_baseline_v1"
CORSI_BRANCH = "vrp_30d_corsi_v1"

BASELINE_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / BASELINE_BRANCH
BASELINE_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / BASELINE_BRANCH

CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / CORSI_BRANCH
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / CORSI_BRANCH
CORSI_NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / CORSI_BRANCH

CORSI_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CORSI_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
CORSI_NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

print("Cell 1: Initialize 30D Corsi research branch")
print(f"Project root:              {PROJECT_ROOT}")
print(f"Baseline processed dir:    {BASELINE_PROCESSED_DIR}")
print(f"Baseline audit dir:        {BASELINE_AUDIT_DIR}")
print(f"Corsi processed dir:       {CORSI_PROCESSED_DIR}")
print(f"Corsi audit dir:           {CORSI_AUDIT_DIR}")
print(f"Corsi notebook dir:        {CORSI_NOTEBOOK_DIR}")
print(f"Run timestamp:             {RUN_TIMESTAMP}")


# =============================================================================
# 1. Helpers
# =============================================================================

def latest_file(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)

    if not matches:
        raise FileNotFoundError(f"No files found in {directory} matching pattern: {pattern}")

    return matches[0]


def as_float(value):
    if pd.isna(value):
        return np.nan
    return float(value)


def as_int(value):
    if pd.isna(value):
        return np.nan
    return int(value)


def pass_fail(condition):
    return "PASS" if bool(condition) else "FAIL"


# =============================================================================
# 2. Locate frozen baseline outputs
# =============================================================================

baseline_signal_path = latest_file(
    BASELINE_PROCESSED_DIR,
    "00_excel_30d_vrp_baseline_signal_tape_*.parquet",
)

baseline_validation_path = latest_file(
    BASELINE_AUDIT_DIR,
    "00_excel_30d_vrp_baseline_validation_*.csv",
)

baseline_performance_path = latest_file(
    BASELINE_AUDIT_DIR,
    "00_excel_30d_vrp_baseline_performance_summary_*.csv",
)

baseline_year_path = latest_file(
    BASELINE_AUDIT_DIR,
    "00_excel_30d_vrp_baseline_year_summary_*.csv",
)

baseline_worst_20_path = latest_file(
    BASELINE_AUDIT_DIR,
    "00_excel_30d_vrp_baseline_worst_20_trades_*.csv",
)

baseline_manifest_path = latest_file(
    BASELINE_AUDIT_DIR,
    "00_excel_30d_vrp_baseline_source_manifest_*.csv",
)

baseline_files = pd.DataFrame([
    {"role": "baseline_signal_tape", "path": str(baseline_signal_path)},
    {"role": "baseline_validation", "path": str(baseline_validation_path)},
    {"role": "baseline_performance_summary", "path": str(baseline_performance_path)},
    {"role": "baseline_year_summary", "path": str(baseline_year_path)},
    {"role": "baseline_worst_20_trades", "path": str(baseline_worst_20_path)},
    {"role": "baseline_source_manifest", "path": str(baseline_manifest_path)},
])

print()
print("Frozen baseline files loaded:")
display(baseline_files)


# =============================================================================
# 3. Load frozen baseline outputs
# =============================================================================

baseline_signal = pd.read_parquet(baseline_signal_path)
baseline_validation = pd.read_csv(baseline_validation_path)
baseline_performance = pd.read_csv(baseline_performance_path)
baseline_year = pd.read_csv(baseline_year_path)
baseline_worst_20 = pd.read_csv(baseline_worst_20_path)
baseline_manifest = pd.read_csv(baseline_manifest_path)

# Normalize dates.
for df in [baseline_signal, baseline_year, baseline_worst_20]:
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Validation gate from frozen baseline.
failed_validation = baseline_validation.loc[
    baseline_validation["status"].astype(str).str.upper().ne("PASS")
].copy()

print()
print("Frozen baseline validation:")
display(baseline_validation)

if len(failed_validation) > 0:
    print()
    print("FAILED BASELINE VALIDATION CHECKS:")
    display(failed_validation)
    raise ValueError("Frozen baseline validation is not clean. Stop before starting Corsi research.")


# =============================================================================
# 4. Extract baseline benchmark metrics
# =============================================================================

if len(baseline_performance) != 1:
    raise ValueError(
        f"Expected exactly one row in baseline performance summary, found {len(baseline_performance)}."
    )

perf = baseline_performance.iloc[0].copy()

baseline_core_trades = as_int(perf["core_trades"])
baseline_core_frequency = as_float(perf["core_frequency"])
baseline_core_win_rate = as_float(perf["core_win_rate"])
baseline_avg_pnl_per_trade = as_float(perf["core_avg_pnl_per_trade"])
baseline_avg_pnl_per_day = as_float(perf["core_avg_pnl_per_day"])
baseline_exposure_weighted_pnl_per_day = as_float(perf["core_exposure_day_weighted_pnl_per_day"])
baseline_total_pnl = as_float(perf["core_total_pnl"])
baseline_profit_factor = as_float(perf["core_profit_factor"])
baseline_worst_trade = as_float(perf["core_worst_trade"])
baseline_best_trade = as_float(perf["core_best_trade"])
baseline_avg_actual_dte = as_float(perf["core_avg_actual_dte"])

baseline_2025 = baseline_year.loc[baseline_year["year"].eq(2025)].copy()

if len(baseline_2025) != 1:
    raise ValueError(f"Expected exactly one 2025 row in baseline year summary, found {len(baseline_2025)}.")

baseline_2025_row = baseline_2025.iloc[0].copy()

baseline_2025_trades = as_int(baseline_2025_row["core_trades"])
baseline_2025_frequency = as_float(baseline_2025_row["core_frequency"])
baseline_2025_win_rate = as_float(baseline_2025_row["core_win_rate"])
baseline_2025_total_pnl = as_float(baseline_2025_row["core_total_pnl"])
baseline_2025_worst_trade = as_float(baseline_2025_row["core_worst_trade"])

baseline_benchmark = pd.DataFrame([{
    "benchmark_name": "excel_30d_vrp_baseline",
    "trade_proxy": "short_naked_atm_put",
    "tenor": 30,
    "vrp_formula": "log(implied_variance_30d / rv21d_variance)",
    "rv21d_formula": "(STDEV.S(21D close-to-close log returns) * sqrt(252))^2",
    "zscore_convention": "prior_only",
    "core_trades": baseline_core_trades,
    "core_frequency": baseline_core_frequency,
    "core_win_rate": baseline_core_win_rate,
    "core_avg_pnl_per_trade": baseline_avg_pnl_per_trade,
    "core_avg_pnl_per_day": baseline_avg_pnl_per_day,
    "core_exposure_weighted_pnl_per_day": baseline_exposure_weighted_pnl_per_day,
    "core_total_pnl": baseline_total_pnl,
    "core_profit_factor": baseline_profit_factor,
    "core_worst_trade": baseline_worst_trade,
    "core_best_trade": baseline_best_trade,
    "core_avg_actual_dte": baseline_avg_actual_dte,
    "core_2025_trades": baseline_2025_trades,
    "core_2025_frequency": baseline_2025_frequency,
    "core_2025_win_rate": baseline_2025_win_rate,
    "core_2025_total_pnl": baseline_2025_total_pnl,
    "core_2025_worst_trade": baseline_2025_worst_trade,
    "baseline_signal_path": str(baseline_signal_path),
    "baseline_performance_path": str(baseline_performance_path),
    "baseline_year_path": str(baseline_year_path),
}])

print()
print("Frozen 30D Excel-style baseline benchmark:")
display(baseline_benchmark.T)


# =============================================================================
# 5. Define Corsi success criteria
# =============================================================================

# Primary goal from the research reset:
#   Build a robust and defensible 30D Corsi/HAR denominator that beats the baseline
#   in frequency and win percentage without adding tail risk.
#
# Exact evaluation rule for a candidate model:
#   Required:
#       candidate_frequency > baseline_frequency
#       candidate_win_rate >= baseline_win_rate
#       candidate_worst_trade >= baseline_worst_trade
#       candidate_2025_total_pnl >= baseline_2025_total_pnl
#       candidate_2025_worst_trade >= baseline_2025_worst_trade
#
#   Additional diagnostics:
#       profit factor should not be materially worse
#       large-loss count should not increase
#       selected-trade drawdown should not worsen
#       worst 20 trades should be reviewed manually

success_criteria = pd.DataFrame([
    {
        "criterion_group": "primary",
        "metric": "core_frequency",
        "baseline_value": baseline_core_frequency,
        "candidate_requirement": "strictly_greater_than_baseline",
        "pass_rule": "candidate_core_frequency > baseline_core_frequency",
    },
    {
        "criterion_group": "primary",
        "metric": "core_win_rate",
        "baseline_value": baseline_core_win_rate,
        "candidate_requirement": "greater_than_or_equal_to_baseline",
        "pass_rule": "candidate_core_win_rate >= baseline_core_win_rate",
    },
    {
        "criterion_group": "tail",
        "metric": "core_worst_trade",
        "baseline_value": baseline_worst_trade,
        "candidate_requirement": "no_worse_than_baseline",
        "pass_rule": "candidate_core_worst_trade >= baseline_core_worst_trade",
    },
    {
        "criterion_group": "tail",
        "metric": "core_2025_total_pnl",
        "baseline_value": baseline_2025_total_pnl,
        "candidate_requirement": "no_worse_than_baseline",
        "pass_rule": "candidate_core_2025_total_pnl >= baseline_core_2025_total_pnl",
    },
    {
        "criterion_group": "tail",
        "metric": "core_2025_worst_trade",
        "baseline_value": baseline_2025_worst_trade,
        "candidate_requirement": "no_worse_than_baseline",
        "pass_rule": "candidate_core_2025_worst_trade >= baseline_core_2025_worst_trade",
    },
    {
        "criterion_group": "diagnostic",
        "metric": "core_profit_factor",
        "baseline_value": baseline_profit_factor,
        "candidate_requirement": "not_materially_worse",
        "pass_rule": "review; do not accept materially worse profit factor",
    },
    {
        "criterion_group": "diagnostic",
        "metric": "large_loss_count",
        "baseline_value": np.nan,
        "candidate_requirement": "not_worse",
        "pass_rule": "candidate large-loss counts below -25k/-50k/-100k should not exceed baseline",
    },
    {
        "criterion_group": "diagnostic",
        "metric": "selected_trade_drawdown",
        "baseline_value": np.nan,
        "candidate_requirement": "not_worse",
        "pass_rule": "candidate selected-trade drawdown should not exceed baseline",
    },
])

print()
print("Corsi denominator success criteria:")
display(success_criteria)


# =============================================================================
# 6. Baseline sanity summaries for reference
# =============================================================================

baseline_signal_summary = pd.DataFrame([{
    "rows": len(baseline_signal),
    "first_date": baseline_signal["date"].min(),
    "last_date": baseline_signal["date"].max(),
    "feature_complete_rows": int(baseline_signal["is_baseline_feature_complete"].sum())
        if "is_baseline_feature_complete" in baseline_signal.columns else np.nan,
    "trade_eligible_rows": int(baseline_signal["is_baseline_trade_eligible"].sum())
        if "is_baseline_trade_eligible" in baseline_signal.columns else np.nan,
    "core_signal_rows": int(baseline_signal["core_signal"].sum())
        if "core_signal" in baseline_signal.columns else np.nan,
    "outcome_non_null": int(baseline_signal["outcome_value"].notna().sum())
        if "outcome_value" in baseline_signal.columns else np.nan,
    "actual_dte_non_null": int(baseline_signal["actual_dte"].notna().sum())
        if "actual_dte" in baseline_signal.columns else np.nan,
}])

print()
print("Baseline signal tape sanity summary:")
display(baseline_signal_summary)

print()
print("Baseline year summary:")
display(baseline_year)

print()
print("Baseline worst 20 trades:")
display(baseline_worst_20)


# =============================================================================
# 7. Save benchmark and handoff files to new Corsi branch
# =============================================================================

safe_start = pd.to_datetime(baseline_signal["date"]).min().strftime("%Y%m%d")
safe_end = pd.to_datetime(baseline_signal["date"]).max().strftime("%Y%m%d")

benchmark_path = (
    CORSI_AUDIT_DIR
    / f"01_30d_corsi_baseline_benchmark_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

success_criteria_path = (
    CORSI_AUDIT_DIR
    / f"01_30d_corsi_success_criteria_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

baseline_files_path = (
    CORSI_AUDIT_DIR
    / f"01_30d_corsi_baseline_file_manifest_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

baseline_signal_summary_path = (
    CORSI_AUDIT_DIR
    / f"01_30d_corsi_baseline_signal_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

baseline_benchmark.to_csv(benchmark_path, index=False)
success_criteria.to_csv(success_criteria_path, index=False)
baseline_files.to_csv(baseline_files_path, index=False)
baseline_signal_summary.to_csv(baseline_signal_summary_path, index=False)

print()
print("Cell 1 outputs saved:")
print(f"Baseline benchmark:       {benchmark_path}")
print(f"Success criteria:         {success_criteria_path}")
print(f"Baseline file manifest:   {baseline_files_path}")
print(f"Signal summary:           {baseline_signal_summary_path}")

print()
print("Cell 1 complete.")
print()
print("Next proposed cell:")
print("Cell 2: Build close-to-close 30D Corsi/HAR feature library and 21-trading-day forward realized variance target.")
print("No model fitting until Cell 2 feature/target construction is reviewed.")

Cell 1: Initialize 30D Corsi research branch
Project root:              C:\Users\patri\vrp_project
Baseline processed dir:    C:\Users\patri\vrp_project\data\processed\vrp_30d_baseline_v1
Baseline audit dir:        C:\Users\patri\vrp_project\data\audit\vrp_30d_baseline_v1
Corsi processed dir:       C:\Users\patri\vrp_project\data\processed\vrp_30d_corsi_v1
Corsi audit dir:           C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1
Corsi notebook dir:        C:\Users\patri\vrp_project\notebooks\vrp_30d_corsi_v1
Run timestamp:             20260704_125621

Frozen baseline files loaded:


,role,path
0,baseline_signal_tape,C:\Users\patri\vrp_project\data\processed\vrp_...
1,baseline_validation,C:\Users\patri\vrp_project\data\audit\vrp_30d_...
2,baseline_performance_summary,C:\Users\patri\vrp_project\data\audit\vrp_30d_...
3,baseline_year_summary,C:\Users\patri\vrp_project\data\audit\vrp_30d_...
4,baseline_worst_20_trades,C:\Users\patri\vrp_project\data\audit\vrp_30d_...
5,baseline_source_manifest,C:\Users\patri\vrp_project\data\audit\vrp_30d_...



Frozen baseline validation:


,check,status,details
0,30D implied vol source found,PASS,path=C:\Users\patri\vrp_project\data\processed...
1,Close source found,PASS,path=C:\Users\patri\vrp_project\data\processed...
2,RV21D formula available,PASS,rv_window=21; ddof=1; annualization=252
3,VRP formula audit,PASS,max_abs_diff=0.0
4,3m z-score prior-only audit,PASS,window=63; min_periods=63; ddof=1; max_abs_dif...
5,1y z-score prior-only audit,PASS,window=252; min_periods=252; ddof=1; max_abs_d...
6,RSI14 available,PASS,Used source RSI column: rsi14
7,Outcome join,PASS,path=C:\Users\patri\vrp_project\data\processed...
8,Actual DTE join,PASS,actual_dte_col=actual_dte; joined_non_null=1606
9,P&L/day formula,PASS,pnl_per_day = outcome_value / actual_dte



Frozen 30D Excel-style baseline benchmark:


...


Baseline year summary:


,year,eligible_rows,core_trades,core_frequency,core_win_rate,core_avg_pnl_per_trade,core_avg_pnl_per_day,core_total_pnl,core_worst_trade
0,2020,1.0,0.0,0.000000,NaN,NaN,NaN,0.000000,NaN
1,2021,252.0,31.0,0.123016,0.967742,18320.032877,603.189600,567921.019190,0.000000
2,2022,251.0,5.0,0.019920,1.000000,28797.454066,943.906153,143987.270331,22822.456481
3,2023,250.0,46.0,0.184000,0.826087,9098.649702,302.092045,418537.886280,-27822.697682
4,2024,252.0,47.0,0.186508,0.936170,13816.600593,460.911118,649380.227890,-3916.346381
5,2025,250.0,23.0,0.092000,0.695652,-9082.997335,-288.744667,-208908.938696,-112316.486943
6,2026,98.0,25.0,0.255102,0.800000,14982.814046,498.837279,374570.351139,-13409.852082



Baseline worst 20 trades:


,date,trade_date,tenor,vix_30d_vol_pct,rv21d_vol_pct,vrp_log,vrp_z_3m,vrp_z_1y,rsi14,outcome_value,actual_dte,pnl_per_day
0,2025-03-03,20250303,30,22.808456,13.932921,0.985754,1.106951,0.810141,37.160997,-112316.486943,32.0,-3509.890217
1,2025-03-04,20250304,30,23.834459,14.368250,1.012223,1.132385,0.853232,33.223807,-100779.661310,31.0,-3250.956816
2,2025-03-06,20250306,30,25.083189,15.656730,0.942594,0.921200,0.711711,33.946232,-92588.332880,29.0,-3192.701134
3,2025-02-24,20250224,30,19.078121,11.772753,0.965509,1.274300,0.806744,43.422979,-51336.648143,32.0,-1604.270254
4,2025-02-26,20250226,30,19.083176,10.753467,1.147158,1.624308,1.136551,41.127776,-47172.123854,30.0,-1572.404128
5,2025-02-25,20250225,30,19.578612,11.824882,1.008464,1.337134,0.883105,41.026556,-46389.320348,31.0,-1496.429689
6,2025-02-27,20250227,30,21.086209,11.428641,1.224993,1.744878,1.273389,33.832673,-30036.321327,29.0,-1035.735218
7,2023-07-28,20230728,30,13.306566,8.708607,0.847891,1.502035,1.534266,69.051460,-27822.697682,28.0,-993.667774
8,2023-09-28,20230928,30,17.320099,10.918979,0.922730,0.775156,1.377954,35.160534,-26311.137986,29.0,-907.280620
9,2023-07-21,20230721,30,13.620362,9.342003,0.754090,0.867219,1.334829,68.093215,-25811.557335,28.0,-921.841333



Cell 1 outputs saved:
Baseline benchmark:       C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\01_30d_corsi_baseline_benchmark_20200102_20260602_20260704_125621.csv
Success criteria:         C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\01_30d_corsi_success_criteria_20200102_20260602_20260704_125621.csv
Baseline file manifest:   C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\01_30d_corsi_baseline_file_manifest_20200102_20260602_20260704_125621.csv
Signal summary:           C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\01_30d_corsi_baseline_signal_summary_20200102_20260602_20260704_125621.csv

Cell 1 complete.

Next proposed cell:
Cell 2: Build close-to-close 30D Corsi/HAR feature library and 21-trading-day forward realized variance target.
No model fitting until Cell 2 feature/target construction is reviewed.


In [2]:
# Cell 2: Build 30D Corsi/HAR feature library and 21-trading-day forward RV target
#
# Objective:
#   Build a clean model-ready feature/target panel for 30D Corsi/HAR denominator research.
#
# Scope:
#   - Load frozen 30D Excel baseline signal tape
#   - Load the original close / RSI source from the baseline manifest
#   - Recompute close-to-close log returns
#   - Recompute Excel-style 21D realized variance and verify it matches baseline
#   - Build a broad realized-variance feature library
#   - Build 21-trading-day forward realized variance target
#   - Save feature-target panel and diagnostics
#
# Explicitly NOT doing in this cell:
#   - No model fitting
#   - No Ridge regression
#   - No forecast generation
#   - No new VRP signal
#   - No threshold testing
#   - No optimization
#   - No cross-tenor work
#
# Key design choice:
#   The simple Corsi primitives rv_1d / rv_5d / rv_21d / rv_63d are included only
#   as raw feature-library inputs. They are not being selected as the final model.
#
# Target:
#   forward_rv_21td_variance_decimal =
#       (STDEV.S(next 21 trading-day close-to-close log returns) * sqrt(252))^2
#
# Timing:
#   Features may use information through the signal date close.
#   Target uses returns t+1 through t+21 only.

from pathlib import Path
import json
import numpy as np
import pandas as pd


# =============================================================================
# 0. Paths and constants
# =============================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

BASELINE_BRANCH = "vrp_30d_baseline_v1"
CORSI_BRANCH = "vrp_30d_corsi_v1"

BASELINE_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / BASELINE_BRANCH
BASELINE_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / BASELINE_BRANCH

CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / CORSI_BRANCH
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / CORSI_BRANCH

CORSI_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CORSI_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

TRADING_DAYS_PER_YEAR = 252
RV_WINDOW_BASELINE = 21
RV_DDOF = 1
TARGET_FORWARD_DAYS = 21
VAR_FLOOR = 1e-10

print("Cell 2: Build 30D Corsi/HAR feature library and forward RV target")
print(f"Project root:         {PROJECT_ROOT}")
print(f"Baseline processed:   {BASELINE_PROCESSED_DIR}")
print(f"Baseline audit:       {BASELINE_AUDIT_DIR}")
print(f"Corsi processed:      {CORSI_PROCESSED_DIR}")
print(f"Corsi audit:          {CORSI_AUDIT_DIR}")
print(f"Run timestamp:        {RUN_TIMESTAMP}")

print()
print("Locked Cell 2 conventions:")
display(pd.DataFrame([{
    "target": "forward 21 trading-day close-to-close realized variance",
    "target_formula": "(STDEV.S(next 21 log returns) * sqrt(252))^2",
    "feature_timing": "features use data through signal date close only",
    "target_timing": "target uses returns t+1 through t+21 only",
    "baseline_rv_check": "recomputed rv_std_21d must match frozen baseline rv21d_variance_decimal",
    "rv_ddof": RV_DDOF,
    "annualization": TRADING_DAYS_PER_YEAR,
    "var_floor_for_logs": VAR_FLOOR,
}]))


# =============================================================================
# 1. Helpers
# =============================================================================

def latest_file(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)

    if not matches:
        raise FileNotFoundError(f"No files found in {directory} matching pattern: {pattern}")

    return matches[0]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)

    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)

    if path.suffix.lower() in [".csv", ".txt"]:
        return pd.read_csv(path)

    raise ValueError(f"Unsupported file extension: {path}")


def first_existing_col(df: pd.DataFrame, candidates) -> str | None:
    cols = set(df.columns)

    for c in candidates:
        if c in cols:
            return c

    lower_map = {str(c).lower(): c for c in df.columns}

    for c in candidates:
        if str(c).lower() in lower_map:
            return lower_map[str(c).lower()]

    return None


def date_to_trade_date(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series).dt.strftime("%Y%m%d").astype("Int64")


def safe_div(num, den):
    out = num / den
    return out.replace([np.inf, -np.inf], np.nan)


def safe_log(x, floor=VAR_FLOOR):
    return np.log(np.maximum(x, floor))


def rolling_sample_variance_annualized(returns: pd.Series, window: int) -> pd.Series:
    return returns.rolling(window=window, min_periods=window).std(ddof=RV_DDOF).pow(2) * TRADING_DAYS_PER_YEAR


def rolling_mean_squared_return_annualized(returns: pd.Series, window: int) -> pd.Series:
    return returns.pow(2).rolling(window=window, min_periods=window).mean() * TRADING_DAYS_PER_YEAR


def rolling_semivariance_annualized(returns: pd.Series, window: int, side: str) -> pd.Series:
    if side == "downside":
        semi = np.where(returns < 0, returns.pow(2), 0.0)
    elif side == "upside":
        semi = np.where(returns > 0, returns.pow(2), 0.0)
    else:
        raise ValueError("side must be 'downside' or 'upside'")

    return pd.Series(semi, index=returns.index).rolling(window=window, min_periods=window).mean() * TRADING_DAYS_PER_YEAR


def diagnostic_summary(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []

    for col in cols:
        s = pd.to_numeric(df[col], errors="coerce")
        finite = s.replace([np.inf, -np.inf], np.nan).dropna()

        rows.append({
            "column": col,
            "non_null": int(s.notna().sum()),
            "null": int(s.isna().sum()),
            "inf_count": int(np.isinf(s.dropna()).sum()),
            "min": float(finite.min()) if len(finite) else np.nan,
            "p01": float(finite.quantile(0.01)) if len(finite) else np.nan,
            "median": float(finite.median()) if len(finite) else np.nan,
            "mean": float(finite.mean()) if len(finite) else np.nan,
            "p99": float(finite.quantile(0.99)) if len(finite) else np.nan,
            "max": float(finite.max()) if len(finite) else np.nan,
        })

    return pd.DataFrame(rows)


def add_validation(rows, check_name, passed, details):
    rows.append({
        "check": check_name,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": details,
    })


# =============================================================================
# 2. Load frozen baseline files
# =============================================================================

baseline_signal_path = latest_file(
    BASELINE_PROCESSED_DIR,
    "00_excel_30d_vrp_baseline_signal_tape_*.parquet",
)

baseline_manifest_path = latest_file(
    BASELINE_AUDIT_DIR,
    "00_excel_30d_vrp_baseline_source_manifest_*.csv",
)

baseline_signal = pd.read_parquet(baseline_signal_path)
baseline_manifest = pd.read_csv(baseline_manifest_path)

baseline_signal = normalize_columns(baseline_signal)
baseline_manifest = normalize_columns(baseline_manifest)

baseline_signal["date"] = pd.to_datetime(baseline_signal["date"], errors="coerce").dt.normalize()
baseline_signal["trade_date"] = pd.to_numeric(baseline_signal["trade_date"], errors="coerce").astype("Int64")

print()
print("Frozen baseline signal tape loaded:")
print(baseline_signal_path)

print()
print("Frozen baseline manifest loaded:")
print(baseline_manifest_path)

print()
print("Baseline manifest:")
display(baseline_manifest)


# =============================================================================
# 3. Load close / RSI source used by baseline
# =============================================================================

feature_manifest = baseline_manifest.loc[
    baseline_manifest["role"].astype(str).eq("close_rsi_features")
].copy()

if len(feature_manifest) != 1:
    raise ValueError(f"Expected exactly one close_rsi_features manifest row, found {len(feature_manifest)}.")

feature_source_path = Path(feature_manifest.iloc[0]["path"])

feature_raw = normalize_columns(read_table(feature_source_path))

# Prefer columns recorded in manifest, but rediscover defensively.
try:
    manifest_cols_used = json.loads(feature_manifest.iloc[0]["columns_used"])
except Exception:
    manifest_cols_used = []

manifest_cols_used = [c for c in manifest_cols_used if isinstance(c, str) and c in feature_raw.columns]

date_col = first_existing_col(feature_raw, ["date", "quote_date"])
trade_date_col = first_existing_col(feature_raw, ["trade_date"])

close_col = first_existing_col(
    feature_raw,
    [
        "spx_close",
        "spy_close",
        "underlying_close",
        "adj_close",
        "adjusted_close",
        "close",
        "close_price",
        "spy_adj_close",
    ],
)

rsi_col = first_existing_col(feature_raw, ["rsi14", "rsi_14", "rsi"])

if date_col is None and trade_date_col is None:
    raise ValueError(f"Close / RSI source missing date or trade_date column: {feature_source_path}")

if close_col is None:
    raise ValueError(f"Close / RSI source missing close column: {feature_source_path}")

feature = feature_raw.copy()

if date_col is not None:
    feature["date"] = pd.to_datetime(feature[date_col], errors="coerce").dt.normalize()
elif trade_date_col is not None:
    feature["date"] = pd.to_datetime(
        feature[trade_date_col].astype(str),
        format="%Y%m%d",
        errors="coerce",
    ).dt.normalize()

if trade_date_col is not None:
    feature["trade_date"] = pd.to_numeric(feature[trade_date_col], errors="coerce").astype("Int64")
else:
    feature["trade_date"] = date_to_trade_date(feature["date"])

feature["underlying_close"] = pd.to_numeric(feature[close_col], errors="coerce")

if rsi_col is not None:
    feature["rsi14_source"] = pd.to_numeric(feature[rsi_col], errors="coerce")
else:
    feature["rsi14_source"] = np.nan

feature = (
    feature[["date", "trade_date", "underlying_close", "rsi14_source"]]
    .dropna(subset=["date", "trade_date", "underlying_close"])
    .drop_duplicates("trade_date")
    .sort_values("date")
    .reset_index(drop=True)
)

print()
print("Close / RSI source selected:")
print(feature_source_path)
print(f"Close column used: {close_col}")
print(f"RSI column used:   {rsi_col}")

print()
print("Close source summary:")
display(pd.DataFrame([{
    "rows": len(feature),
    "first_date": feature["date"].min(),
    "last_date": feature["date"].max(),
    "close_non_null": int(feature["underlying_close"].notna().sum()),
    "rsi_source_non_null": int(feature["rsi14_source"].notna().sum()),
    "duplicate_trade_dates": int(feature["trade_date"].duplicated().sum()),
}]))


# =============================================================================
# 4. Compute close-to-close returns
# =============================================================================

feature["daily_log_return"] = np.log(feature["underlying_close"] / feature["underlying_close"].shift(1))
feature["daily_rv_1d"] = feature["daily_log_return"].pow(2) * TRADING_DAYS_PER_YEAR

feature["return_formula_check"] = (
    feature["daily_log_return"]
    - np.log(feature["underlying_close"] / feature["underlying_close"].shift(1))
)

print()
print("Return summary:")
display(pd.DataFrame([{
    "log_return_non_null": int(feature["daily_log_return"].notna().sum()),
    "first_non_null_return_date": feature.loc[feature["daily_log_return"].notna(), "date"].min(),
    "max_abs_return_formula_diff": feature["return_formula_check"].dropna().abs().max(),
    "min_daily_log_return": feature["daily_log_return"].min(),
    "max_daily_log_return": feature["daily_log_return"].max(),
}]))


# =============================================================================
# 5. Build realized variance feature library
# =============================================================================

# -------------------------------------------------------------------------
# 5.1 Excel-style rolling sample variance features
#
# These match the Excel denominator style:
#   (STDEV.S(window returns) * sqrt(252))^2
# -------------------------------------------------------------------------

feature["rv_1d"] = feature["daily_rv_1d"]

for window in [5, 21, 63]:
    feature[f"rv_std_{window}d"] = rolling_sample_variance_annualized(
        feature["daily_log_return"],
        window=window,
    )

# Alias the baseline-style 21D denominator.
feature["rv_std_21d_vol_pct"] = np.sqrt(feature["rv_std_21d"]) * 100.0


# -------------------------------------------------------------------------
# 5.2 HAR/Corsi-style rolling average of daily squared returns
#
# These are closer to the usual HAR realized-variance inputs:
#   rolling mean of daily RV
# -------------------------------------------------------------------------

for window in [5, 21, 63]:
    feature[f"rv_mean_{window}d"] = rolling_mean_squared_return_annualized(
        feature["daily_log_return"],
        window=window,
    )


# -------------------------------------------------------------------------
# 5.3 Log variance features
# -------------------------------------------------------------------------

log_variance_cols = [
    "rv_1d",
    "rv_std_5d",
    "rv_std_21d",
    "rv_std_63d",
    "rv_mean_5d",
    "rv_mean_21d",
    "rv_mean_63d",
]

for col in log_variance_cols:
    feature[f"log_{col}"] = safe_log(feature[col])


# -------------------------------------------------------------------------
# 5.4 Acceleration / slope features
# -------------------------------------------------------------------------

feature["rv_std_5d_over_21d"] = safe_div(feature["rv_std_5d"], feature["rv_std_21d"])
feature["rv_std_21d_over_63d"] = safe_div(feature["rv_std_21d"], feature["rv_std_63d"])

feature["rv_mean_5d_over_21d"] = safe_div(feature["rv_mean_5d"], feature["rv_mean_21d"])
feature["rv_mean_21d_over_63d"] = safe_div(feature["rv_mean_21d"], feature["rv_mean_63d"])

feature["log_rv_std_5d_minus_21d"] = feature["log_rv_std_5d"] - feature["log_rv_std_21d"]
feature["log_rv_std_21d_minus_63d"] = feature["log_rv_std_21d"] - feature["log_rv_std_63d"]

feature["rv_std_21d_change_5d"] = feature["rv_std_21d"] - feature["rv_std_21d"].shift(5)
feature["rv_std_21d_change_21d"] = feature["rv_std_21d"] - feature["rv_std_21d"].shift(21)

feature["log_rv_std_21d_change_5d"] = feature["log_rv_std_21d"] - feature["log_rv_std_21d"].shift(5)
feature["log_rv_std_21d_change_21d"] = feature["log_rv_std_21d"] - feature["log_rv_std_21d"].shift(21)


# -------------------------------------------------------------------------
# 5.5 Shock / instability features
# -------------------------------------------------------------------------

feature["abs_daily_log_return"] = feature["daily_log_return"].abs()

feature["max_abs_return_5d"] = feature["abs_daily_log_return"].rolling(5, min_periods=5).max()
feature["max_abs_return_21d"] = feature["abs_daily_log_return"].rolling(21, min_periods=21).max()
feature["max_abs_return_63d"] = feature["abs_daily_log_return"].rolling(63, min_periods=63).max()

feature["rv_1d_over_std_21d"] = safe_div(feature["rv_1d"], feature["rv_std_21d"])

feature["rv_5d_max_daily_rv"] = feature["daily_rv_1d"].rolling(5, min_periods=5).max()
feature["rv_21d_max_daily_rv"] = feature["daily_rv_1d"].rolling(21, min_periods=21).max()
feature["rv_63d_max_daily_rv"] = feature["daily_rv_1d"].rolling(63, min_periods=63).max()

feature["std_daily_rv_21d"] = feature["daily_rv_1d"].rolling(21, min_periods=21).std(ddof=RV_DDOF)
feature["std_daily_rv_63d"] = feature["daily_rv_1d"].rolling(63, min_periods=63).std(ddof=RV_DDOF)


# -------------------------------------------------------------------------
# 5.6 Downside / upside semivariance-style features
#
# Definition:
#   downside_rv_window = rolling mean of r^2 for negative-return days, zero otherwise, annualized
#   upside_rv_window   = rolling mean of r^2 for positive-return days, zero otherwise, annualized
#
# These are features only. They are not the target denominator.
# -------------------------------------------------------------------------

for window in [5, 21, 63]:
    feature[f"downside_rv_{window}d"] = rolling_semivariance_annualized(
        feature["daily_log_return"],
        window=window,
        side="downside",
    )

    feature[f"upside_rv_{window}d"] = rolling_semivariance_annualized(
        feature["daily_log_return"],
        window=window,
        side="upside",
    )

    feature[f"negative_return_count_{window}d"] = (
        feature["daily_log_return"].lt(0)
        .rolling(window=window, min_periods=window)
        .sum()
    )

    feature[f"downside_share_{window}d"] = safe_div(
        feature[f"downside_rv_{window}d"],
        feature[f"downside_rv_{window}d"] + feature[f"upside_rv_{window}d"],
    )

for col in [
    "downside_rv_5d",
    "downside_rv_21d",
    "downside_rv_63d",
    "upside_rv_5d",
    "upside_rv_21d",
    "upside_rv_63d",
]:
    feature[f"log_{col}"] = safe_log(feature[col])


# -------------------------------------------------------------------------
# 5.7 Trend / drawdown features
# -------------------------------------------------------------------------

for window in [5, 21, 63]:
    feature[f"log_return_{window}d"] = np.log(
        feature["underlying_close"] / feature["underlying_close"].shift(window)
    )

    rolling_high = feature["underlying_close"].rolling(window=window, min_periods=window).max()
    feature[f"drawdown_from_{window}d_high"] = feature["underlying_close"] / rolling_high - 1.0


# =============================================================================
# 6. Build forward 21-trading-day realized variance target
# =============================================================================

future_return_cols = []

for i in range(1, TARGET_FORWARD_DAYS + 1):
    col = f"fwd_return_{i}"
    feature[col] = feature["daily_log_return"].shift(-i)
    future_return_cols.append(col)

future_returns = feature[future_return_cols]

target_valid = future_returns.notna().all(axis=1)

feature["target_start_date"] = feature["date"].shift(-1)
feature["target_end_date"] = feature["date"].shift(-TARGET_FORWARD_DAYS)
feature["target_start_trade_date"] = feature["trade_date"].shift(-1)
feature["target_end_trade_date"] = feature["trade_date"].shift(-TARGET_FORWARD_DAYS)
feature["target_num_returns"] = future_returns.notna().sum(axis=1)

feature["forward_rv_21td_variance_decimal"] = np.where(
    target_valid,
    future_returns.var(axis=1, ddof=RV_DDOF) * TRADING_DAYS_PER_YEAR,
    np.nan,
)

feature["forward_rv_21td_vol_pct"] = np.sqrt(feature["forward_rv_21td_variance_decimal"]) * 100.0

feature["forward_log_return_21td"] = np.where(
    target_valid,
    future_returns.sum(axis=1),
    np.nan,
)

feature["forward_target_valid"] = target_valid


# =============================================================================
# 7. Merge feature/target library onto frozen baseline signal dates
# =============================================================================

baseline_rename_map = {
    "rv21d_vol_decimal": "baseline_rv21d_vol_decimal",
    "rv21d_variance_decimal": "baseline_rv21d_variance_decimal",
    "rv21d_vol_pct": "baseline_rv21d_vol_pct",
    "vrp_log": "baseline_vrp_log",
    "vrp_z_3m": "baseline_vrp_z_3m",
    "vrp_z_1y": "baseline_vrp_z_1y",
    "core_signal": "baseline_core_signal",
}

baseline_for_merge = baseline_signal.rename(columns=baseline_rename_map).copy()

# Columns from feature library to merge.
feature_library_cols = [
    "date",
    "trade_date",
    "underlying_close",
    "daily_log_return",
    "daily_rv_1d",

    # Excel-style rolling sample variance features
    "rv_1d",
    "rv_std_5d",
    "rv_std_21d",
    "rv_std_63d",
    "rv_std_21d_vol_pct",

    # HAR/Corsi mean-RV features
    "rv_mean_5d",
    "rv_mean_21d",
    "rv_mean_63d",

    # Logs
    "log_rv_1d",
    "log_rv_std_5d",
    "log_rv_std_21d",
    "log_rv_std_63d",
    "log_rv_mean_5d",
    "log_rv_mean_21d",
    "log_rv_mean_63d",

    # Acceleration
    "rv_std_5d_over_21d",
    "rv_std_21d_over_63d",
    "rv_mean_5d_over_21d",
    "rv_mean_21d_over_63d",
    "log_rv_std_5d_minus_21d",
    "log_rv_std_21d_minus_63d",
    "rv_std_21d_change_5d",
    "rv_std_21d_change_21d",
    "log_rv_std_21d_change_5d",
    "log_rv_std_21d_change_21d",

    # Shock / instability
    "abs_daily_log_return",
    "max_abs_return_5d",
    "max_abs_return_21d",
    "max_abs_return_63d",
    "rv_1d_over_std_21d",
    "rv_5d_max_daily_rv",
    "rv_21d_max_daily_rv",
    "rv_63d_max_daily_rv",
    "std_daily_rv_21d",
    "std_daily_rv_63d",

    # Downside / upside
    "downside_rv_5d",
    "downside_rv_21d",
    "downside_rv_63d",
    "upside_rv_5d",
    "upside_rv_21d",
    "upside_rv_63d",
    "log_downside_rv_5d",
    "log_downside_rv_21d",
    "log_downside_rv_63d",
    "log_upside_rv_5d",
    "log_upside_rv_21d",
    "log_upside_rv_63d",
    "negative_return_count_5d",
    "negative_return_count_21d",
    "negative_return_count_63d",
    "downside_share_5d",
    "downside_share_21d",
    "downside_share_63d",

    # Trend / drawdown
    "log_return_5d",
    "log_return_21d",
    "log_return_63d",
    "drawdown_from_5d_high",
    "drawdown_from_21d_high",
    "drawdown_from_63d_high",

    # Target
    "target_start_date",
    "target_end_date",
    "target_start_trade_date",
    "target_end_trade_date",
    "target_num_returns",
    "forward_rv_21td_variance_decimal",
    "forward_rv_21td_vol_pct",
    "forward_log_return_21td",
    "forward_target_valid",
]

feature_library = feature[feature_library_cols].copy()

panel = baseline_for_merge.merge(
    feature_library,
    on=["date", "trade_date"],
    how="left",
    validate="one_to_one",
)

# Recompute baseline-compatible VRP formula using baseline implied variance and recomputed rv_std_21d.
panel["baseline_vrp_log_recomputed_from_cell2_rv"] = np.log(
    panel["implied_variance_decimal"] / panel["rv_std_21d"]
)

panel["baseline_vrp_log_recomputed_diff"] = (
    panel["baseline_vrp_log"] - panel["baseline_vrp_log_recomputed_from_cell2_rv"]
)

panel["rv21d_recomputed_diff"] = (
    panel["baseline_rv21d_variance_decimal"] - panel["rv_std_21d"]
)

# Model target convenience logs.
panel["log_forward_rv_21td_variance_decimal"] = safe_log(
    panel["forward_rv_21td_variance_decimal"]
)

panel["is_corsi_target_available"] = panel["forward_rv_21td_variance_decimal"].notna()
panel["is_corsi_feature_base_available"] = panel[
    [
        "log_rv_1d",
        "log_rv_std_5d",
        "log_rv_std_21d",
        "log_rv_std_63d",
        "log_rv_mean_5d",
        "log_rv_mean_21d",
        "log_rv_mean_63d",
    ]
].notna().all(axis=1)

panel["is_corsi_model_row_candidate"] = (
    panel["is_corsi_feature_base_available"]
    & panel["is_corsi_target_available"]
    & panel["implied_variance_decimal"].notna()
    & panel["outcome_value"].notna()
    & panel["actual_dte"].notna()
)


# =============================================================================
# 8. Feature group manifest
# =============================================================================

feature_groups = {
    "har_total_std_v1": [
        "log_rv_1d",
        "log_rv_std_5d",
        "log_rv_std_21d",
        "log_rv_std_63d",
    ],
    "har_total_mean_v1": [
        "log_rv_1d",
        "log_rv_mean_5d",
        "log_rv_mean_21d",
        "log_rv_mean_63d",
    ],
    "har_total_accel_v1": [
        "log_rv_1d",
        "log_rv_std_5d",
        "log_rv_std_21d",
        "log_rv_std_63d",
        "rv_std_5d_over_21d",
        "rv_std_21d_over_63d",
        "log_rv_std_5d_minus_21d",
        "log_rv_std_21d_minus_63d",
        "log_rv_std_21d_change_5d",
        "log_rv_std_21d_change_21d",
    ],
    "har_downside_v1": [
        "log_rv_1d",
        "log_rv_std_5d",
        "log_rv_std_21d",
        "log_rv_std_63d",
        "log_downside_rv_5d",
        "log_downside_rv_21d",
        "log_downside_rv_63d",
        "downside_share_21d",
        "negative_return_count_21d",
    ],
    "har_regime_v1": [
        "log_rv_1d",
        "log_rv_std_5d",
        "log_rv_std_21d",
        "log_rv_std_63d",
        "max_abs_return_5d",
        "max_abs_return_21d",
        "rv_1d_over_std_21d",
        "std_daily_rv_21d",
        "log_return_5d",
        "log_return_21d",
        "drawdown_from_21d_high",
        "drawdown_from_63d_high",
    ],
    "har_full_defensible_v1": [
        "log_rv_1d",
        "log_rv_std_5d",
        "log_rv_std_21d",
        "log_rv_std_63d",
        "log_rv_mean_5d",
        "log_rv_mean_21d",
        "log_rv_mean_63d",
        "rv_std_5d_over_21d",
        "rv_std_21d_over_63d",
        "log_rv_std_5d_minus_21d",
        "log_rv_std_21d_minus_63d",
        "log_rv_std_21d_change_5d",
        "log_rv_std_21d_change_21d",
        "log_downside_rv_5d",
        "log_downside_rv_21d",
        "log_downside_rv_63d",
        "downside_share_21d",
        "negative_return_count_21d",
        "max_abs_return_5d",
        "max_abs_return_21d",
        "rv_1d_over_std_21d",
        "std_daily_rv_21d",
        "log_return_5d",
        "log_return_21d",
        "drawdown_from_21d_high",
        "drawdown_from_63d_high",
    ],
}

feature_group_rows = []

for group_name, cols in feature_groups.items():
    for col in cols:
        feature_group_rows.append({
            "feature_group": group_name,
            "feature": col,
        })

feature_group_manifest = pd.DataFrame(feature_group_rows)


# =============================================================================
# 9. Validation checks
# =============================================================================

validation_rows = []

add_validation(
    validation_rows,
    "Baseline signal tape loaded",
    len(baseline_signal) > 0,
    f"path={baseline_signal_path}; rows={len(baseline_signal)}",
)

add_validation(
    validation_rows,
    "Close source loaded",
    len(feature) > 0 and feature["underlying_close"].notna().any(),
    f"path={feature_source_path}; close_col={close_col}; rows={len(feature)}",
)

add_validation(
    validation_rows,
    "No duplicate baseline trade_date rows",
    baseline_signal["trade_date"].duplicated().sum() == 0,
    f"duplicate_trade_dates={int(baseline_signal['trade_date'].duplicated().sum())}",
)

add_validation(
    validation_rows,
    "No duplicate close-source trade_date rows",
    feature["trade_date"].duplicated().sum() == 0,
    f"duplicate_trade_dates={int(feature['trade_date'].duplicated().sum())}",
)

max_return_formula_diff = feature["return_formula_check"].dropna().abs().max()

add_validation(
    validation_rows,
    "Return formula valid",
    max_return_formula_diff == 0,
    f"max_abs_return_formula_diff={max_return_formula_diff}",
)

rv21d_max_abs_diff = panel["rv21d_recomputed_diff"].dropna().abs().max()

add_validation(
    validation_rows,
    "RV21D recomputation matches frozen baseline",
    rv21d_max_abs_diff < 1e-12,
    f"max_abs_diff={rv21d_max_abs_diff}",
)

vrp_max_abs_diff = panel["baseline_vrp_log_recomputed_diff"].dropna().abs().max()

add_validation(
    validation_rows,
    "Baseline VRP recomputation matches using Cell 2 RV",
    vrp_max_abs_diff < 1e-12,
    f"max_abs_diff={vrp_max_abs_diff}",
)

target_timing_valid = (
    panel.loc[panel["forward_target_valid"], "target_start_date"].gt(
        panel.loc[panel["forward_target_valid"], "date"]
    ).all()
    and panel.loc[panel["forward_target_valid"], "target_end_date"].gt(
        panel.loc[panel["forward_target_valid"], "date"]
    ).all()
)

add_validation(
    validation_rows,
    "Forward target uses future dates only",
    target_timing_valid,
    "target_start_date and target_end_date are both greater than signal date for valid targets",
)

add_validation(
    validation_rows,
    "Forward target has exactly 21 returns when valid",
    panel.loc[panel["forward_target_valid"], "target_num_returns"].eq(TARGET_FORWARD_DAYS).all(),
    f"target_forward_days={TARGET_FORWARD_DAYS}",
)

numeric_cols = panel.select_dtypes(include=[np.number]).columns.tolist()
inf_count_total = int(np.isinf(panel[numeric_cols]).sum().sum())

add_validation(
    validation_rows,
    "No infinite numeric values in panel",
    inf_count_total == 0,
    f"total_inf_count={inf_count_total}",
)

add_validation(
    validation_rows,
    "Model row candidates available",
    panel["is_corsi_model_row_candidate"].sum() > 0,
    f"candidate_rows={int(panel['is_corsi_model_row_candidate'].sum())}",
)

validation = pd.DataFrame(validation_rows)

print()
print("Cell 2 validation checks:")
display(validation)

if validation["status"].eq("FAIL").any():
    print()
    print("FAILED VALIDATION CHECKS:")
    display(validation.loc[validation["status"].eq("FAIL")])
    raise ValueError("Cell 2 validation failed. Stop before model fitting.")


# =============================================================================
# 10. Diagnostics
# =============================================================================

feature_cols_for_diag = sorted(set(
    [
        "daily_log_return",
        "daily_rv_1d",
        "rv_1d",
        "rv_std_5d",
        "rv_std_21d",
        "rv_std_63d",
        "rv_mean_5d",
        "rv_mean_21d",
        "rv_mean_63d",
        "rv_std_5d_over_21d",
        "rv_std_21d_over_63d",
        "log_rv_std_21d_change_5d",
        "log_rv_std_21d_change_21d",
        "downside_rv_21d",
        "upside_rv_21d",
        "downside_share_21d",
        "negative_return_count_21d",
        "max_abs_return_21d",
        "rv_1d_over_std_21d",
        "std_daily_rv_21d",
        "log_return_21d",
        "drawdown_from_21d_high",
        "forward_rv_21td_variance_decimal",
        "forward_rv_21td_vol_pct",
        "log_forward_rv_21td_variance_decimal",
    ]
))

feature_diagnostics = diagnostic_summary(panel, feature_cols_for_diag)

target_diagnostics = pd.DataFrame([{
    "rows": len(panel),
    "first_date": panel["date"].min(),
    "last_date": panel["date"].max(),
    "target_valid_rows": int(panel["forward_target_valid"].sum()),
    "target_missing_rows": int(panel["forward_rv_21td_variance_decimal"].isna().sum()),
    "first_valid_target_date": panel.loc[panel["forward_target_valid"], "date"].min(),
    "last_valid_target_date": panel.loc[panel["forward_target_valid"], "date"].max(),
    "median_forward_vol_pct": panel["forward_rv_21td_vol_pct"].median(),
    "min_forward_vol_pct": panel["forward_rv_21td_vol_pct"].min(),
    "max_forward_vol_pct": panel["forward_rv_21td_vol_pct"].max(),
    "model_row_candidates": int(panel["is_corsi_model_row_candidate"].sum()),
}])

feature_availability_by_group_rows = []

for group_name, cols in feature_groups.items():
    feature_availability_by_group_rows.append({
        "feature_group": group_name,
        "feature_count": len(cols),
        "rows_all_features_available": int(panel[cols].notna().all(axis=1).sum()),
        "rows_all_features_and_target_available": int(
            (panel[cols].notna().all(axis=1) & panel["is_corsi_target_available"]).sum()
        ),
    })

feature_availability_by_group = pd.DataFrame(feature_availability_by_group_rows)

target_by_year = (
    panel.assign(year=panel["date"].dt.year)
    .groupby("year", dropna=False)
    .agg(
        rows=("date", "size"),
        target_valid_rows=("forward_target_valid", "sum"),
        median_forward_vol_pct=("forward_rv_21td_vol_pct", "median"),
        mean_forward_vol_pct=("forward_rv_21td_vol_pct", "mean"),
        max_forward_vol_pct=("forward_rv_21td_vol_pct", "max"),
    )
    .reset_index()
)

print()
print("Feature-target panel summary:")
display(pd.DataFrame([{
    "rows": len(panel),
    "first_date": panel["date"].min(),
    "last_date": panel["date"].max(),
    "baseline_core_trades": int(panel["baseline_core_signal"].sum()),
    "target_valid_rows": int(panel["forward_target_valid"].sum()),
    "model_row_candidates": int(panel["is_corsi_model_row_candidate"].sum()),
    "rv21d_recompute_max_abs_diff": rv21d_max_abs_diff,
    "vrp_recompute_max_abs_diff": vrp_max_abs_diff,
}]))


print()
print("Target diagnostics:")
display(target_diagnostics)

print()
print("Target diagnostics by year:")
display(target_by_year)

print()
print("Feature availability by proposed model group:")
display(feature_availability_by_group)

print()
print("Feature diagnostics:")
display(feature_diagnostics)

print()
print("Feature group manifest:")
display(feature_group_manifest)


# =============================================================================
# 11. Save outputs
# =============================================================================

safe_start = panel["date"].min().strftime("%Y%m%d")
safe_end = panel["date"].max().strftime("%Y%m%d")

feature_target_panel_path = (
    CORSI_PROCESSED_DIR
    / f"02_30d_corsi_feature_target_panel_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.parquet"
)

feature_target_panel_csv_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_feature_target_panel_preview_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

validation_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_feature_target_validation_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

feature_diagnostics_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_feature_diagnostics_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_diagnostics_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_target_diagnostics_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_by_year_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_target_by_year_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

feature_group_manifest_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_feature_group_manifest_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

feature_availability_by_group_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_feature_availability_by_group_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

panel.to_parquet(feature_target_panel_path, index=False)

preview_cols = [
    "date",
    "trade_date",
    "vix_30d_vol_pct",
    "implied_variance_decimal",
    "baseline_rv21d_vol_pct",
    "baseline_rv21d_variance_decimal",
    "rv_std_21d",
    "rv21d_recomputed_diff",
    "baseline_vrp_log",
    "baseline_vrp_log_recomputed_from_cell2_rv",
    "baseline_vrp_log_recomputed_diff",
    "rv_1d",
    "rv_std_5d",
    "rv_std_21d",
    "rv_std_63d",
    "rv_mean_5d",
    "rv_mean_21d",
    "rv_mean_63d",
    "downside_rv_21d",
    "upside_rv_21d",
    "downside_share_21d",
    "negative_return_count_21d",
    "max_abs_return_21d",
    "drawdown_from_21d_high",
    "target_start_date",
    "target_end_date",
    "target_num_returns",
    "forward_rv_21td_variance_decimal",
    "forward_rv_21td_vol_pct",
    "log_forward_rv_21td_variance_decimal",
    "forward_target_valid",
    "baseline_core_signal",
    "outcome_value",
    "actual_dte",
    "pnl_per_day",
]

panel[preview_cols].to_csv(feature_target_panel_csv_path, index=False)

validation.to_csv(validation_path, index=False)
feature_diagnostics.to_csv(feature_diagnostics_path, index=False)
target_diagnostics.to_csv(target_diagnostics_path, index=False)
target_by_year.to_csv(target_by_year_path, index=False)
feature_group_manifest.to_csv(feature_group_manifest_path, index=False)
feature_availability_by_group.to_csv(feature_availability_by_group_path, index=False)

print()
print("Cell 2 outputs saved:")
print(f"Feature-target panel parquet:      {feature_target_panel_path}")
print(f"Feature-target preview CSV:        {feature_target_panel_csv_path}")
print(f"Validation checks:                 {validation_path}")
print(f"Feature diagnostics:               {feature_diagnostics_path}")
print(f"Target diagnostics:                {target_diagnostics_path}")
print(f"Target by year:                    {target_by_year_path}")
print(f"Feature group manifest:            {feature_group_manifest_path}")
print(f"Feature availability by group:     {feature_availability_by_group_path}")

print()
print("Cell 2 complete.")
print()
print("Next proposed cell:")
print("Cell 3: Fit walk-forward 30D Corsi/HAR candidate denominators.")
print("No threshold optimization in Cell 3; first apply the same baseline thresholds to model-derived VRP.")

Cell 2: Build 30D Corsi/HAR feature library and forward RV target
Project root:         C:\Users\patri\vrp_project
Baseline processed:   C:\Users\patri\vrp_project\data\processed\vrp_30d_baseline_v1
Baseline audit:       C:\Users\patri\vrp_project\data\audit\vrp_30d_baseline_v1
Corsi processed:      C:\Users\patri\vrp_project\data\processed\vrp_30d_corsi_v1
Corsi audit:          C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1
Run timestamp:        20260704_130328

Locked Cell 2 conventions:


,target,target_formula,feature_timing,target_timing,baseline_rv_check,rv_ddof,annualization,var_floor_for_logs
0,forward 21 trading-day close-to-close realized...,(STDEV.S(next 21 log returns) * sqrt(252))^2,features use data through signal date close only,target uses returns t+1 through t+21 only,recomputed rv_std_21d must match frozen baseli...,1,252,1.000000e-10



Frozen baseline signal tape loaded:
C:\Users\patri\vrp_project\data\processed\vrp_30d_baseline_v1\00_excel_30d_vrp_baseline_signal_tape_20200102_20260602_20260704_122658.parquet

Frozen baseline manifest loaded:
C:\Users\patri\vrp_project\data\audit\vrp_30d_baseline_v1\00_excel_30d_vrp_baseline_source_manifest_20200102_20260602_20260704_122658.csv

Baseline manifest:


,role,path,exists,row_count_file,columns_used,notes,row_count_loaded,first_date_loaded,last_date_loaded
0,30d_implied_vol_variance,C:\Users\patri\vrp_project\data\processed\fore...,True,14565,"[""date"", ""trade_date"", ""tenor"", ""vix_style_vol""]",Used vix_style_vol as percent vol.,1612,2020-01-02,2026-06-02
1,close_rsi_features,C:\Users\patri\vrp_project\data\processed\stag...,True,18144,"[""date"", ""trade_date"", ""spx_close"", ""rsi14""]",Used source RSI column: rsi14,2016,2018-06-25,2026-07-02
2,trade_outcomes,C:\Users\patri\vrp_project\data\processed\put_...,True,18099,"[""date"", ""trade_date"", ""tenor"", ""normalized_pn...",30D outcomes only,1989,2018-06-25,2026-05-22



Close / RSI source selected:
C:\Users\patri\vrp_project\data\processed\staging\production_feature_panel_v0_1_candidate_20180625_20260702.parquet
Close column used: spx_close
RSI column used:   rsi14

Close source summary:


...


Return summary:


,log_return_non_null,first_non_null_return_date,max_abs_return_formula_diff,min_daily_log_return,max_daily_log_return
0,2015,2018-06-26,0.0,-0.127652,0.090895



Cell 2 validation checks:


,check,status,details
0,Baseline signal tape loaded,PASS,path=C:\Users\patri\vrp_project\data\processed...
1,Close source loaded,PASS,path=C:\Users\patri\vrp_project\data\processed...
2,No duplicate baseline trade_date rows,PASS,duplicate_trade_dates=0
3,No duplicate close-source trade_date rows,PASS,duplicate_trade_dates=0
4,Return formula valid,PASS,max_abs_return_formula_diff=0.0
5,RV21D recomputation matches frozen baseline,PASS,max_abs_diff=2.220446049250313e-16
6,Baseline VRP recomputation matches using Cell ...,PASS,max_abs_diff=4.996003610813204e-16
7,Forward target uses future dates only,PASS,target_start_date and target_end_date are both...
8,Forward target has exactly 21 returns when valid,PASS,target_forward_days=21
9,No infinite numeric values in panel,PASS,total_inf_count=0


KeyError: 'daily_log_return'

In [3]:
# Cell 2A: Repair duplicate merge-column names and finish Cell 2 diagnostics/saves
#
# Purpose:
#   Fix pandas merge suffix issue from Cell 2.
#
# No methodology change:
#   - No feature changes
#   - No target changes
#   - No validation changes
#   - No model fitting
#
# Cause:
#   baseline_signal and feature_library both contained some columns like daily_log_return.
#   pandas created suffixed columns such as daily_log_return_x / daily_log_return_y.
#   Diagnostics expected the canonical unsuffixed name.

from pathlib import Path
import numpy as np
import pandas as pd


# =============================================================================
# 1. Canonicalize duplicated merge columns
# =============================================================================

def canonicalize_from_suffixes(df, base_name, prefer="feature"):
    """
    Restore a canonical unsuffixed column if pandas created _x / _y suffixes.

    prefer="feature" means:
        use base_name_y first, then base_name, then base_name_x

    In this notebook:
        _x usually comes from the frozen baseline signal tape
        _y usually comes from the newly built feature library
    """
    candidates_feature_first = [f"{base_name}_y", base_name, f"{base_name}_x"]
    candidates_existing_first = [base_name, f"{base_name}_y", f"{base_name}_x"]

    candidates = candidates_feature_first if prefer == "feature" else candidates_existing_first

    available = [c for c in candidates if c in df.columns]

    if available:
        df[base_name] = df[available[0]]

    return df, available


columns_to_canonicalize = [
    "underlying_close",
    "daily_log_return",
    "daily_rv_1d",
]

canonicalization_rows = []

for col in columns_to_canonicalize:
    panel, available = canonicalize_from_suffixes(panel, col, prefer="feature")
    canonicalization_rows.append({
        "canonical_column": col,
        "available_source_columns": ", ".join(available),
        "status": "RESTORED" if col in panel.columns else "MISSING",
    })

canonicalization_summary = pd.DataFrame(canonicalization_rows)

print("Column canonicalization summary:")
display(canonicalization_summary)

missing_after_canonicalization = canonicalization_summary.loc[
    canonicalization_summary["status"].ne("RESTORED")
]

if len(missing_after_canonicalization) > 0:
    raise ValueError("Missing required canonical columns after suffix repair.")


# =============================================================================
# 2. Re-run diagnostics from Cell 2
# =============================================================================

feature_cols_for_diag = sorted(set(
    [
        "daily_log_return",
        "daily_rv_1d",
        "rv_1d",
        "rv_std_5d",
        "rv_std_21d",
        "rv_std_63d",
        "rv_mean_5d",
        "rv_mean_21d",
        "rv_mean_63d",
        "rv_std_5d_over_21d",
        "rv_std_21d_over_63d",
        "log_rv_std_21d_change_5d",
        "log_rv_std_21d_change_21d",
        "downside_rv_21d",
        "upside_rv_21d",
        "downside_share_21d",
        "negative_return_count_21d",
        "max_abs_return_21d",
        "rv_1d_over_std_21d",
        "std_daily_rv_21d",
        "log_return_21d",
        "drawdown_from_21d_high",
        "forward_rv_21td_variance_decimal",
        "forward_rv_21td_vol_pct",
        "log_forward_rv_21td_variance_decimal",
    ]
))

missing_diag_cols = [c for c in feature_cols_for_diag if c not in panel.columns]

if missing_diag_cols:
    print("Missing diagnostic columns:")
    print(missing_diag_cols)
    raise KeyError(f"Missing diagnostic columns: {missing_diag_cols}")

feature_diagnostics = diagnostic_summary(panel, feature_cols_for_diag)

target_diagnostics = pd.DataFrame([{
    "rows": len(panel),
    "first_date": panel["date"].min(),
    "last_date": panel["date"].max(),
    "target_valid_rows": int(panel["forward_target_valid"].sum()),
    "target_missing_rows": int(panel["forward_rv_21td_variance_decimal"].isna().sum()),
    "first_valid_target_date": panel.loc[panel["forward_target_valid"], "date"].min(),
    "last_valid_target_date": panel.loc[panel["forward_target_valid"], "date"].max(),
    "median_forward_vol_pct": panel["forward_rv_21td_vol_pct"].median(),
    "min_forward_vol_pct": panel["forward_rv_21td_vol_pct"].min(),
    "max_forward_vol_pct": panel["forward_rv_21td_vol_pct"].max(),
    "model_row_candidates": int(panel["is_corsi_model_row_candidate"].sum()),
}])

feature_availability_by_group_rows = []

for group_name, cols in feature_groups.items():
    missing_group_cols = [c for c in cols if c not in panel.columns]

    if missing_group_cols:
        rows_all_features_available = 0
        rows_all_features_and_target_available = 0
        status = "MISSING_COLUMNS"
    else:
        rows_all_features_available = int(panel[cols].notna().all(axis=1).sum())
        rows_all_features_and_target_available = int(
            (panel[cols].notna().all(axis=1) & panel["is_corsi_target_available"]).sum()
        )
        status = "OK"

    feature_availability_by_group_rows.append({
        "feature_group": group_name,
        "feature_count": len(cols),
        "rows_all_features_available": rows_all_features_available,
        "rows_all_features_and_target_available": rows_all_features_and_target_available,
        "status": status,
        "missing_columns": ", ".join(missing_group_cols),
    })

feature_availability_by_group = pd.DataFrame(feature_availability_by_group_rows)

target_by_year = (
    panel.assign(year=panel["date"].dt.year)
    .groupby("year", dropna=False)
    .agg(
        rows=("date", "size"),
        target_valid_rows=("forward_target_valid", "sum"),
        median_forward_vol_pct=("forward_rv_21td_vol_pct", "median"),
        mean_forward_vol_pct=("forward_rv_21td_vol_pct", "mean"),
        max_forward_vol_pct=("forward_rv_21td_vol_pct", "max"),
    )
    .reset_index()
)

print()
print("Feature-target panel summary:")
display(pd.DataFrame([{
    "rows": len(panel),
    "first_date": panel["date"].min(),
    "last_date": panel["date"].max(),
    "baseline_core_trades": int(panel["baseline_core_signal"].sum()),
    "target_valid_rows": int(panel["forward_target_valid"].sum()),
    "model_row_candidates": int(panel["is_corsi_model_row_candidate"].sum()),
    "rv21d_recompute_max_abs_diff": rv21d_max_abs_diff,
    "vrp_recompute_max_abs_diff": vrp_max_abs_diff,
}]))


print()
print("Target diagnostics:")
display(target_diagnostics)

print()
print("Target diagnostics by year:")
display(target_by_year)

print()
print("Feature availability by proposed model group:")
display(feature_availability_by_group)

print()
print("Feature diagnostics:")
display(feature_diagnostics)

print()
print("Feature group manifest:")
display(feature_group_manifest)


# =============================================================================
# 3. Save outputs
# =============================================================================

safe_start = panel["date"].min().strftime("%Y%m%d")
safe_end = panel["date"].max().strftime("%Y%m%d")

feature_target_panel_path = (
    CORSI_PROCESSED_DIR
    / f"02_30d_corsi_feature_target_panel_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.parquet"
)

feature_target_panel_csv_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_feature_target_panel_preview_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

validation_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_feature_target_validation_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

feature_diagnostics_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_feature_diagnostics_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_diagnostics_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_target_diagnostics_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_by_year_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_target_by_year_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

feature_group_manifest_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_feature_group_manifest_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

feature_availability_by_group_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_feature_availability_by_group_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

canonicalization_path = (
    CORSI_AUDIT_DIR
    / f"02_30d_corsi_column_canonicalization_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

panel.to_parquet(feature_target_panel_path, index=False)

preview_cols = [
    "date",
    "trade_date",
    "vix_30d_vol_pct",
    "implied_variance_decimal",
    "baseline_rv21d_vol_pct",
    "baseline_rv21d_variance_decimal",
    "rv_std_21d",
    "rv21d_recomputed_diff",
    "baseline_vrp_log",
    "baseline_vrp_log_recomputed_from_cell2_rv",
    "baseline_vrp_log_recomputed_diff",
    "rv_1d",
    "rv_std_5d",
    "rv_std_21d",
    "rv_std_63d",
    "rv_mean_5d",
    "rv_mean_21d",
    "rv_mean_63d",
    "downside_rv_21d",
    "upside_rv_21d",
    "downside_share_21d",
    "negative_return_count_21d",
    "max_abs_return_21d",
    "drawdown_from_21d_high",
    "target_start_date",
    "target_end_date",
    "target_num_returns",
    "forward_rv_21td_variance_decimal",
    "forward_rv_21td_vol_pct",
    "log_forward_rv_21td_variance_decimal",
    "forward_target_valid",
    "baseline_core_signal",
    "outcome_value",
    "actual_dte",
    "pnl_per_day",
]

missing_preview_cols = [c for c in preview_cols if c not in panel.columns]

if missing_preview_cols:
    raise KeyError(f"Missing preview columns before save: {missing_preview_cols}")

panel[preview_cols].to_csv(feature_target_panel_csv_path, index=False)

validation.to_csv(validation_path, index=False)
feature_diagnostics.to_csv(feature_diagnostics_path, index=False)
target_diagnostics.to_csv(target_diagnostics_path, index=False)
target_by_year.to_csv(target_by_year_path, index=False)
feature_group_manifest.to_csv(feature_group_manifest_path, index=False)
feature_availability_by_group.to_csv(feature_availability_by_group_path, index=False)
canonicalization_summary.to_csv(canonicalization_path, index=False)

print()
print("Cell 2A outputs saved:")
print(f"Feature-target panel parquet:      {feature_target_panel_path}")
print(f"Feature-target preview CSV:        {feature_target_panel_csv_path}")
print(f"Validation checks:                 {validation_path}")
print(f"Feature diagnostics:               {feature_diagnostics_path}")
print(f"Target diagnostics:                {target_diagnostics_path}")
print(f"Target by year:                    {target_by_year_path}")
print(f"Feature group manifest:            {feature_group_manifest_path}")
print(f"Feature availability by group:     {feature_availability_by_group_path}")
print(f"Column canonicalization:           {canonicalization_path}")

print()
print("Cell 2A complete.")
print()
print("Next proposed cell:")
print("Cell 3: Fit walk-forward 30D Corsi/HAR candidate denominators.")
print("No threshold optimization in Cell 3; first apply the same baseline thresholds to model-derived VRP.")

Column canonicalization summary:


,canonical_column,available_source_columns,status
0,underlying_close,"underlying_close_y, underlying_close_x",RESTORED
1,daily_log_return,"daily_log_return_y, daily_log_return_x",RESTORED
2,daily_rv_1d,daily_rv_1d,RESTORED



Feature-target panel summary:


,rows,first_date,last_date,baseline_core_trades,target_valid_rows,model_row_candidates,rv21d_recompute_max_abs_diff,vrp_recompute_max_abs_diff
0,1612,2020-01-02,2026-06-02,177,1612,1606,2.220446e-16,4.996004e-16



Target diagnostics:


...


Feature diagnostics:


,column,non_null,null,inf_count,min,p01,median,mean,p99,max
0,daily_log_return,1612,0,0,-1.276521e-01,-0.035907,0.000994,0.000531,0.030114,0.090895
1,daily_rv_1d,1612,0,0,2.715917e-09,0.000001,0.008624,0.042384,0.514067,4.106357
2,downside_rv_21d,1612,0,0,4.219738e-04,0.000807,0.009159,0.022111,0.458210,0.566983
3,downside_share_21d,1612,0,0,3.927854e-02,0.081237,0.454657,0.448507,0.816215,0.924422
4,drawdown_from_21d_high,1612,0,0,-3.094390e-01,-0.162531,-0.008700,-0.022661,0.000000,0.000000
5,forward_rv_21td_variance_decimal,1612,0,0,2.935604e-03,0.004217,0.019697,0.042917,0.709638,0.951701
6,forward_rv_21td_vol_pct,1612,0,0,5.418121e+00,6.493861,14.034640,17.166148,84.239948,97.555150
7,log_forward_rv_21td_variance_decimal,1612,0,0,-5.830842e+00,-5.468632,-3.927283,-3.784255,-0.343003,-0.049505
8,log_return_21d,1612,0,0,-3.999824e-01,-0.168466,0.018948,0.011086,0.117765,0.224058
9,log_rv_std_21d_change_21d,1612,0,0,-2.361625e+00,-2.036493,-0.023868,0.009527,2.551620,3.749859



Feature group manifest:


,feature_group,feature
0,har_total_std_v1,log_rv_1d
1,har_total_std_v1,log_rv_std_5d
2,har_total_std_v1,log_rv_std_21d
3,har_total_std_v1,log_rv_std_63d
4,har_total_mean_v1,log_rv_1d
...,...,...
60,har_full_defensible_v1,std_daily_rv_21d
61,har_full_defensible_v1,log_return_5d
62,har_full_defensible_v1,log_return_21d
63,har_full_defensible_v1,drawdown_from_21d_high



Cell 2A outputs saved:
Feature-target panel parquet:      C:\Users\patri\vrp_project\data\processed\vrp_30d_corsi_v1\02_30d_corsi_feature_target_panel_20200102_20260602_20260704_130328.parquet
Feature-target preview CSV:        C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\02_30d_corsi_feature_target_panel_preview_20200102_20260602_20260704_130328.csv
Validation checks:                 C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\02_30d_corsi_feature_target_validation_20200102_20260602_20260704_130328.csv
Feature diagnostics:               C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\02_30d_corsi_feature_diagnostics_20200102_20260602_20260704_130328.csv
Target diagnostics:                C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\02_30d_corsi_target_diagnostics_20200102_20260602_20260704_130328.csv
Target by year:                    C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\02_30d_corsi_target_by_year_20200102_20260602_20260704_130

In [4]:
# Cell 2B: Build extended close-to-close Corsi training history
#
# Objective:
#   Create an extended feature/target panel from the full available close-source history.
#
# Why:
#   The frozen 30D baseline panel starts in 2020, but the close-source history starts in 2018.
#   Walk-forward Corsi models need pre-2020 history for training and for fair OOS forecasts.
#
# Scope:
#   - Reload baseline manifest
#   - Reload original close / RSI source
#   - Rebuild the same feature library from the full close-source date range
#   - Rebuild the same 21-trading-day forward realized variance target
#   - Load the saved Cell 2 feature-target panel
#   - Verify overlapping feature/target columns match exactly or within floating tolerance
#   - Save extended training history
#
# Explicitly NOT doing:
#   - No model fitting
#   - No Ridge regression
#   - No forecast generation
#   - No VRP signal creation
#   - No threshold testing
#   - No optimization

from pathlib import Path
import json
import numpy as np
import pandas as pd


# =============================================================================
# 0. Paths and constants
# =============================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

BASELINE_BRANCH = "vrp_30d_baseline_v1"
CORSI_BRANCH = "vrp_30d_corsi_v1"

BASELINE_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / BASELINE_BRANCH

CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / CORSI_BRANCH
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / CORSI_BRANCH

CORSI_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CORSI_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

TRADING_DAYS_PER_YEAR = 252
RV_DDOF = 1
TARGET_FORWARD_DAYS = 21
VAR_FLOOR = 1e-10

print("Cell 2B: Build extended close-to-close Corsi training history")
print(f"Project root:       {PROJECT_ROOT}")
print(f"Baseline audit:     {BASELINE_AUDIT_DIR}")
print(f"Corsi processed:    {CORSI_PROCESSED_DIR}")
print(f"Corsi audit:        {CORSI_AUDIT_DIR}")
print(f"Run timestamp:      {RUN_TIMESTAMP}")


# =============================================================================
# 1. Helpers
# =============================================================================

def latest_file(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f"No files found in {directory} matching pattern: {pattern}")
    return matches[0]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in [".csv", ".txt"]:
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file extension: {path}")


def first_existing_col(df: pd.DataFrame, candidates):
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c

    lower_map = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if str(c).lower() in lower_map:
            return lower_map[str(c).lower()]
    return None


def date_to_trade_date(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series).dt.strftime("%Y%m%d").astype("Int64")


def safe_div(num, den):
    out = num / den
    return out.replace([np.inf, -np.inf], np.nan)


def safe_log(x, floor=VAR_FLOOR):
    return np.log(np.maximum(x, floor))


def rolling_sample_variance_annualized(returns: pd.Series, window: int) -> pd.Series:
    return returns.rolling(window=window, min_periods=window).std(ddof=RV_DDOF).pow(2) * TRADING_DAYS_PER_YEAR


def rolling_mean_squared_return_annualized(returns: pd.Series, window: int) -> pd.Series:
    return returns.pow(2).rolling(window=window, min_periods=window).mean() * TRADING_DAYS_PER_YEAR


def rolling_semivariance_annualized(returns: pd.Series, window: int, side: str) -> pd.Series:
    if side == "downside":
        semi = np.where(returns < 0, returns.pow(2), 0.0)
    elif side == "upside":
        semi = np.where(returns > 0, returns.pow(2), 0.0)
    else:
        raise ValueError("side must be 'downside' or 'upside'")

    return pd.Series(semi, index=returns.index).rolling(window=window, min_periods=window).mean() * TRADING_DAYS_PER_YEAR


def diagnostic_summary(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []

    for col in cols:
        if col not in df.columns:
            rows.append({
                "column": col,
                "status": "MISSING",
                "non_null": 0,
                "null": len(df),
                "inf_count": np.nan,
                "min": np.nan,
                "p01": np.nan,
                "median": np.nan,
                "mean": np.nan,
                "p99": np.nan,
                "max": np.nan,
            })
            continue

        s = pd.to_numeric(df[col], errors="coerce")
        finite = s.replace([np.inf, -np.inf], np.nan).dropna()

        rows.append({
            "column": col,
            "status": "OK",
            "non_null": int(s.notna().sum()),
            "null": int(s.isna().sum()),
            "inf_count": int(np.isinf(s.dropna()).sum()),
            "min": float(finite.min()) if len(finite) else np.nan,
            "p01": float(finite.quantile(0.01)) if len(finite) else np.nan,
            "median": float(finite.median()) if len(finite) else np.nan,
            "mean": float(finite.mean()) if len(finite) else np.nan,
            "p99": float(finite.quantile(0.99)) if len(finite) else np.nan,
            "max": float(finite.max()) if len(finite) else np.nan,
        })

    return pd.DataFrame(rows)


def add_validation(rows, check_name, passed, details):
    rows.append({
        "check": check_name,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": details,
    })


# =============================================================================
# 2. Load baseline manifest and close source
# =============================================================================

baseline_manifest_path = latest_file(
    BASELINE_AUDIT_DIR,
    "00_excel_30d_vrp_baseline_source_manifest_*.csv",
)

baseline_manifest = normalize_columns(pd.read_csv(baseline_manifest_path))

feature_manifest = baseline_manifest.loc[
    baseline_manifest["role"].astype(str).eq("close_rsi_features")
].copy()

if len(feature_manifest) != 1:
    raise ValueError(f"Expected exactly one close_rsi_features manifest row, found {len(feature_manifest)}.")

feature_source_path = Path(feature_manifest.iloc[0]["path"])

feature_raw = normalize_columns(read_table(feature_source_path))

date_col = first_existing_col(feature_raw, ["date", "quote_date"])
trade_date_col = first_existing_col(feature_raw, ["trade_date"])

close_col = first_existing_col(
    feature_raw,
    [
        "spx_close",
        "spy_close",
        "underlying_close",
        "adj_close",
        "adjusted_close",
        "close",
        "close_price",
        "spy_adj_close",
    ],
)

rsi_col = first_existing_col(feature_raw, ["rsi14", "rsi_14", "rsi"])

if date_col is None and trade_date_col is None:
    raise ValueError(f"Close source missing date/trade_date column: {feature_source_path}")

if close_col is None:
    raise ValueError(f"Close source missing close column: {feature_source_path}")

feature = feature_raw.copy()

if date_col is not None:
    feature["date"] = pd.to_datetime(feature[date_col], errors="coerce").dt.normalize()
elif trade_date_col is not None:
    feature["date"] = pd.to_datetime(
        feature[trade_date_col].astype(str),
        format="%Y%m%d",
        errors="coerce",
    ).dt.normalize()

if trade_date_col is not None:
    feature["trade_date"] = pd.to_numeric(feature[trade_date_col], errors="coerce").astype("Int64")
else:
    feature["trade_date"] = date_to_trade_date(feature["date"])

feature["underlying_close"] = pd.to_numeric(feature[close_col], errors="coerce")

if rsi_col is not None:
    feature["rsi14_source"] = pd.to_numeric(feature[rsi_col], errors="coerce")
else:
    feature["rsi14_source"] = np.nan

feature = (
    feature[["date", "trade_date", "underlying_close", "rsi14_source"]]
    .dropna(subset=["date", "trade_date", "underlying_close"])
    .drop_duplicates("trade_date")
    .sort_values("date")
    .reset_index(drop=True)
)

print()
print("Close / RSI source selected for extended history:")
print(feature_source_path)
print(f"Close column used: {close_col}")
print(f"RSI column used:   {rsi_col}")

print()
print("Extended close-source summary:")
display(pd.DataFrame([{
    "rows": len(feature),
    "first_date": feature["date"].min(),
    "last_date": feature["date"].max(),
    "close_non_null": int(feature["underlying_close"].notna().sum()),
    "rsi_source_non_null": int(feature["rsi14_source"].notna().sum()),
    "duplicate_trade_dates": int(feature["trade_date"].duplicated().sum()),
}]))


# =============================================================================
# 3. Build extended feature library
# =============================================================================

feature["daily_log_return"] = np.log(feature["underlying_close"] / feature["underlying_close"].shift(1))
feature["daily_rv_1d"] = feature["daily_log_return"].pow(2) * TRADING_DAYS_PER_YEAR

feature["return_formula_check"] = (
    feature["daily_log_return"]
    - np.log(feature["underlying_close"] / feature["underlying_close"].shift(1))
)

# Excel-style sample variance features.
feature["rv_1d"] = feature["daily_rv_1d"]

for window in [5, 21, 63]:
    feature[f"rv_std_{window}d"] = rolling_sample_variance_annualized(
        feature["daily_log_return"],
        window=window,
    )

feature["rv_std_21d_vol_pct"] = np.sqrt(feature["rv_std_21d"]) * 100.0

# HAR mean-squared-return features.
for window in [5, 21, 63]:
    feature[f"rv_mean_{window}d"] = rolling_mean_squared_return_annualized(
        feature["daily_log_return"],
        window=window,
    )

# Log variance features.
log_variance_cols = [
    "rv_1d",
    "rv_std_5d",
    "rv_std_21d",
    "rv_std_63d",
    "rv_mean_5d",
    "rv_mean_21d",
    "rv_mean_63d",
]

for col in log_variance_cols:
    feature[f"log_{col}"] = safe_log(feature[col])

# Acceleration / slope.
feature["rv_std_5d_over_21d"] = safe_div(feature["rv_std_5d"], feature["rv_std_21d"])
feature["rv_std_21d_over_63d"] = safe_div(feature["rv_std_21d"], feature["rv_std_63d"])

feature["rv_mean_5d_over_21d"] = safe_div(feature["rv_mean_5d"], feature["rv_mean_21d"])
feature["rv_mean_21d_over_63d"] = safe_div(feature["rv_mean_21d"], feature["rv_mean_63d"])

feature["log_rv_std_5d_minus_21d"] = feature["log_rv_std_5d"] - feature["log_rv_std_21d"]
feature["log_rv_std_21d_minus_63d"] = feature["log_rv_std_21d"] - feature["log_rv_std_63d"]

feature["rv_std_21d_change_5d"] = feature["rv_std_21d"] - feature["rv_std_21d"].shift(5)
feature["rv_std_21d_change_21d"] = feature["rv_std_21d"] - feature["rv_std_21d"].shift(21)

feature["log_rv_std_21d_change_5d"] = feature["log_rv_std_21d"] - feature["log_rv_std_21d"].shift(5)
feature["log_rv_std_21d_change_21d"] = feature["log_rv_std_21d"] - feature["log_rv_std_21d"].shift(21)

# Shock / instability.
feature["abs_daily_log_return"] = feature["daily_log_return"].abs()

feature["max_abs_return_5d"] = feature["abs_daily_log_return"].rolling(5, min_periods=5).max()
feature["max_abs_return_21d"] = feature["abs_daily_log_return"].rolling(21, min_periods=21).max()
feature["max_abs_return_63d"] = feature["abs_daily_log_return"].rolling(63, min_periods=63).max()

feature["rv_1d_over_std_21d"] = safe_div(feature["rv_1d"], feature["rv_std_21d"])

feature["rv_5d_max_daily_rv"] = feature["daily_rv_1d"].rolling(5, min_periods=5).max()
feature["rv_21d_max_daily_rv"] = feature["daily_rv_1d"].rolling(21, min_periods=21).max()
feature["rv_63d_max_daily_rv"] = feature["daily_rv_1d"].rolling(63, min_periods=63).max()

feature["std_daily_rv_21d"] = feature["daily_rv_1d"].rolling(21, min_periods=21).std(ddof=RV_DDOF)
feature["std_daily_rv_63d"] = feature["daily_rv_1d"].rolling(63, min_periods=63).std(ddof=RV_DDOF)

# Downside / upside.
for window in [5, 21, 63]:
    feature[f"downside_rv_{window}d"] = rolling_semivariance_annualized(
        feature["daily_log_return"],
        window=window,
        side="downside",
    )

    feature[f"upside_rv_{window}d"] = rolling_semivariance_annualized(
        feature["daily_log_return"],
        window=window,
        side="upside",
    )

    feature[f"negative_return_count_{window}d"] = (
        feature["daily_log_return"].lt(0)
        .rolling(window=window, min_periods=window)
        .sum()
    )

    feature[f"downside_share_{window}d"] = safe_div(
        feature[f"downside_rv_{window}d"],
        feature[f"downside_rv_{window}d"] + feature[f"upside_rv_{window}d"],
    )

for col in [
    "downside_rv_5d",
    "downside_rv_21d",
    "downside_rv_63d",
    "upside_rv_5d",
    "upside_rv_21d",
    "upside_rv_63d",
]:
    feature[f"log_{col}"] = safe_log(feature[col])

# Trend / drawdown.
for window in [5, 21, 63]:
    feature[f"log_return_{window}d"] = np.log(
        feature["underlying_close"] / feature["underlying_close"].shift(window)
    )

    rolling_high = feature["underlying_close"].rolling(window=window, min_periods=window).max()
    feature[f"drawdown_from_{window}d_high"] = feature["underlying_close"] / rolling_high - 1.0


# =============================================================================
# 4. Build extended forward target
# =============================================================================

future_return_cols = []

for i in range(1, TARGET_FORWARD_DAYS + 1):
    col = f"fwd_return_{i}"
    feature[col] = feature["daily_log_return"].shift(-i)
    future_return_cols.append(col)

future_returns = feature[future_return_cols]
target_valid = future_returns.notna().all(axis=1)

feature["target_start_date"] = feature["date"].shift(-1)
feature["target_end_date"] = feature["date"].shift(-TARGET_FORWARD_DAYS)
feature["target_start_trade_date"] = feature["trade_date"].shift(-1)
feature["target_end_trade_date"] = feature["trade_date"].shift(-TARGET_FORWARD_DAYS)
feature["target_num_returns"] = future_returns.notna().sum(axis=1)

feature["forward_rv_21td_variance_decimal"] = np.where(
    target_valid,
    future_returns.var(axis=1, ddof=RV_DDOF) * TRADING_DAYS_PER_YEAR,
    np.nan,
)

feature["forward_rv_21td_vol_pct"] = np.sqrt(feature["forward_rv_21td_variance_decimal"]) * 100.0

feature["forward_log_return_21td"] = np.where(
    target_valid,
    future_returns.sum(axis=1),
    np.nan,
)

feature["forward_target_valid"] = target_valid
feature["log_forward_rv_21td_variance_decimal"] = safe_log(feature["forward_rv_21td_variance_decimal"])


# =============================================================================
# 5. Define feature groups
# =============================================================================

feature_groups = {
    "har_total_std_v1": [
        "log_rv_1d",
        "log_rv_std_5d",
        "log_rv_std_21d",
        "log_rv_std_63d",
    ],
    "har_total_mean_v1": [
        "log_rv_1d",
        "log_rv_mean_5d",
        "log_rv_mean_21d",
        "log_rv_mean_63d",
    ],
    "har_total_accel_v1": [
        "log_rv_1d",
        "log_rv_std_5d",
        "log_rv_std_21d",
        "log_rv_std_63d",
        "rv_std_5d_over_21d",
        "rv_std_21d_over_63d",
        "log_rv_std_5d_minus_21d",
        "log_rv_std_21d_minus_63d",
        "log_rv_std_21d_change_5d",
        "log_rv_std_21d_change_21d",
    ],
    "har_downside_v1": [
        "log_rv_1d",
        "log_rv_std_5d",
        "log_rv_std_21d",
        "log_rv_std_63d",
        "log_downside_rv_5d",
        "log_downside_rv_21d",
        "log_downside_rv_63d",
        "downside_share_21d",
        "negative_return_count_21d",
    ],
    "har_regime_v1": [
        "log_rv_1d",
        "log_rv_std_5d",
        "log_rv_std_21d",
        "log_rv_std_63d",
        "max_abs_return_5d",
        "max_abs_return_21d",
        "rv_1d_over_std_21d",
        "std_daily_rv_21d",
        "log_return_5d",
        "log_return_21d",
        "drawdown_from_21d_high",
        "drawdown_from_63d_high",
    ],
    "har_full_defensible_v1": [
        "log_rv_1d",
        "log_rv_std_5d",
        "log_rv_std_21d",
        "log_rv_std_63d",
        "log_rv_mean_5d",
        "log_rv_mean_21d",
        "log_rv_mean_63d",
        "rv_std_5d_over_21d",
        "rv_std_21d_over_63d",
        "log_rv_std_5d_minus_21d",
        "log_rv_std_21d_minus_63d",
        "log_rv_std_21d_change_5d",
        "log_rv_std_21d_change_21d",
        "log_downside_rv_5d",
        "log_downside_rv_21d",
        "log_downside_rv_63d",
        "downside_share_21d",
        "negative_return_count_21d",
        "max_abs_return_5d",
        "max_abs_return_21d",
        "rv_1d_over_std_21d",
        "std_daily_rv_21d",
        "log_return_5d",
        "log_return_21d",
        "drawdown_from_21d_high",
        "drawdown_from_63d_high",
    ],
}

feature_group_rows = []

for group_name, cols in feature_groups.items():
    for col in cols:
        feature_group_rows.append({
            "feature_group": group_name,
            "feature": col,
        })

feature_group_manifest = pd.DataFrame(feature_group_rows)


# =============================================================================
# 6. Build extended panel columns
# =============================================================================

extended_cols = [
    "date",
    "trade_date",
    "underlying_close",
    "rsi14_source",
    "daily_log_return",
    "daily_rv_1d",

    "rv_1d",
    "rv_std_5d",
    "rv_std_21d",
    "rv_std_63d",
    "rv_std_21d_vol_pct",

    "rv_mean_5d",
    "rv_mean_21d",
    "rv_mean_63d",

    "log_rv_1d",
    "log_rv_std_5d",
    "log_rv_std_21d",
    "log_rv_std_63d",
    "log_rv_mean_5d",
    "log_rv_mean_21d",
    "log_rv_mean_63d",

    "rv_std_5d_over_21d",
    "rv_std_21d_over_63d",
    "rv_mean_5d_over_21d",
    "rv_mean_21d_over_63d",
    "log_rv_std_5d_minus_21d",
    "log_rv_std_21d_minus_63d",
    "rv_std_21d_change_5d",
    "rv_std_21d_change_21d",
    "log_rv_std_21d_change_5d",
    "log_rv_std_21d_change_21d",

    "abs_daily_log_return",
    "max_abs_return_5d",
    "max_abs_return_21d",
    "max_abs_return_63d",
    "rv_1d_over_std_21d",
    "rv_5d_max_daily_rv",
    "rv_21d_max_daily_rv",
    "rv_63d_max_daily_rv",
    "std_daily_rv_21d",
    "std_daily_rv_63d",

    "downside_rv_5d",
    "downside_rv_21d",
    "downside_rv_63d",
    "upside_rv_5d",
    "upside_rv_21d",
    "upside_rv_63d",
    "log_downside_rv_5d",
    "log_downside_rv_21d",
    "log_downside_rv_63d",
    "log_upside_rv_5d",
    "log_upside_rv_21d",
    "log_upside_rv_63d",
    "negative_return_count_5d",
    "negative_return_count_21d",
    "negative_return_count_63d",
    "downside_share_5d",
    "downside_share_21d",
    "downside_share_63d",

    "log_return_5d",
    "log_return_21d",
    "log_return_63d",
    "drawdown_from_5d_high",
    "drawdown_from_21d_high",
    "drawdown_from_63d_high",

    "target_start_date",
    "target_end_date",
    "target_start_trade_date",
    "target_end_trade_date",
    "target_num_returns",
    "forward_rv_21td_variance_decimal",
    "forward_rv_21td_vol_pct",
    "forward_log_return_21td",
    "forward_target_valid",
    "log_forward_rv_21td_variance_decimal",
]

extended_panel = feature[extended_cols].copy()

extended_panel["is_extended_target_available"] = extended_panel["forward_rv_21td_variance_decimal"].notna()

base_feature_cols = [
    "log_rv_1d",
    "log_rv_std_5d",
    "log_rv_std_21d",
    "log_rv_std_63d",
    "log_rv_mean_5d",
    "log_rv_mean_21d",
    "log_rv_mean_63d",
]

extended_panel["is_extended_base_feature_available"] = extended_panel[base_feature_cols].notna().all(axis=1)
extended_panel["is_extended_model_row_candidate"] = (
    extended_panel["is_extended_base_feature_available"]
    & extended_panel["is_extended_target_available"]
)


# =============================================================================
# 7. Verify overlap against saved Cell 2 panel
# =============================================================================

cell2_panel_path = latest_file(
    CORSI_PROCESSED_DIR,
    "02_30d_corsi_feature_target_panel_*.parquet",
)

cell2_panel = normalize_columns(pd.read_parquet(cell2_panel_path))
cell2_panel["date"] = pd.to_datetime(cell2_panel["date"], errors="coerce").dt.normalize()
cell2_panel["trade_date"] = pd.to_numeric(cell2_panel["trade_date"], errors="coerce").astype("Int64")

compare_cols = [
    "underlying_close",
    "daily_log_return",
    "daily_rv_1d",
    "rv_1d",
    "rv_std_5d",
    "rv_std_21d",
    "rv_std_63d",
    "rv_mean_5d",
    "rv_mean_21d",
    "rv_mean_63d",
    "log_rv_1d",
    "log_rv_std_5d",
    "log_rv_std_21d",
    "log_rv_std_63d",
    "downside_rv_21d",
    "upside_rv_21d",
    "downside_share_21d",
    "negative_return_count_21d",
    "max_abs_return_21d",
    "drawdown_from_21d_high",
    "forward_rv_21td_variance_decimal",
    "forward_rv_21td_vol_pct",
    "log_forward_rv_21td_variance_decimal",
]

overlap = cell2_panel[["date", "trade_date"] + compare_cols].merge(
    extended_panel[["date", "trade_date"] + compare_cols],
    on=["date", "trade_date"],
    how="inner",
    suffixes=("_cell2", "_extended"),
    validate="one_to_one",
)

overlap_rows = []

for col in compare_cols:
    a = pd.to_numeric(overlap[f"{col}_cell2"], errors="coerce")
    b = pd.to_numeric(overlap[f"{col}_extended"], errors="coerce")

    both_non_null = a.notna() & b.notna()
    null_mismatch = int((a.isna() != b.isna()).sum())

    if both_non_null.any():
        max_abs_diff = float((a.loc[both_non_null] - b.loc[both_non_null]).abs().max())
    else:
        max_abs_diff = np.nan

    overlap_rows.append({
        "column": col,
        "overlap_rows": len(overlap),
        "both_non_null_rows": int(both_non_null.sum()),
        "null_mismatch_count": null_mismatch,
        "max_abs_diff": max_abs_diff,
        "status": "PASS" if null_mismatch == 0 and (pd.isna(max_abs_diff) or max_abs_diff < 1e-12) else "FAIL",
    })

overlap_comparison = pd.DataFrame(overlap_rows)


# =============================================================================
# 8. Diagnostics and validations
# =============================================================================

validation_rows = []

add_validation(
    validation_rows,
    "Extended close source loaded",
    len(feature) > 0,
    f"path={feature_source_path}; rows={len(feature)}",
)

add_validation(
    validation_rows,
    "No duplicate trade_date rows",
    extended_panel["trade_date"].duplicated().sum() == 0,
    f"duplicate_trade_dates={int(extended_panel['trade_date'].duplicated().sum())}",
)

max_return_formula_diff = feature["return_formula_check"].dropna().abs().max()

add_validation(
    validation_rows,
    "Return formula valid",
    max_return_formula_diff == 0,
    f"max_abs_return_formula_diff={max_return_formula_diff}",
)

target_timing_valid = (
    extended_panel.loc[extended_panel["forward_target_valid"], "target_start_date"].gt(
        extended_panel.loc[extended_panel["forward_target_valid"], "date"]
    ).all()
    and extended_panel.loc[extended_panel["forward_target_valid"], "target_end_date"].gt(
        extended_panel.loc[extended_panel["forward_target_valid"], "date"]
    ).all()
)

add_validation(
    validation_rows,
    "Forward target uses future dates only",
    target_timing_valid,
    "target_start_date and target_end_date are both greater than signal date for valid targets",
)

add_validation(
    validation_rows,
    "Forward target has exactly 21 returns when valid",
    extended_panel.loc[extended_panel["forward_target_valid"], "target_num_returns"].eq(TARGET_FORWARD_DAYS).all(),
    f"target_forward_days={TARGET_FORWARD_DAYS}",
)

numeric_cols = extended_panel.select_dtypes(include=[np.number]).columns.tolist()
inf_count_total = int(np.isinf(extended_panel[numeric_cols]).sum().sum())

add_validation(
    validation_rows,
    "No infinite numeric values in extended panel",
    inf_count_total == 0,
    f"total_inf_count={inf_count_total}",
)

add_validation(
    validation_rows,
    "Overlap with saved Cell 2 panel exists",
    len(overlap) == len(cell2_panel),
    f"overlap_rows={len(overlap)}; cell2_rows={len(cell2_panel)}",
)

add_validation(
    validation_rows,
    "Extended overlap matches saved Cell 2 panel",
    overlap_comparison["status"].eq("PASS").all(),
    f"failed_columns={overlap_comparison.loc[overlap_comparison['status'].ne('PASS'), 'column'].tolist()}",
)

add_validation(
    validation_rows,
    "Extended model row candidates available",
    extended_panel["is_extended_model_row_candidate"].sum() > 0,
    f"candidate_rows={int(extended_panel['is_extended_model_row_candidate'].sum())}",
)

validation = pd.DataFrame(validation_rows)


feature_cols_for_diag = sorted(set(
    [
        "daily_log_return",
        "daily_rv_1d",
        "rv_1d",
        "rv_std_5d",
        "rv_std_21d",
        "rv_std_63d",
        "rv_mean_5d",
        "rv_mean_21d",
        "rv_mean_63d",
        "rv_std_5d_over_21d",
        "rv_std_21d_over_63d",
        "log_rv_std_21d_change_5d",
        "log_rv_std_21d_change_21d",
        "downside_rv_21d",
        "upside_rv_21d",
        "downside_share_21d",
        "negative_return_count_21d",
        "max_abs_return_21d",
        "rv_1d_over_std_21d",
        "std_daily_rv_21d",
        "log_return_21d",
        "drawdown_from_21d_high",
        "forward_rv_21td_variance_decimal",
        "forward_rv_21td_vol_pct",
        "log_forward_rv_21td_variance_decimal",
    ]
))

feature_diagnostics = diagnostic_summary(extended_panel, feature_cols_for_diag)

target_diagnostics = pd.DataFrame([{
    "rows": len(extended_panel),
    "first_date": extended_panel["date"].min(),
    "last_date": extended_panel["date"].max(),
    "target_valid_rows": int(extended_panel["forward_target_valid"].sum()),
    "target_missing_rows": int(extended_panel["forward_rv_21td_variance_decimal"].isna().sum()),
    "first_valid_target_date": extended_panel.loc[extended_panel["forward_target_valid"], "date"].min(),
    "last_valid_target_date": extended_panel.loc[extended_panel["forward_target_valid"], "date"].max(),
    "median_forward_vol_pct": extended_panel["forward_rv_21td_vol_pct"].median(),
    "min_forward_vol_pct": extended_panel["forward_rv_21td_vol_pct"].min(),
    "max_forward_vol_pct": extended_panel["forward_rv_21td_vol_pct"].max(),
    "extended_model_row_candidates": int(extended_panel["is_extended_model_row_candidate"].sum()),
}])

target_by_year = (
    extended_panel.assign(year=extended_panel["date"].dt.year)
    .groupby("year", dropna=False)
    .agg(
        rows=("date", "size"),
        target_valid_rows=("forward_target_valid", "sum"),
        model_row_candidates=("is_extended_model_row_candidate", "sum"),
        median_forward_vol_pct=("forward_rv_21td_vol_pct", "median"),
        mean_forward_vol_pct=("forward_rv_21td_vol_pct", "mean"),
        max_forward_vol_pct=("forward_rv_21td_vol_pct", "max"),
    )
    .reset_index()
)

feature_availability_by_group_rows = []

for group_name, cols in feature_groups.items():
    missing_group_cols = [c for c in cols if c not in extended_panel.columns]

    if missing_group_cols:
        rows_all_features_available = 0
        rows_all_features_and_target_available = 0
        status = "MISSING_COLUMNS"
    else:
        rows_all_features_available = int(extended_panel[cols].notna().all(axis=1).sum())
        rows_all_features_and_target_available = int(
            (extended_panel[cols].notna().all(axis=1) & extended_panel["is_extended_target_available"]).sum()
        )
        status = "OK"

    feature_availability_by_group_rows.append({
        "feature_group": group_name,
        "feature_count": len(cols),
        "rows_all_features_available": rows_all_features_available,
        "rows_all_features_and_target_available": rows_all_features_and_target_available,
        "status": status,
        "missing_columns": ", ".join(missing_group_cols),
    })

feature_availability_by_group = pd.DataFrame(feature_availability_by_group_rows)


print()
print("Cell 2B validation checks:")
display(validation)

if validation["status"].eq("FAIL").any():
    print()
    print("FAILED VALIDATION CHECKS:")
    display(validation.loc[validation["status"].eq("FAIL")])
    print()
    print("Overlap comparison failures:")
    display(overlap_comparison.loc[overlap_comparison["status"].ne("PASS")])
    raise ValueError("Cell 2B validation failed. Stop before model fitting.")

print()
print("Extended feature-target panel summary:")
display(pd.DataFrame([{
    "rows": len(extended_panel),
    "first_date": extended_panel["date"].min(),
    "last_date": extended_panel["date"].max(),
    "target_valid_rows": int(extended_panel["forward_target_valid"].sum()),
    "target_missing_rows": int(extended_panel["forward_rv_21td_variance_decimal"].isna().sum()),
    "extended_model_row_candidates": int(extended_panel["is_extended_model_row_candidate"].sum()),
    "cell2_overlap_rows": len(overlap),
}]))


print()
print("Extended target diagnostics:")
display(target_diagnostics)

print()
print("Extended target diagnostics by year:")
display(target_by_year)

print()
print("Extended feature availability by proposed model group:")
display(feature_availability_by_group)

print()
print("Overlap comparison versus saved Cell 2 panel:")
display(overlap_comparison)

print()
print("Extended feature diagnostics:")
display(feature_diagnostics)


# =============================================================================
# 9. Save outputs
# =============================================================================

safe_start = extended_panel["date"].min().strftime("%Y%m%d")
safe_end = extended_panel["date"].max().strftime("%Y%m%d")

extended_panel_path = (
    CORSI_PROCESSED_DIR
    / f"02B_30d_corsi_extended_feature_target_panel_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.parquet"
)

extended_preview_path = (
    CORSI_AUDIT_DIR
    / f"02B_30d_corsi_extended_feature_target_panel_preview_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

validation_path = (
    CORSI_AUDIT_DIR
    / f"02B_30d_corsi_extended_validation_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

feature_diagnostics_path = (
    CORSI_AUDIT_DIR
    / f"02B_30d_corsi_extended_feature_diagnostics_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_diagnostics_path = (
    CORSI_AUDIT_DIR
    / f"02B_30d_corsi_extended_target_diagnostics_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

target_by_year_path = (
    CORSI_AUDIT_DIR
    / f"02B_30d_corsi_extended_target_by_year_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

feature_group_manifest_path = (
    CORSI_AUDIT_DIR
    / f"02B_30d_corsi_extended_feature_group_manifest_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

feature_availability_by_group_path = (
    CORSI_AUDIT_DIR
    / f"02B_30d_corsi_extended_feature_availability_by_group_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

overlap_comparison_path = (
    CORSI_AUDIT_DIR
    / f"02B_30d_corsi_extended_overlap_comparison_vs_cell2_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

extended_panel.to_parquet(extended_panel_path, index=False)

preview_cols = [
    "date",
    "trade_date",
    "underlying_close",
    "rsi14_source",
    "daily_log_return",
    "rv_std_21d",
    "rv_std_21d_vol_pct",
    "rv_mean_21d",
    "downside_rv_21d",
    "upside_rv_21d",
    "downside_share_21d",
    "negative_return_count_21d",
    "max_abs_return_21d",
    "drawdown_from_21d_high",
    "target_start_date",
    "target_end_date",
    "target_num_returns",
    "forward_rv_21td_variance_decimal",
    "forward_rv_21td_vol_pct",
    "log_forward_rv_21td_variance_decimal",
    "forward_target_valid",
    "is_extended_model_row_candidate",
]

extended_panel[preview_cols].to_csv(extended_preview_path, index=False)

validation.to_csv(validation_path, index=False)
feature_diagnostics.to_csv(feature_diagnostics_path, index=False)
target_diagnostics.to_csv(target_diagnostics_path, index=False)
target_by_year.to_csv(target_by_year_path, index=False)
feature_group_manifest.to_csv(feature_group_manifest_path, index=False)
feature_availability_by_group.to_csv(feature_availability_by_group_path, index=False)
overlap_comparison.to_csv(overlap_comparison_path, index=False)

print()
print("Cell 2B outputs saved:")
print(f"Extended feature-target panel parquet:    {extended_panel_path}")
print(f"Extended feature-target preview CSV:      {extended_preview_path}")
print(f"Validation checks:                        {validation_path}")
print(f"Feature diagnostics:                      {feature_diagnostics_path}")
print(f"Target diagnostics:                       {target_diagnostics_path}")
print(f"Target by year:                           {target_by_year_path}")
print(f"Feature group manifest:                   {feature_group_manifest_path}")
print(f"Feature availability by group:            {feature_availability_by_group_path}")
print(f"Overlap comparison vs Cell 2:             {overlap_comparison_path}")

print()
print("Cell 2B complete.")
print()
print("Next proposed cell:")
print("Cell 3: Fit walk-forward 30D Corsi/HAR candidate denominators.")
print("No threshold optimization in Cell 3; first apply the same baseline thresholds to model-derived VRP.")

Cell 2B: Build extended close-to-close Corsi training history
Project root:       C:\Users\patri\vrp_project
Baseline audit:     C:\Users\patri\vrp_project\data\audit\vrp_30d_baseline_v1
Corsi processed:    C:\Users\patri\vrp_project\data\processed\vrp_30d_corsi_v1
Corsi audit:        C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1
Run timestamp:      20260704_130914

Close / RSI source selected for extended history:
C:\Users\patri\vrp_project\data\processed\staging\production_feature_panel_v0_1_candidate_20180625_20260702.parquet
Close column used: spx_close
RSI column used:   rsi14

Extended close-source summary:


,rows,first_date,last_date,close_non_null,rsi_source_non_null,duplicate_trade_dates
0,2016,2018-06-25,2026-07-02,2016,2016,0



Cell 2B validation checks:


,check,status,details
0,Extended close source loaded,PASS,path=C:\Users\patri\vrp_project\data\processed...
1,No duplicate trade_date rows,PASS,duplicate_trade_dates=0
2,Return formula valid,PASS,max_abs_return_formula_diff=0.0
3,Forward target uses future dates only,PASS,target_start_date and target_end_date are both...
4,Forward target has exactly 21 returns when valid,PASS,target_forward_days=21
5,No infinite numeric values in extended panel,PASS,total_inf_count=0
6,Overlap with saved Cell 2 panel exists,PASS,overlap_rows=1612; cell2_rows=1612
7,Extended overlap matches saved Cell 2 panel,PASS,failed_columns=[]
8,Extended model row candidates available,PASS,candidate_rows=1932



Extended feature-target panel summary:


...


Overlap comparison versus saved Cell 2 panel:


,column,overlap_rows,both_non_null_rows,null_mismatch_count,max_abs_diff,status
0,underlying_close,1612,1612,0,0.0,PASS
1,daily_log_return,1612,1612,0,0.0,PASS
2,daily_rv_1d,1612,1612,0,0.0,PASS
3,rv_1d,1612,1612,0,0.0,PASS
4,rv_std_5d,1612,1612,0,0.0,PASS
5,rv_std_21d,1612,1612,0,0.0,PASS
6,rv_std_63d,1612,1612,0,0.0,PASS
7,rv_mean_5d,1612,1612,0,0.0,PASS
8,rv_mean_21d,1612,1612,0,0.0,PASS
9,rv_mean_63d,1612,1612,0,0.0,PASS



Extended feature diagnostics:


,column,status,non_null,null,inf_count,min,p01,median,mean,p99,max
0,daily_log_return,OK,2015,1,0,-1.276521e-01,-0.034268,0.000903,0.000503,0.029326,0.090895
1,daily_rv_1d,OK,2015,1,0,4.500096e-10,0.000001,0.007997,0.038144,0.494763,4.106357
2,downside_rv_21d,OK,1996,20,0,3.348760e-04,0.000545,0.008710,0.020101,0.309456,0.566983
3,downside_share_21d,OK,1996,20,0,3.927854e-02,0.084706,0.450531,0.449202,0.819704,0.924422
4,drawdown_from_21d_high,OK,1996,20,0,-3.094390e-01,-0.149248,-0.008546,-0.021925,0.000000,0.000000
5,forward_rv_21td_variance_decimal,OK,1995,21,0,2.721716e-03,0.003394,0.018485,0.038675,0.622618,0.951701
6,forward_rv_21td_vol_pct,OK,1995,21,0,5.217007e+00,5.825891,13.595818,16.341240,78.906068,97.555150
7,log_forward_rv_21td_variance_decimal,OK,1995,21,0,-5.906493e+00,-5.685718,-3.990816,-3.890004,-0.473826,-0.049505
8,log_return_21d,OK,1995,21,0,-3.999824e-01,-0.135878,0.018528,0.010408,0.115122,0.224058
9,log_rv_std_21d_change_21d,OK,1974,42,0,-2.361625e+00,-1.997040,-0.023184,0.013863,2.678489,3.749859



Cell 2B outputs saved:
Extended feature-target panel parquet:    C:\Users\patri\vrp_project\data\processed\vrp_30d_corsi_v1\02B_30d_corsi_extended_feature_target_panel_20180625_20260702_20260704_130914.parquet
Extended feature-target preview CSV:      C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\02B_30d_corsi_extended_feature_target_panel_preview_20180625_20260702_20260704_130914.csv
Validation checks:                        C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\02B_30d_corsi_extended_validation_20180625_20260702_20260704_130914.csv
Feature diagnostics:                      C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\02B_30d_corsi_extended_feature_diagnostics_20180625_20260702_20260704_130914.csv
Target diagnostics:                       C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\02B_30d_corsi_extended_target_diagnostics_20180625_20260702_20260704_130914.csv
Target by year:                           C:\Users\patri\vrp_project\data\audi

In [5]:
# Cell 3: Fit walk-forward 30D Corsi/HAR candidate denominators and test same baseline Core thresholds
#
# Objective:
#   Test whether model-derived 30D forecast variance denominators improve the existing
#   30D Excel-style VRP signal without changing thresholds.
#
# Scope:
#   - Load frozen baseline signal tape and benchmark
#   - Load extended 2018+ feature/target panel from Cell 2B
#   - Fit walk-forward Ridge Corsi/HAR candidate models
#   - Generate OOS 2020-2026 forecasts
#   - Compute model VRP = log(implied variance / forecast variance)
#   - Compute prior-only 3m and 1y z-scores
#   - Apply same Core thresholds as baseline:
#       forecast_vol > 8.5
#       3m z > 0.7
#       1y z > 0.7
#       RSI14 < 70
#       VRP log > 0.7
#   - Compare each model to frozen baseline
#
# Explicitly NOT doing:
#   - No threshold optimization
#   - No cross-tenor work
#   - No Secondary signal
#   - No sizing
#   - No ensemble yet
#
# Model target:
#   log_forward_rv_21td_variance_decimal
#
# Walk-forward rule:
#   For test year Y:
#       train rows must satisfy:
#           date < Y-01-01
#           target_end_date < Y-01-01
#       test rows:
#           Y-01-01 <= date < (Y+1)-01-01
#
# Ridge alpha:
#   Chosen inside each training set using time-series CV on forecast-target MSE only.
#   No trade outcomes, implied variance, VRP, or RSI are used in model fitting.

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# =============================================================================
# 0. Paths and constants
# =============================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

BASELINE_BRANCH = "vrp_30d_baseline_v1"
CORSI_BRANCH = "vrp_30d_corsi_v1"

BASELINE_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / BASELINE_BRANCH
BASELINE_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / BASELINE_BRANCH

CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / CORSI_BRANCH
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / CORSI_BRANCH

CORSI_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CORSI_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

TEST_YEARS = list(range(2020, 2027))

TARGET_COL = "log_forward_rv_21td_variance_decimal"
TARGET_VAR_COL = "forward_rv_21td_variance_decimal"
TARGET_VOL_COL = "forward_rv_21td_vol_pct"

RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
MIN_TRAIN_ROWS = 100
CV_SPLITS_MAX = 5

Z3M_WINDOW = 63
Z3M_MIN_PERIODS = 63
Z1Y_WINDOW = 252
Z1Y_MIN_PERIODS = 252
Z_DDOF = 1

CORE_FORECAST_VOL_MIN = 8.5
CORE_Z3M_MIN = 0.7
CORE_Z1Y_MIN = 0.7
CORE_RSI14_MAX = 70
CORE_VRP_LOG_MIN = 0.7

LARGE_LOSS_LEVELS = [-25000, -50000, -100000]

print("Cell 3: Fit walk-forward 30D Corsi/HAR candidate denominators")
print(f"Project root:          {PROJECT_ROOT}")
print(f"Baseline processed:    {BASELINE_PROCESSED_DIR}")
print(f"Baseline audit:        {BASELINE_AUDIT_DIR}")
print(f"Corsi processed:       {CORSI_PROCESSED_DIR}")
print(f"Corsi audit:           {CORSI_AUDIT_DIR}")
print(f"Run timestamp:         {RUN_TIMESTAMP}")

print()
print("Locked Cell 3 scope:")
display(pd.DataFrame([{
    "target": TARGET_COL,
    "model": "StandardScaler + Ridge",
    "ridge_alphas": RIDGE_ALPHAS,
    "walk_forward_years": TEST_YEARS,
    "threshold_policy": "same baseline thresholds; no optimization",
    "forecast_vol_min": CORE_FORECAST_VOL_MIN,
    "z3m_min": CORE_Z3M_MIN,
    "z1y_min": CORE_Z1Y_MIN,
    "rsi14_max": CORE_RSI14_MAX,
    "vrp_log_min": CORE_VRP_LOG_MIN,
    "zscore_convention": "prior_only_shift1",
}]))


# =============================================================================
# 1. Helpers
# =============================================================================

def latest_file(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f"No files found in {directory} matching pattern: {pattern}")
    return matches[0]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def compute_prior_only_z(series: pd.Series, window: int, min_periods: int, ddof: int) -> pd.Series:
    history = series.shift(1)
    mu = history.rolling(window=window, min_periods=min_periods).mean()
    sigma = history.rolling(window=window, min_periods=min_periods).std(ddof=ddof)
    z = (series - mu) / sigma
    return z.replace([np.inf, -np.inf], np.nan)


def profit_factor(values: pd.Series) -> float:
    values = pd.Series(values).dropna()
    if values.empty:
        return np.nan

    gains = values.loc[values > 0].sum()
    losses = values.loc[values < 0].sum()

    if losses < 0:
        return float(gains / abs(losses))

    if gains > 0 and losses == 0:
        return np.inf

    return np.nan


def selected_trade_drawdown(values: pd.Series) -> float:
    values = pd.Series(values).dropna()

    if values.empty:
        return np.nan

    cumulative = values.cumsum()
    running_peak = cumulative.cummax()
    drawdown = cumulative - running_peak
    return float(drawdown.min())


def summarize_selected_trades(df: pd.DataFrame, signal_col: str, prefix: str) -> dict:
    selected = df.loc[df[signal_col].astype(bool)].copy()

    out = {
        f"{prefix}_trades": int(len(selected)),
        f"{prefix}_frequency": float(len(selected) / len(df)) if len(df) else np.nan,
    }

    if selected.empty:
        out.update({
            f"{prefix}_win_rate": np.nan,
            f"{prefix}_avg_pnl_per_trade": np.nan,
            f"{prefix}_avg_pnl_per_day": np.nan,
            f"{prefix}_exposure_weighted_pnl_per_day": np.nan,
            f"{prefix}_total_pnl": 0.0,
            f"{prefix}_profit_factor": np.nan,
            f"{prefix}_worst_trade": np.nan,
            f"{prefix}_best_trade": np.nan,
            f"{prefix}_avg_actual_dte": np.nan,
            f"{prefix}_selected_trade_drawdown": np.nan,
            f"{prefix}_worst_5_trade_sum": np.nan,
            f"{prefix}_worst_20_trade_sum": np.nan,
        })

        for level in LARGE_LOSS_LEVELS:
            out[f"{prefix}_loss_count_le_{abs(level):.0f}"] = 0

        return out

    total_dte = selected["actual_dte"].sum()

    out.update({
        f"{prefix}_win_rate": float((selected["outcome_value"] > 0).mean()),
        f"{prefix}_avg_pnl_per_trade": float(selected["outcome_value"].mean()),
        f"{prefix}_avg_pnl_per_day": float(selected["pnl_per_day"].mean()),
        f"{prefix}_exposure_weighted_pnl_per_day": (
            float(selected["outcome_value"].sum() / total_dte) if total_dte > 0 else np.nan
        ),
        f"{prefix}_total_pnl": float(selected["outcome_value"].sum()),
        f"{prefix}_profit_factor": float(profit_factor(selected["outcome_value"])),
        f"{prefix}_worst_trade": float(selected["outcome_value"].min()),
        f"{prefix}_best_trade": float(selected["outcome_value"].max()),
        f"{prefix}_avg_actual_dte": float(selected["actual_dte"].mean()),
        f"{prefix}_selected_trade_drawdown": selected_trade_drawdown(selected.sort_values("date")["outcome_value"]),
        f"{prefix}_worst_5_trade_sum": float(selected["outcome_value"].nsmallest(min(5, len(selected))).sum()),
        f"{prefix}_worst_20_trade_sum": float(selected["outcome_value"].nsmallest(min(20, len(selected))).sum()),
    })

    for level in LARGE_LOSS_LEVELS:
        out[f"{prefix}_loss_count_le_{abs(level):.0f}"] = int((selected["outcome_value"] <= level).sum())

    return out


def summarize_year(df: pd.DataFrame, signal_col: str, model_name: str) -> pd.DataFrame:
    rows = []

    for year, g in df.groupby(df["date"].dt.year):
        selected = g.loc[g[signal_col].astype(bool)].copy()

        row = {
            "model_name": model_name,
            "year": int(year),
            "eligible_rows": int(len(g)),
            "core_trades": int(len(selected)),
            "core_frequency": float(len(selected) / len(g)) if len(g) else np.nan,
        }

        if selected.empty:
            row.update({
                "core_win_rate": np.nan,
                "core_avg_pnl_per_trade": np.nan,
                "core_avg_pnl_per_day": np.nan,
                "core_total_pnl": 0.0,
                "core_worst_trade": np.nan,
                "core_best_trade": np.nan,
                "core_profit_factor": np.nan,
                "core_selected_trade_drawdown": np.nan,
            })

            for level in LARGE_LOSS_LEVELS:
                row[f"loss_count_le_{abs(level):.0f}"] = 0
        else:
            row.update({
                "core_win_rate": float((selected["outcome_value"] > 0).mean()),
                "core_avg_pnl_per_trade": float(selected["outcome_value"].mean()),
                "core_avg_pnl_per_day": float(selected["pnl_per_day"].mean()),
                "core_total_pnl": float(selected["outcome_value"].sum()),
                "core_worst_trade": float(selected["outcome_value"].min()),
                "core_best_trade": float(selected["outcome_value"].max()),
                "core_profit_factor": float(profit_factor(selected["outcome_value"])),
                "core_selected_trade_drawdown": selected_trade_drawdown(selected.sort_values("date")["outcome_value"]),
            })

            for level in LARGE_LOSS_LEVELS:
                row[f"loss_count_le_{abs(level):.0f}"] = int((selected["outcome_value"] <= level).sum())

        rows.append(row)

    return pd.DataFrame(rows)


def forecast_diagnostics(df: pd.DataFrame, forecast_var_col: str, model_name: str) -> dict:
    valid = df[[forecast_var_col, TARGET_VAR_COL, TARGET_VOL_COL]].replace([np.inf, -np.inf], np.nan).dropna()

    out = {
        "model_name": model_name,
        "rows": int(len(valid)),
    }

    if valid.empty:
        out.update({
            "target_variance_mean": np.nan,
            "forecast_variance_mean": np.nan,
            "target_vol_mean": np.nan,
            "forecast_vol_mean": np.nan,
            "variance_bias": np.nan,
            "variance_mae": np.nan,
            "variance_rmse": np.nan,
            "log_variance_mae": np.nan,
            "log_variance_rmse": np.nan,
            "correlation": np.nan,
            "r2": np.nan,
        })
        return out

    y = valid[TARGET_VAR_COL]
    pred = valid[forecast_var_col]

    log_y = np.log(y)
    log_pred = np.log(pred)

    out.update({
        "target_variance_mean": float(y.mean()),
        "forecast_variance_mean": float(pred.mean()),
        "target_vol_mean": float(valid[TARGET_VOL_COL].mean()),
        "forecast_vol_mean": float(np.sqrt(pred).mean() * 100.0),
        "variance_bias": float((pred - y).mean()),
        "variance_mae": float(mean_absolute_error(y, pred)),
        "variance_rmse": float(np.sqrt(mean_squared_error(y, pred))),
        "log_variance_mae": float(mean_absolute_error(log_y, log_pred)),
        "log_variance_rmse": float(np.sqrt(mean_squared_error(log_y, log_pred))),
        "correlation": float(np.corrcoef(y, pred)[0, 1]) if len(valid) > 1 else np.nan,
        "r2": float(r2_score(log_y, log_pred)) if len(valid) > 1 else np.nan,
    })

    return out


def choose_alpha_time_series_cv(X: pd.DataFrame, y: pd.Series, alphas: list[float]) -> tuple[float, pd.DataFrame]:
    n = len(X)

    if n < 120:
        # Use a conservative fixed alpha when the training set is too small for stable CV.
        cv_summary = pd.DataFrame([{
            "alpha": 10.0,
            "cv_mse_mean": np.nan,
            "cv_mse_std": np.nan,
            "fold_count": 0,
            "note": "fixed_alpha_due_to_small_training_sample",
        }])
        return 10.0, cv_summary

    n_splits = min(CV_SPLITS_MAX, max(2, n // 100))
    tscv = TimeSeriesSplit(n_splits=n_splits)

    rows = []

    for alpha in alphas:
        fold_mse = []

        for train_idx, val_idx in tscv.split(X):
            X_train_fold = X.iloc[train_idx]
            y_train_fold = y.iloc[train_idx]
            X_val_fold = X.iloc[val_idx]
            y_val_fold = y.iloc[val_idx]

            model = Pipeline([
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=alpha)),
            ])

            model.fit(X_train_fold, y_train_fold)
            pred = model.predict(X_val_fold)
            fold_mse.append(mean_squared_error(y_val_fold, pred))

        rows.append({
            "alpha": alpha,
            "cv_mse_mean": float(np.mean(fold_mse)),
            "cv_mse_std": float(np.std(fold_mse, ddof=1)) if len(fold_mse) > 1 else 0.0,
            "fold_count": len(fold_mse),
            "note": "time_series_cv",
        })

    cv_summary = pd.DataFrame(rows).sort_values(["cv_mse_mean", "alpha"]).reset_index(drop=True)
    chosen_alpha = float(cv_summary.iloc[0]["alpha"])

    return chosen_alpha, cv_summary


def pass_fail(condition) -> str:
    return "PASS" if bool(condition) else "FAIL"


def add_validation(rows, check_name, passed, details):
    rows.append({
        "check": check_name,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": details,
    })


# =============================================================================
# 2. Load inputs
# =============================================================================

baseline_signal_path = latest_file(
    BASELINE_PROCESSED_DIR,
    "00_excel_30d_vrp_baseline_signal_tape_*.parquet",
)

baseline_benchmark_path = latest_file(
    CORSI_AUDIT_DIR,
    "01_30d_corsi_baseline_benchmark_*.csv",
)

success_criteria_path = latest_file(
    CORSI_AUDIT_DIR,
    "01_30d_corsi_success_criteria_*.csv",
)

extended_panel_path = latest_file(
    CORSI_PROCESSED_DIR,
    "02B_30d_corsi_extended_feature_target_panel_*.parquet",
)

feature_group_manifest_path = latest_file(
    CORSI_AUDIT_DIR,
    "02B_30d_corsi_extended_feature_group_manifest_*.csv",
)

baseline_signal = normalize_columns(pd.read_parquet(baseline_signal_path))
baseline_benchmark = normalize_columns(pd.read_csv(baseline_benchmark_path))
success_criteria = normalize_columns(pd.read_csv(success_criteria_path))
extended_panel = normalize_columns(pd.read_parquet(extended_panel_path))
feature_group_manifest = normalize_columns(pd.read_csv(feature_group_manifest_path))

for df in [baseline_signal, extended_panel]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["trade_date"] = pd.to_numeric(df["trade_date"], errors="coerce").astype("Int64")

extended_panel["target_end_date"] = pd.to_datetime(extended_panel["target_end_date"], errors="coerce").dt.normalize()

feature_groups = (
    feature_group_manifest
    .groupby("feature_group")["feature"]
    .apply(list)
    .to_dict()
)

# Keep explicit order.
MODEL_ORDER = [
    "har_total_std_v1",
    "har_total_mean_v1",
    "har_total_accel_v1",
    "har_downside_v1",
    "har_regime_v1",
    "har_full_defensible_v1",
]

feature_groups = {k: feature_groups[k] for k in MODEL_ORDER if k in feature_groups}

print()
print("Inputs loaded:")
display(pd.DataFrame([
    {"role": "baseline_signal", "path": str(baseline_signal_path), "rows": len(baseline_signal)},
    {"role": "baseline_benchmark", "path": str(baseline_benchmark_path), "rows": len(baseline_benchmark)},
    {"role": "success_criteria", "path": str(success_criteria_path), "rows": len(success_criteria)},
    {"role": "extended_panel", "path": str(extended_panel_path), "rows": len(extended_panel)},
    {"role": "feature_group_manifest", "path": str(feature_group_manifest_path), "rows": len(feature_group_manifest)},
]))

print()
print("Candidate model feature groups:")
display(pd.DataFrame([
    {"model_name": k, "feature_count": len(v), "features": ", ".join(v)}
    for k, v in feature_groups.items()
]))


# =============================================================================
# 3. Input validation
# =============================================================================

validation_rows = []

required_baseline_cols = [
    "date",
    "trade_date",
    "implied_variance_decimal",
    "vix_30d_vol_pct",
    "rsi14",
    "outcome_value",
    "actual_dte",
    "pnl_per_day",
    "core_signal",
    "is_baseline_trade_eligible",
]

required_extended_cols = [
    "date",
    "trade_date",
    "target_end_date",
    TARGET_COL,
    TARGET_VAR_COL,
    TARGET_VOL_COL,
]

missing_baseline_cols = [c for c in required_baseline_cols if c not in baseline_signal.columns]
missing_extended_cols = [c for c in required_extended_cols if c not in extended_panel.columns]

add_validation(
    validation_rows,
    "Baseline required columns available",
    len(missing_baseline_cols) == 0,
    f"missing={missing_baseline_cols}",
)

add_validation(
    validation_rows,
    "Extended required columns available",
    len(missing_extended_cols) == 0,
    f"missing={missing_extended_cols}",
)

missing_model_features = []

for model_name, cols in feature_groups.items():
    for col in cols:
        if col not in extended_panel.columns:
            missing_model_features.append((model_name, col))

add_validation(
    validation_rows,
    "All model features exist in extended panel",
    len(missing_model_features) == 0,
    f"missing={missing_model_features[:10]}",
)

# Ensure model features do not use disallowed columns.
disallowed_tokens = [
    "implied",
    "vix",
    "vrp",
    "outcome",
    "pnl",
    "actual_dte",
    "core_signal",
    "rsi",
    "forward_rv",
    "forward_log",
    "target",
]

bad_feature_cols = []

for model_name, cols in feature_groups.items():
    for col in cols:
        lower = col.lower()
        if any(token in lower for token in disallowed_tokens):
            bad_feature_cols.append((model_name, col))

add_validation(
    validation_rows,
    "Model feature leakage name screen",
    len(bad_feature_cols) == 0,
    f"bad_features={bad_feature_cols}",
)

add_validation(
    validation_rows,
    "Baseline has no duplicate trade dates",
    baseline_signal["trade_date"].duplicated().sum() == 0,
    f"duplicates={int(baseline_signal['trade_date'].duplicated().sum())}",
)

add_validation(
    validation_rows,
    "Extended panel has no duplicate trade dates",
    extended_panel["trade_date"].duplicated().sum() == 0,
    f"duplicates={int(extended_panel['trade_date'].duplicated().sum())}",
)

validation = pd.DataFrame(validation_rows)

print()
print("Cell 3 input validation:")
display(validation)

if validation["status"].eq("FAIL").any():
    print()
    print("FAILED INPUT VALIDATION CHECKS:")
    display(validation.loc[validation["status"].eq("FAIL")])
    raise ValueError("Cell 3 input validation failed. Stop before model fitting.")


# =============================================================================
# 4. Fit walk-forward candidate denominators
# =============================================================================

all_forecasts = []
fit_summary_rows = []
cv_summary_rows = []

for model_name, feature_cols in feature_groups.items():
    print()
    print("=" * 100)
    print(f"Fitting model: {model_name}")
    print("=" * 100)

    model_forecast = extended_panel[["date", "trade_date", "target_end_date", TARGET_COL, TARGET_VAR_COL, TARGET_VOL_COL]].copy()
    model_forecast["model_name"] = model_name
    model_forecast["predicted_log_variance"] = np.nan
    model_forecast["forecast_variance_30d"] = np.nan
    model_forecast["forecast_vol_30d_pct"] = np.nan
    model_forecast["test_year"] = np.nan
    model_forecast["train_rows_used"] = np.nan
    model_forecast["selected_alpha"] = np.nan

    for test_year in TEST_YEARS:
        test_start = pd.Timestamp(f"{test_year}-01-01")
        test_end = pd.Timestamp(f"{test_year + 1}-01-01")

        train_mask = (
            extended_panel["date"].lt(test_start)
            & extended_panel["target_end_date"].lt(test_start)
            & extended_panel[TARGET_COL].notna()
            & extended_panel[feature_cols].notna().all(axis=1)
        )

        test_mask = (
            extended_panel["date"].ge(test_start)
            & extended_panel["date"].lt(test_end)
            & extended_panel[feature_cols].notna().all(axis=1)
        )

        train_df = extended_panel.loc[train_mask].sort_values("date").copy()
        test_df = extended_panel.loc[test_mask].sort_values("date").copy()

        if len(train_df) < MIN_TRAIN_ROWS or len(test_df) == 0:
            fit_summary_rows.append({
                "model_name": model_name,
                "test_year": test_year,
                "status": "SKIPPED",
                "train_rows": int(len(train_df)),
                "test_rows": int(len(test_df)),
                "selected_alpha": np.nan,
                "train_first_date": train_df["date"].min() if len(train_df) else pd.NaT,
                "train_last_date": train_df["date"].max() if len(train_df) else pd.NaT,
                "train_max_target_end_date": train_df["target_end_date"].max() if len(train_df) else pd.NaT,
                "test_first_date": test_df["date"].min() if len(test_df) else pd.NaT,
                "test_last_date": test_df["date"].max() if len(test_df) else pd.NaT,
                "reason": f"train_rows<{MIN_TRAIN_ROWS} or no test rows",
            })
            continue

        X_train = train_df[feature_cols].astype(float)
        y_train = train_df[TARGET_COL].astype(float)
        X_test = test_df[feature_cols].astype(float)

        selected_alpha, cv_summary = choose_alpha_time_series_cv(X_train, y_train, RIDGE_ALPHAS)

        cv_summary = cv_summary.copy()
        cv_summary["model_name"] = model_name
        cv_summary["test_year"] = test_year
        cv_summary_rows.append(cv_summary)

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=selected_alpha)),
        ])

        model.fit(X_train, y_train)
        pred_log = model.predict(X_test)

        pred_var = np.exp(pred_log)
        pred_vol_pct = np.sqrt(pred_var) * 100.0

        idx = test_df.index

        model_forecast.loc[idx, "predicted_log_variance"] = pred_log
        model_forecast.loc[idx, "forecast_variance_30d"] = pred_var
        model_forecast.loc[idx, "forecast_vol_30d_pct"] = pred_vol_pct
        model_forecast.loc[idx, "test_year"] = test_year
        model_forecast.loc[idx, "train_rows_used"] = len(train_df)
        model_forecast.loc[idx, "selected_alpha"] = selected_alpha

        # Coefficients in scaled space.
        ridge = model.named_steps["ridge"]

        fit_summary_rows.append({
            "model_name": model_name,
            "test_year": test_year,
            "status": "FIT",
            "train_rows": int(len(train_df)),
            "test_rows": int(len(test_df)),
            "selected_alpha": selected_alpha,
            "train_first_date": train_df["date"].min(),
            "train_last_date": train_df["date"].max(),
            "train_max_target_end_date": train_df["target_end_date"].max(),
            "test_first_date": test_df["date"].min(),
            "test_last_date": test_df["date"].max(),
            "reason": "",
        })

    all_forecasts.append(model_forecast)

forecast_panel_long = pd.concat(all_forecasts, ignore_index=True)
fit_summary = pd.DataFrame(fit_summary_rows)
cv_summary_all = pd.concat(cv_summary_rows, ignore_index=True) if cv_summary_rows else pd.DataFrame()

forecast_panel_long = forecast_panel_long.sort_values(["model_name", "date"]).reset_index(drop=True)

print()
print("Walk-forward fit summary:")
display(fit_summary)

print()
print("Forecast availability by model:")
display(
    forecast_panel_long
    .groupby("model_name")
    .agg(
        forecast_rows=("forecast_variance_30d", lambda s: int(s.notna().sum())),
        first_forecast_date=("date", lambda s: forecast_panel_long.loc[s.index, "date"][forecast_panel_long.loc[s.index, "forecast_variance_30d"].notna()].min()),
        last_forecast_date=("date", lambda s: forecast_panel_long.loc[s.index, "date"][forecast_panel_long.loc[s.index, "forecast_variance_30d"].notna()].max()),
        median_forecast_vol_pct=("forecast_vol_30d_pct", "median"),
        min_forecast_vol_pct=("forecast_vol_30d_pct", "min"),
        max_forecast_vol_pct=("forecast_vol_30d_pct", "max"),
    )
    .reset_index()
)


# =============================================================================
# 5. Build model signal panels and apply same baseline Core thresholds
# =============================================================================

# Baseline trade universe used for comparison.
baseline_eval = baseline_signal.copy()

if "is_baseline_trade_eligible" not in baseline_eval.columns:
    baseline_eval["is_baseline_trade_eligible"] = (
        baseline_eval["outcome_value"].notna()
        & baseline_eval["actual_dte"].notna()
        & baseline_eval["actual_dte"].gt(0)
    )

baseline_eval = baseline_eval.loc[baseline_eval["is_baseline_trade_eligible"]].copy()
baseline_eval = baseline_eval.sort_values("date").reset_index(drop=True)

model_signal_panels = []
forecast_diag_rows = []
forecast_diag_year_rows = []
model_comparison_rows = []
year_summary_rows = []
worst_trade_rows = []
overlap_rows = []

# Baseline comparison row.
baseline_metrics = summarize_selected_trades(baseline_eval, "core_signal", "core")

baseline_2025_df = baseline_eval.loc[baseline_eval["date"].dt.year.eq(2025)].copy()
baseline_2025_metrics = summarize_selected_trades(baseline_2025_df, "core_signal", "core_2025")

baseline_comparison_row = {
    "model_name": "excel_30d_vrp_baseline",
    "model_type": "baseline",
    "eligible_rows": int(len(baseline_eval)),
    **baseline_metrics,
    **baseline_2025_metrics,
    "beats_frequency": np.nan,
    "beats_win_rate": np.nan,
    "passes_worst_trade": np.nan,
    "passes_2025_total_pnl": np.nan,
    "passes_2025_worst_trade": np.nan,
    "passes_primary_and_tail": np.nan,
}

model_comparison_rows.append(baseline_comparison_row)

baseline_year_summary = summarize_year(baseline_eval, "core_signal", "excel_30d_vrp_baseline")
year_summary_rows.append(baseline_year_summary)

baseline_selected = baseline_eval.loc[baseline_eval["core_signal"]].copy()
baseline_selected_dates = set(baseline_selected["trade_date"].astype(int).tolist())

tmp_worst = baseline_selected.sort_values("outcome_value", ascending=True).head(20).copy()
tmp_worst["model_name"] = "excel_30d_vrp_baseline"
tmp_worst["rank_worst"] = np.arange(1, len(tmp_worst) + 1)
worst_trade_rows.append(tmp_worst)

# Candidate model signals.
for model_name in feature_groups.keys():
    mf = forecast_panel_long.loc[forecast_panel_long["model_name"].eq(model_name)].copy()

    eval_df = baseline_eval.merge(
        mf[
            [
                "trade_date",
                "model_name",
                "predicted_log_variance",
                "forecast_variance_30d",
                "forecast_vol_30d_pct",
                "test_year",
                "train_rows_used",
                "selected_alpha",
                TARGET_VAR_COL,
                TARGET_VOL_COL,
            ]
        ],
        on="trade_date",
        how="left",
        validate="one_to_one",
    )

    eval_df["model_name"] = model_name
    eval_df = eval_df.sort_values("date").reset_index(drop=True)

    eval_df["model_vrp_log"] = np.log(
        eval_df["implied_variance_decimal"] / eval_df["forecast_variance_30d"]
    )

    eval_df["model_vrp_z_3m"] = compute_prior_only_z(
        eval_df["model_vrp_log"],
        window=Z3M_WINDOW,
        min_periods=Z3M_MIN_PERIODS,
        ddof=Z_DDOF,
    )

    eval_df["model_vrp_z_1y"] = compute_prior_only_z(
        eval_df["model_vrp_log"],
        window=Z1Y_WINDOW,
        min_periods=Z1Y_MIN_PERIODS,
        ddof=Z_DDOF,
    )

    eval_df["model_trade_eligible"] = (
        eval_df["forecast_variance_30d"].notna()
        & eval_df["forecast_vol_30d_pct"].notna()
        & eval_df["model_vrp_log"].notna()
        & eval_df["model_vrp_z_3m"].notna()
        & eval_df["model_vrp_z_1y"].notna()
        & eval_df["rsi14"].notna()
        & eval_df["outcome_value"].notna()
        & eval_df["actual_dte"].notna()
        & eval_df["actual_dte"].gt(0)
    )

    eval_df["model_core_signal_same_thresholds"] = (
        eval_df["model_trade_eligible"]
        & eval_df["forecast_vol_30d_pct"].gt(CORE_FORECAST_VOL_MIN)
        & eval_df["model_vrp_z_3m"].gt(CORE_Z3M_MIN)
        & eval_df["model_vrp_z_1y"].gt(CORE_Z1Y_MIN)
        & eval_df["rsi14"].lt(CORE_RSI14_MAX)
        & eval_df["model_vrp_log"].gt(CORE_VRP_LOG_MIN)
    )

    model_eval_eligible = eval_df.loc[eval_df["model_trade_eligible"]].copy()

    metrics = summarize_selected_trades(
        model_eval_eligible,
        "model_core_signal_same_thresholds",
        "core",
    )

    model_2025_df = model_eval_eligible.loc[model_eval_eligible["date"].dt.year.eq(2025)].copy()

    metrics_2025 = summarize_selected_trades(
        model_2025_df,
        "model_core_signal_same_thresholds",
        "core_2025",
    )

    row = {
        "model_name": model_name,
        "model_type": "corsi_candidate_same_thresholds",
        "eligible_rows": int(len(model_eval_eligible)),
        **metrics,
        **metrics_2025,
    }

    row["beats_frequency"] = bool(row["core_frequency"] > baseline_metrics["core_frequency"])
    row["beats_win_rate"] = bool(row["core_win_rate"] >= baseline_metrics["core_win_rate"])
    row["passes_worst_trade"] = bool(row["core_worst_trade"] >= baseline_metrics["core_worst_trade"])
    row["passes_2025_total_pnl"] = bool(row["core_2025_total_pnl"] >= baseline_2025_metrics["core_2025_total_pnl"])
    row["passes_2025_worst_trade"] = bool(row["core_2025_worst_trade"] >= baseline_2025_metrics["core_2025_worst_trade"])

    row["passes_primary_and_tail"] = bool(
        row["beats_frequency"]
        and row["beats_win_rate"]
        and row["passes_worst_trade"]
        and row["passes_2025_total_pnl"]
        and row["passes_2025_worst_trade"]
    )

    model_comparison_rows.append(row)

    forecast_diag_rows.append(
        forecast_diagnostics(
            model_eval_eligible,
            forecast_var_col="forecast_variance_30d",
            model_name=model_name,
        )
    )

    for year, g in model_eval_eligible.groupby(model_eval_eligible["date"].dt.year):
        fd = forecast_diagnostics(
            g,
            forecast_var_col="forecast_variance_30d",
            model_name=model_name,
        )
        fd["year"] = int(year)
        forecast_diag_year_rows.append(fd)

    year_summary_rows.append(
        summarize_year(
            model_eval_eligible,
            "model_core_signal_same_thresholds",
            model_name,
        )
    )

    selected = model_eval_eligible.loc[model_eval_eligible["model_core_signal_same_thresholds"]].copy()
    selected_dates = set(selected["trade_date"].astype(int).tolist())

    both_dates = baseline_selected_dates.intersection(selected_dates)
    model_only_dates = selected_dates.difference(baseline_selected_dates)
    baseline_only_dates = baseline_selected_dates.difference(selected_dates)

    overlap_rows.append({
        "model_name": model_name,
        "baseline_selected_count": len(baseline_selected_dates),
        "model_selected_count": len(selected_dates),
        "both_selected_count": len(both_dates),
        "model_only_count": len(model_only_dates),
        "baseline_only_count": len(baseline_only_dates),
        "jaccard_overlap": (
            len(both_dates) / len(baseline_selected_dates.union(selected_dates))
            if len(baseline_selected_dates.union(selected_dates)) else np.nan
        ),
    })

    worst = selected.sort_values("outcome_value", ascending=True).head(20).copy()
    worst["model_name"] = model_name
    worst["rank_worst"] = np.arange(1, len(worst) + 1)
    worst_trade_rows.append(worst)

    model_signal_panels.append(eval_df)

signal_panel_long = pd.concat(model_signal_panels, ignore_index=True)
model_comparison = pd.DataFrame(model_comparison_rows)
year_summary = pd.concat(year_summary_rows, ignore_index=True)
forecast_diagnostics_summary = pd.DataFrame(forecast_diag_rows)
forecast_diagnostics_by_year = pd.DataFrame(forecast_diag_year_rows)
selected_overlap = pd.DataFrame(overlap_rows)

worst_20_trades_by_model = pd.concat(worst_trade_rows, ignore_index=True) if worst_trade_rows else pd.DataFrame()

# Clean display order.
comparison_cols_order = [
    "model_name",
    "model_type",
    "eligible_rows",
    "core_trades",
    "core_frequency",
    "core_win_rate",
    "core_avg_pnl_per_trade",
    "core_avg_pnl_per_day",
    "core_exposure_weighted_pnl_per_day",
    "core_total_pnl",
    "core_profit_factor",
    "core_worst_trade",
    "core_selected_trade_drawdown",
    "core_worst_5_trade_sum",
    "core_worst_20_trade_sum",
    "core_loss_count_le_25000",
    "core_loss_count_le_50000",
    "core_loss_count_le_100000",
    "core_2025_trades",
    "core_2025_frequency",
    "core_2025_win_rate",
    "core_2025_total_pnl",
    "core_2025_worst_trade",
    "beats_frequency",
    "beats_win_rate",
    "passes_worst_trade",
    "passes_2025_total_pnl",
    "passes_2025_worst_trade",
    "passes_primary_and_tail",
]

comparison_cols_order = [c for c in comparison_cols_order if c in model_comparison.columns]
model_comparison = model_comparison[comparison_cols_order].copy()


# =============================================================================
# 6. Result validation
# =============================================================================

result_validation_rows = []

add_validation(
    result_validation_rows,
    "Forecasts generated for all candidate models",
    forecast_panel_long.groupby("model_name")["forecast_variance_30d"].apply(lambda s: s.notna().sum()).gt(0).all(),
    forecast_panel_long.groupby("model_name")["forecast_variance_30d"].apply(lambda s: int(s.notna().sum())).to_dict(),
)

add_validation(
    result_validation_rows,
    "Candidate signal panels created",
    len(signal_panel_long) > 0,
    f"rows={len(signal_panel_long)}",
)

add_validation(
    result_validation_rows,
    "No non-positive forecast variances",
    signal_panel_long["forecast_variance_30d"].dropna().gt(0).all(),
    f"min_forecast_variance={signal_panel_long['forecast_variance_30d'].min()}",
)

add_validation(
    result_validation_rows,
    "No infinite model VRP/z values",
    int(np.isinf(signal_panel_long.select_dtypes(include=[np.number])).sum().sum()) == 0,
    "numeric inf count is zero",
)

result_validation = pd.DataFrame(result_validation_rows)

print()
print("Cell 3 result validation:")
display(result_validation)

if result_validation["status"].eq("FAIL").any():
    print()
    print("FAILED RESULT VALIDATION CHECKS:")
    display(result_validation.loc[result_validation["status"].eq("FAIL")])
    raise ValueError("Cell 3 result validation failed.")


# =============================================================================
# 7. Display results
# =============================================================================

print()
print("Model comparison versus frozen 30D Excel baseline:")
display(model_comparison)

print()
print("Models passing primary + tail criteria:")
passing_models = model_comparison.loc[model_comparison["passes_primary_and_tail"].eq(True)].copy()
display(passing_models)

print()
print("Forecast diagnostics:")
display(forecast_diagnostics_summary)

print()
print("Forecast diagnostics by year:")
display(forecast_diagnostics_by_year)

print()
print("Year-by-year trade summary:")
display(year_summary)

print()
print("Selected-trade overlap versus baseline:")
display(selected_overlap)

print()
print("Worst 20 selected trades by model:")
worst_display_cols = [
    "model_name",
    "rank_worst",
    "date",
    "trade_date",
    "vix_30d_vol_pct",
    "forecast_vol_30d_pct",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "rsi14",
    "outcome_value",
    "actual_dte",
    "pnl_per_day",
]

worst_display_cols = [c for c in worst_display_cols if c in worst_20_trades_by_model.columns]
display(worst_20_trades_by_model[worst_display_cols])


# =============================================================================
# 8. Save outputs
# =============================================================================

safe_start = baseline_signal["date"].min().strftime("%Y%m%d")
safe_end = baseline_signal["date"].max().strftime("%Y%m%d")

forecast_panel_path = (
    CORSI_PROCESSED_DIR
    / f"03_30d_corsi_oos_forecast_panel_long_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.parquet"
)

signal_panel_path = (
    CORSI_PROCESSED_DIR
    / f"03_30d_corsi_same_threshold_signal_panel_long_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.parquet"
)

fit_summary_path = (
    CORSI_AUDIT_DIR
    / f"03_30d_corsi_fit_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

cv_summary_path = (
    CORSI_AUDIT_DIR
    / f"03_30d_corsi_alpha_cv_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

model_comparison_path = (
    CORSI_AUDIT_DIR
    / f"03_30d_corsi_same_threshold_model_comparison_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

forecast_diagnostics_path = (
    CORSI_AUDIT_DIR
    / f"03_30d_corsi_forecast_diagnostics_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

forecast_diagnostics_by_year_path = (
    CORSI_AUDIT_DIR
    / f"03_30d_corsi_forecast_diagnostics_by_year_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

year_summary_path = (
    CORSI_AUDIT_DIR
    / f"03_30d_corsi_same_threshold_year_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

selected_overlap_path = (
    CORSI_AUDIT_DIR
    / f"03_30d_corsi_selected_trade_overlap_vs_baseline_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

worst_20_path = (
    CORSI_AUDIT_DIR
    / f"03_30d_corsi_worst_20_trades_by_model_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

input_validation_path = (
    CORSI_AUDIT_DIR
    / f"03_30d_corsi_input_validation_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

result_validation_path = (
    CORSI_AUDIT_DIR
    / f"03_30d_corsi_result_validation_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

forecast_panel_long.to_parquet(forecast_panel_path, index=False)
signal_panel_long.to_parquet(signal_panel_path, index=False)

fit_summary.to_csv(fit_summary_path, index=False)
cv_summary_all.to_csv(cv_summary_path, index=False)
model_comparison.to_csv(model_comparison_path, index=False)
forecast_diagnostics_summary.to_csv(forecast_diagnostics_path, index=False)
forecast_diagnostics_by_year.to_csv(forecast_diagnostics_by_year_path, index=False)
year_summary.to_csv(year_summary_path, index=False)
selected_overlap.to_csv(selected_overlap_path, index=False)
worst_20_trades_by_model.to_csv(worst_20_path, index=False)
validation.to_csv(input_validation_path, index=False)
result_validation.to_csv(result_validation_path, index=False)

print()
print("Cell 3 outputs saved:")
print(f"OOS forecast panel:                 {forecast_panel_path}")
print(f"Same-threshold signal panel:        {signal_panel_path}")
print(f"Fit summary:                        {fit_summary_path}")
print(f"Alpha CV summary:                   {cv_summary_path}")
print(f"Model comparison:                   {model_comparison_path}")
print(f"Forecast diagnostics:               {forecast_diagnostics_path}")
print(f"Forecast diagnostics by year:       {forecast_diagnostics_by_year_path}")
print(f"Year summary:                       {year_summary_path}")
print(f"Selected-trade overlap:             {selected_overlap_path}")
print(f"Worst 20 trades by model:           {worst_20_path}")
print(f"Input validation:                   {input_validation_path}")
print(f"Result validation:                  {result_validation_path}")

print()
print("Cell 3 complete.")
print()
print("Next decision:")
print("Review whether any denominator passes same-threshold criteria.")
print("If none pass, do not optimize yet; first inspect why: frequency, win rate, 2025 behavior, or tail.")

Cell 3: Fit walk-forward 30D Corsi/HAR candidate denominators
Project root:          C:\Users\patri\vrp_project
Baseline processed:    C:\Users\patri\vrp_project\data\processed\vrp_30d_baseline_v1
Baseline audit:        C:\Users\patri\vrp_project\data\audit\vrp_30d_baseline_v1
Corsi processed:       C:\Users\patri\vrp_project\data\processed\vrp_30d_corsi_v1
Corsi audit:           C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1
Run timestamp:         20260704_140452

Locked Cell 3 scope:


,target,model,ridge_alphas,walk_forward_years,threshold_policy,forecast_vol_min,z3m_min,z1y_min,rsi14_max,vrp_log_min,zscore_convention
0,log_forward_rv_21td_variance_decimal,StandardScaler + Ridge,"[0.01, 0.1, 1.0, 10.0, 100.0]","[2020, 2021, 2022, 2023, 2024, 2025, 2026]",same baseline thresholds; no optimization,8.5,0.7,0.7,70,0.7,prior_only_shift1



Inputs loaded:


,role,path,rows
0,baseline_signal,C:\Users\patri\vrp_project\data\processed\vrp_...,1612
1,baseline_benchmark,C:\Users\patri\vrp_project\data\audit\vrp_30d_...,1
2,success_criteria,C:\Users\patri\vrp_project\data\audit\vrp_30d_...,8
3,extended_panel,C:\Users\patri\vrp_project\data\processed\vrp_...,2016
4,feature_group_manifest,C:\Users\patri\vrp_project\data\audit\vrp_30d_...,65



Candidate model feature groups:


...


Selected-trade overlap versus baseline:


,model_name,baseline_selected_count,model_selected_count,both_selected_count,model_only_count,baseline_only_count,jaccard_overlap
0,har_total_std_v1,177,161,70,91,107,0.261194
1,har_total_mean_v1,177,173,71,102,106,0.254480
2,har_total_accel_v1,177,183,69,114,108,0.237113
3,har_downside_v1,177,150,53,97,124,0.193431
4,har_regime_v1,177,151,47,104,130,0.167260
5,har_full_defensible_v1,177,172,48,124,129,0.159468



Worst 20 selected trades by model:


,model_name,rank_worst,date,trade_date,vix_30d_vol_pct,forecast_vol_30d_pct,model_vrp_log,model_vrp_z_3m,model_vrp_z_1y,rsi14,outcome_value,actual_dte,pnl_per_day
0,excel_30d_vrp_baseline,1,2025-03-03,20250303,22.808456,NaN,NaN,NaN,NaN,37.160997,-112316.486943,32.0,-3509.890217
1,excel_30d_vrp_baseline,2,2025-03-04,20250304,23.834459,NaN,NaN,NaN,NaN,33.223807,-100779.661310,31.0,-3250.956816
2,excel_30d_vrp_baseline,3,2025-03-06,20250306,25.083189,NaN,NaN,NaN,NaN,33.946232,-92588.332880,29.0,-3192.701134
3,excel_30d_vrp_baseline,4,2025-02-24,20250224,19.078121,NaN,NaN,NaN,NaN,43.422979,-51336.648143,32.0,-1604.270254
4,excel_30d_vrp_baseline,5,2025-02-26,20250226,19.083176,NaN,NaN,NaN,NaN,41.127776,-47172.123854,30.0,-1572.404128
...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,har_full_defensible_v1,16,2022-08-03,20220803,21.891383,13.036265,1.036716,1.232120,1.085455,65.646223,-33510.060960,30.0,-1117.002032
136,har_full_defensible_v1,17,2022-08-04,20220804,21.626952,12.103113,1.160955,1.611378,1.502226,65.283338,-32982.172189,29.0,-1137.316282
137,har_full_defensible_v1,18,2022-08-05,20220805,20.883404,11.585659,1.178374,1.616019,1.549588,64.481145,-32891.134061,28.0,-1174.683359
138,har_full_defensible_v1,19,2022-03-22,20220322,22.898829,13.233831,1.096618,0.723593,1.367533,59.958315,-31611.774954,31.0,-1019.734676



Cell 3 outputs saved:
OOS forecast panel:                 C:\Users\patri\vrp_project\data\processed\vrp_30d_corsi_v1\03_30d_corsi_oos_forecast_panel_long_20200102_20260602_20260704_140452.parquet
Same-threshold signal panel:        C:\Users\patri\vrp_project\data\processed\vrp_30d_corsi_v1\03_30d_corsi_same_threshold_signal_panel_long_20200102_20260602_20260704_140452.parquet
Fit summary:                        C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\03_30d_corsi_fit_summary_20200102_20260602_20260704_140452.csv
Alpha CV summary:                   C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\03_30d_corsi_alpha_cv_summary_20200102_20260602_20260704_140452.csv
Model comparison:                   C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\03_30d_corsi_same_threshold_model_comparison_20200102_20260602_20260704_140452.csv
Forecast diagnostics:               C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\03_30d_corsi_forecast_diagnostics_20200102

In [6]:
# Cell 3A: Repair evaluation sequence — compute model VRP z-scores on full daily series
#
# Objective:
#   Re-evaluate the saved Cell 3 OOS Corsi forecasts correctly.
#
# Why:
#   Cell 3 fit/forecast generation was fine, but the signal evaluation computed model VRP z-scores
#   after filtering to the baseline trade-eligible panel. That artificially delayed z-score availability
#   and made the model eligible universe smaller than it should be.
#
# Scope:
#   - No model refitting
#   - No feature changes
#   - No target changes
#   - No threshold optimization
#   - No cross-tenor work
#
# Correct sequence:
#   1. Load all frozen baseline daily 30D rows.
#   2. Merge each model's saved OOS forecasts onto all daily rows.
#   3. Compute model VRP on the full daily series.
#   4. Compute prior-only 3m and 1y z-scores on the full daily model VRP series.
#   5. Then define trade-eligible/common rows.
#   6. Compare baseline vs model on the same eligible date universe.

from pathlib import Path
import numpy as np
import pandas as pd


# =============================================================================
# 0. Paths and constants
# =============================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

BASELINE_BRANCH = "vrp_30d_baseline_v1"
CORSI_BRANCH = "vrp_30d_corsi_v1"

BASELINE_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / BASELINE_BRANCH
CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / CORSI_BRANCH
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / CORSI_BRANCH

CORSI_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CORSI_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

Z3M_WINDOW = 63
Z3M_MIN_PERIODS = 63
Z1Y_WINDOW = 252
Z1Y_MIN_PERIODS = 252
Z_DDOF = 1

CORE_FORECAST_VOL_MIN = 8.5
CORE_Z3M_MIN = 0.7
CORE_Z1Y_MIN = 0.7
CORE_RSI14_MAX = 70
CORE_VRP_LOG_MIN = 0.7

LARGE_LOSS_LEVELS = [-25000, -50000, -100000]

print("Cell 3A: Repair Corsi same-threshold evaluation")
print(f"Project root:        {PROJECT_ROOT}")
print(f"Baseline processed:  {BASELINE_PROCESSED_DIR}")
print(f"Corsi processed:     {CORSI_PROCESSED_DIR}")
print(f"Corsi audit:         {CORSI_AUDIT_DIR}")
print(f"Run timestamp:       {RUN_TIMESTAMP}")

print()
print("Locked Cell 3A scope:")
display(pd.DataFrame([{
    "model_refit": "NO",
    "threshold_optimization": "NO",
    "zscore_series": "full_daily_30d_signal_panel",
    "zscore_convention": "prior_only_shift1",
    "z3m_window": Z3M_WINDOW,
    "z1y_window": Z1Y_WINDOW,
    "comparison": "baseline_vs_model_on_common_eligible_universe",
    "forecast_vol_min": CORE_FORECAST_VOL_MIN,
    "z3m_min": CORE_Z3M_MIN,
    "z1y_min": CORE_Z1Y_MIN,
    "rsi14_max": CORE_RSI14_MAX,
    "vrp_log_min": CORE_VRP_LOG_MIN,
}]))


# =============================================================================
# 1. Helpers
# =============================================================================

def latest_file(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f"No files found in {directory} matching pattern: {pattern}")
    return matches[0]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def compute_prior_only_z(series: pd.Series, window: int, min_periods: int, ddof: int) -> pd.Series:
    history = series.shift(1)
    mu = history.rolling(window=window, min_periods=min_periods).mean()
    sigma = history.rolling(window=window, min_periods=min_periods).std(ddof=ddof)
    z = (series - mu) / sigma
    return z.replace([np.inf, -np.inf], np.nan)


def profit_factor(values: pd.Series) -> float:
    values = pd.Series(values).dropna()

    if values.empty:
        return np.nan

    gains = values.loc[values > 0].sum()
    losses = values.loc[values < 0].sum()

    if losses < 0:
        return float(gains / abs(losses))

    if gains > 0 and losses == 0:
        return np.inf

    return np.nan


def selected_trade_drawdown(values: pd.Series) -> float:
    values = pd.Series(values).dropna()

    if values.empty:
        return np.nan

    cumulative = values.cumsum()
    running_peak = cumulative.cummax()
    drawdown = cumulative - running_peak

    return float(drawdown.min())


def summarize_selected_trades(df: pd.DataFrame, signal_col: str, prefix: str) -> dict:
    selected = df.loc[df[signal_col].astype(bool)].sort_values("date").copy()

    out = {
        f"{prefix}_eligible_rows": int(len(df)),
        f"{prefix}_trades": int(len(selected)),
        f"{prefix}_frequency": float(len(selected) / len(df)) if len(df) else np.nan,
    }

    if selected.empty:
        out.update({
            f"{prefix}_win_rate": np.nan,
            f"{prefix}_avg_pnl_per_trade": np.nan,
            f"{prefix}_avg_pnl_per_day": np.nan,
            f"{prefix}_exposure_weighted_pnl_per_day": np.nan,
            f"{prefix}_total_pnl": 0.0,
            f"{prefix}_profit_factor": np.nan,
            f"{prefix}_worst_trade": np.nan,
            f"{prefix}_best_trade": np.nan,
            f"{prefix}_avg_actual_dte": np.nan,
            f"{prefix}_selected_trade_drawdown": np.nan,
            f"{prefix}_worst_5_trade_sum": np.nan,
            f"{prefix}_worst_20_trade_sum": np.nan,
        })

        for level in LARGE_LOSS_LEVELS:
            out[f"{prefix}_loss_count_le_{abs(level):.0f}"] = 0

        return out

    total_dte = selected["actual_dte"].sum()

    out.update({
        f"{prefix}_win_rate": float((selected["outcome_value"] > 0).mean()),
        f"{prefix}_avg_pnl_per_trade": float(selected["outcome_value"].mean()),
        f"{prefix}_avg_pnl_per_day": float(selected["pnl_per_day"].mean()),
        f"{prefix}_exposure_weighted_pnl_per_day": (
            float(selected["outcome_value"].sum() / total_dte) if total_dte > 0 else np.nan
        ),
        f"{prefix}_total_pnl": float(selected["outcome_value"].sum()),
        f"{prefix}_profit_factor": float(profit_factor(selected["outcome_value"])),
        f"{prefix}_worst_trade": float(selected["outcome_value"].min()),
        f"{prefix}_best_trade": float(selected["outcome_value"].max()),
        f"{prefix}_avg_actual_dte": float(selected["actual_dte"].mean()),
        f"{prefix}_selected_trade_drawdown": selected_trade_drawdown(selected["outcome_value"]),
        f"{prefix}_worst_5_trade_sum": float(selected["outcome_value"].nsmallest(min(5, len(selected))).sum()),
        f"{prefix}_worst_20_trade_sum": float(selected["outcome_value"].nsmallest(min(20, len(selected))).sum()),
    })

    for level in LARGE_LOSS_LEVELS:
        out[f"{prefix}_loss_count_le_{abs(level):.0f}"] = int((selected["outcome_value"] <= level).sum())

    return out


def summarize_year(df: pd.DataFrame, signal_col: str, model_name: str, evaluated_model: str, universe: str) -> pd.DataFrame:
    rows = []

    for year, g in df.groupby(df["date"].dt.year):
        selected = g.loc[g[signal_col].astype(bool)].sort_values("date").copy()

        row = {
            "model_name": model_name,
            "evaluated_model": evaluated_model,
            "universe": universe,
            "year": int(year),
            "eligible_rows": int(len(g)),
            "core_trades": int(len(selected)),
            "core_frequency": float(len(selected) / len(g)) if len(g) else np.nan,
        }

        if selected.empty:
            row.update({
                "core_win_rate": np.nan,
                "core_avg_pnl_per_trade": np.nan,
                "core_avg_pnl_per_day": np.nan,
                "core_total_pnl": 0.0,
                "core_worst_trade": np.nan,
                "core_best_trade": np.nan,
                "core_profit_factor": np.nan,
                "core_selected_trade_drawdown": np.nan,
            })

            for level in LARGE_LOSS_LEVELS:
                row[f"loss_count_le_{abs(level):.0f}"] = 0
        else:
            row.update({
                "core_win_rate": float((selected["outcome_value"] > 0).mean()),
                "core_avg_pnl_per_trade": float(selected["outcome_value"].mean()),
                "core_avg_pnl_per_day": float(selected["pnl_per_day"].mean()),
                "core_total_pnl": float(selected["outcome_value"].sum()),
                "core_worst_trade": float(selected["outcome_value"].min()),
                "core_best_trade": float(selected["outcome_value"].max()),
                "core_profit_factor": float(profit_factor(selected["outcome_value"])),
                "core_selected_trade_drawdown": selected_trade_drawdown(selected["outcome_value"]),
            })

            for level in LARGE_LOSS_LEVELS:
                row[f"loss_count_le_{abs(level):.0f}"] = int((selected["outcome_value"] <= level).sum())

        rows.append(row)

    return pd.DataFrame(rows)


def add_validation(rows, check_name, passed, details):
    rows.append({
        "check": check_name,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": details,
    })


# =============================================================================
# 2. Load saved inputs
# =============================================================================

baseline_signal_path = latest_file(
    BASELINE_PROCESSED_DIR,
    "00_excel_30d_vrp_baseline_signal_tape_*.parquet",
)

forecast_panel_path = latest_file(
    CORSI_PROCESSED_DIR,
    "03_30d_corsi_oos_forecast_panel_long_*.parquet",
)

cell3_old_comparison_path = latest_file(
    CORSI_AUDIT_DIR,
    "03_30d_corsi_same_threshold_model_comparison_*.csv",
)

baseline_signal = normalize_columns(pd.read_parquet(baseline_signal_path))
forecast_panel_long = normalize_columns(pd.read_parquet(forecast_panel_path))
cell3_old_comparison = normalize_columns(pd.read_csv(cell3_old_comparison_path))

baseline_signal["date"] = pd.to_datetime(baseline_signal["date"], errors="coerce").dt.normalize()
baseline_signal["trade_date"] = pd.to_numeric(baseline_signal["trade_date"], errors="coerce").astype("Int64")

forecast_panel_long["date"] = pd.to_datetime(forecast_panel_long["date"], errors="coerce").dt.normalize()
forecast_panel_long["trade_date"] = pd.to_numeric(forecast_panel_long["trade_date"], errors="coerce").astype("Int64")

model_names = sorted(forecast_panel_long["model_name"].dropna().unique().tolist())

print()
print("Inputs loaded:")
display(pd.DataFrame([
    {"role": "baseline_signal_all_daily_rows", "path": str(baseline_signal_path), "rows": len(baseline_signal)},
    {"role": "saved_oos_forecast_panel_long", "path": str(forecast_panel_path), "rows": len(forecast_panel_long)},
    {"role": "old_cell3_model_comparison", "path": str(cell3_old_comparison_path), "rows": len(cell3_old_comparison)},
]))

print()
print("Models to rescore:")
display(pd.DataFrame({"model_name": model_names}))


# =============================================================================
# 3. Baseline signal eligibility and baseline Core recomputation
# =============================================================================

required_baseline_cols = [
    "date",
    "trade_date",
    "implied_variance_decimal",
    "vix_30d_vol_pct",
    "rv21d_vol_pct",
    "rv21d_variance_decimal",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rsi14",
    "outcome_value",
    "actual_dte",
    "pnl_per_day",
    "core_signal",
]

missing_baseline_cols = [c for c in required_baseline_cols if c not in baseline_signal.columns]

if missing_baseline_cols:
    raise ValueError(f"Baseline signal missing required columns: {missing_baseline_cols}")

baseline_signal = baseline_signal.sort_values("date").reset_index(drop=True)

baseline_signal["baseline_signal_eligible_recomputed"] = (
    baseline_signal["implied_variance_decimal"].notna()
    & baseline_signal["rv21d_vol_pct"].notna()
    & baseline_signal["rv21d_variance_decimal"].notna()
    & baseline_signal["vrp_log"].notna()
    & baseline_signal["vrp_z_3m"].notna()
    & baseline_signal["vrp_z_1y"].notna()
    & baseline_signal["rsi14"].notna()
    & baseline_signal["outcome_value"].notna()
    & baseline_signal["actual_dte"].notna()
    & baseline_signal["actual_dte"].gt(0)
)

baseline_signal["baseline_core_signal_recomputed"] = (
    baseline_signal["baseline_signal_eligible_recomputed"]
    & baseline_signal["rv21d_vol_pct"].gt(CORE_FORECAST_VOL_MIN)
    & baseline_signal["vrp_z_3m"].gt(CORE_Z3M_MIN)
    & baseline_signal["vrp_z_1y"].gt(CORE_Z1Y_MIN)
    & baseline_signal["rsi14"].lt(CORE_RSI14_MAX)
    & baseline_signal["vrp_log"].gt(CORE_VRP_LOG_MIN)
)

baseline_core_diff_count = int(
    baseline_signal["baseline_core_signal_recomputed"].astype(bool).ne(
        baseline_signal["core_signal"].fillna(False).astype(bool)
    ).sum()
)

baseline_native_df = baseline_signal.loc[baseline_signal["baseline_signal_eligible_recomputed"]].copy()

print()
print("Baseline native recomputed summary:")
display(pd.DataFrame([{
    "all_daily_rows": len(baseline_signal),
    "baseline_signal_eligible_recomputed": int(baseline_signal["baseline_signal_eligible_recomputed"].sum()),
    "stored_core_signal_count": int(baseline_signal["core_signal"].fillna(False).astype(bool).sum()),
    "recomputed_core_signal_count": int(baseline_signal["baseline_core_signal_recomputed"].sum()),
    "stored_vs_recomputed_core_diff_count": baseline_core_diff_count,
}]))


# =============================================================================
# 4. Rescore each model on the full daily series
# =============================================================================

rescored_signal_panels = []
z_availability_rows = []
z_availability_year_rows = []

for model_name in model_names:
    mf = forecast_panel_long.loc[forecast_panel_long["model_name"].eq(model_name)].copy()

    dup_count = int(mf["trade_date"].duplicated().sum())

    if dup_count > 0:
        raise ValueError(f"Duplicate trade_date rows found for model {model_name}: {dup_count}")

    eval_df = baseline_signal.merge(
        mf[
            [
                "trade_date",
                "model_name",
                "predicted_log_variance",
                "forecast_variance_30d",
                "forecast_vol_30d_pct",
                "test_year",
                "train_rows_used",
                "selected_alpha",
            ]
        ],
        on="trade_date",
        how="left",
        validate="one_to_one",
    )

    eval_df["model_name"] = model_name
    eval_df = eval_df.sort_values("date").reset_index(drop=True)

    eval_df["model_vrp_log"] = np.log(
        eval_df["implied_variance_decimal"] / eval_df["forecast_variance_30d"]
    )

    # Critical repair:
    # compute z-scores on the full daily model VRP series, before trade/outcome filtering.
    eval_df["model_vrp_z_3m"] = compute_prior_only_z(
        eval_df["model_vrp_log"],
        window=Z3M_WINDOW,
        min_periods=Z3M_MIN_PERIODS,
        ddof=Z_DDOF,
    )

    eval_df["model_vrp_z_1y"] = compute_prior_only_z(
        eval_df["model_vrp_log"],
        window=Z1Y_WINDOW,
        min_periods=Z1Y_MIN_PERIODS,
        ddof=Z_DDOF,
    )

    eval_df["model_signal_eligible"] = (
        eval_df["forecast_variance_30d"].notna()
        & eval_df["forecast_vol_30d_pct"].notna()
        & eval_df["model_vrp_log"].notna()
        & eval_df["model_vrp_z_3m"].notna()
        & eval_df["model_vrp_z_1y"].notna()
        & eval_df["rsi14"].notna()
        & eval_df["outcome_value"].notna()
        & eval_df["actual_dte"].notna()
        & eval_df["actual_dte"].gt(0)
    )

    eval_df["model_core_signal_same_thresholds"] = (
        eval_df["model_signal_eligible"]
        & eval_df["forecast_vol_30d_pct"].gt(CORE_FORECAST_VOL_MIN)
        & eval_df["model_vrp_z_3m"].gt(CORE_Z3M_MIN)
        & eval_df["model_vrp_z_1y"].gt(CORE_Z1Y_MIN)
        & eval_df["rsi14"].lt(CORE_RSI14_MAX)
        & eval_df["model_vrp_log"].gt(CORE_VRP_LOG_MIN)
    )

    eval_df["common_signal_eligible"] = (
        eval_df["baseline_signal_eligible_recomputed"]
        & eval_df["model_signal_eligible"]
    )

    z3 = eval_df.loc[eval_df["model_vrp_z_3m"].notna(), "date"]
    z1 = eval_df.loc[eval_df["model_vrp_z_1y"].notna(), "date"]

    z_availability_rows.append({
        "model_name": model_name,
        "all_daily_rows": len(eval_df),
        "forecast_non_null_rows": int(eval_df["forecast_variance_30d"].notna().sum()),
        "model_vrp_non_null_rows": int(eval_df["model_vrp_log"].notna().sum()),
        "model_z3m_non_null_rows": int(eval_df["model_vrp_z_3m"].notna().sum()),
        "model_z1y_non_null_rows": int(eval_df["model_vrp_z_1y"].notna().sum()),
        "model_signal_eligible_rows": int(eval_df["model_signal_eligible"].sum()),
        "common_signal_eligible_rows": int(eval_df["common_signal_eligible"].sum()),
        "first_z3m_date": z3.min() if len(z3) else pd.NaT,
        "first_z1y_date": z1.min() if len(z1) else pd.NaT,
        "first_model_trade_eligible_date": eval_df.loc[eval_df["model_signal_eligible"], "date"].min(),
        "last_model_trade_eligible_date": eval_df.loc[eval_df["model_signal_eligible"], "date"].max(),
    })

    for year, g in eval_df.groupby(eval_df["date"].dt.year):
        z_availability_year_rows.append({
            "model_name": model_name,
            "year": int(year),
            "all_daily_rows": len(g),
            "forecast_non_null_rows": int(g["forecast_variance_30d"].notna().sum()),
            "z3m_non_null_rows": int(g["model_vrp_z_3m"].notna().sum()),
            "z1y_non_null_rows": int(g["model_vrp_z_1y"].notna().sum()),
            "model_signal_eligible_rows": int(g["model_signal_eligible"].sum()),
            "common_signal_eligible_rows": int(g["common_signal_eligible"].sum()),
            "model_core_trades": int(g["model_core_signal_same_thresholds"].sum()),
            "baseline_core_trades_on_same_rows": int(
                g.loc[g["common_signal_eligible"], "baseline_core_signal_recomputed"].sum()
            ),
        })

    rescored_signal_panels.append(eval_df)

rescored_signal_panel_long = pd.concat(rescored_signal_panels, ignore_index=True)
z_availability = pd.DataFrame(z_availability_rows)
z_availability_by_year = pd.DataFrame(z_availability_year_rows)

print()
print("Z-score and eligibility availability after full-series rescore:")
display(z_availability)

print()
print("Z-score and eligibility availability by year:")
display(z_availability_by_year)


# =============================================================================
# 5. Native and common-universe comparisons
# =============================================================================

native_comparison_rows = []
common_comparison_rows = []
common_year_rows = []
worst_20_rows = []
overlap_rows = []

# Native baseline row.
baseline_native_metrics = summarize_selected_trades(
    baseline_native_df,
    "baseline_core_signal_recomputed",
    "core",
)

native_comparison_rows.append({
    "model_name": "excel_30d_vrp_baseline",
    "evaluation": "native_baseline_universe",
    **baseline_native_metrics,
})

for model_name in model_names:
    eval_df = rescored_signal_panel_long.loc[
        rescored_signal_panel_long["model_name"].eq(model_name)
    ].copy()

    model_native_df = eval_df.loc[eval_df["model_signal_eligible"]].copy()

    # Native model universe.
    model_native_metrics = summarize_selected_trades(
        model_native_df,
        "model_core_signal_same_thresholds",
        "core",
    )

    native_comparison_rows.append({
        "model_name": model_name,
        "evaluation": "native_model_universe",
        **model_native_metrics,
    })

    # Common universe for this model.
    common_df = eval_df.loc[eval_df["common_signal_eligible"]].copy()

    baseline_common_metrics = summarize_selected_trades(
        common_df,
        "baseline_core_signal_recomputed",
        "baseline",
    )

    model_common_metrics = summarize_selected_trades(
        common_df,
        "model_core_signal_same_thresholds",
        "model",
    )

    common_2025 = common_df.loc[common_df["date"].dt.year.eq(2025)].copy()

    baseline_2025_metrics = summarize_selected_trades(
        common_2025,
        "baseline_core_signal_recomputed",
        "baseline_2025",
    )

    model_2025_metrics = summarize_selected_trades(
        common_2025,
        "model_core_signal_same_thresholds",
        "model_2025",
    )

    row = {
        "model_name": model_name,
        "evaluation": "common_universe",
        "common_eligible_rows": int(len(common_df)),
        **baseline_common_metrics,
        **model_common_metrics,
        **baseline_2025_metrics,
        **model_2025_metrics,
    }

    row["delta_trades"] = row["model_trades"] - row["baseline_trades"]
    row["delta_frequency"] = row["model_frequency"] - row["baseline_frequency"]
    row["delta_win_rate"] = row["model_win_rate"] - row["baseline_win_rate"]
    row["delta_total_pnl"] = row["model_total_pnl"] - row["baseline_total_pnl"]
    row["delta_avg_pnl_per_day"] = row["model_avg_pnl_per_day"] - row["baseline_avg_pnl_per_day"]
    row["delta_worst_trade"] = row["model_worst_trade"] - row["baseline_worst_trade"]
    row["delta_2025_total_pnl"] = row["model_2025_total_pnl"] - row["baseline_2025_total_pnl"]
    row["delta_2025_worst_trade"] = row["model_2025_worst_trade"] - row["baseline_2025_worst_trade"]

    row["beats_trade_count"] = bool(row["model_trades"] > row["baseline_trades"])
    row["beats_frequency"] = bool(row["model_frequency"] > row["baseline_frequency"])
    row["beats_win_rate"] = bool(row["model_win_rate"] >= row["baseline_win_rate"])
    row["passes_worst_trade"] = bool(row["model_worst_trade"] >= row["baseline_worst_trade"])
    row["passes_2025_total_pnl"] = bool(row["model_2025_total_pnl"] >= row["baseline_2025_total_pnl"])
    row["passes_2025_worst_trade"] = bool(row["model_2025_worst_trade"] >= row["baseline_2025_worst_trade"])

    row["passes_large_loss_25k"] = bool(row["model_loss_count_le_25000"] <= row["baseline_loss_count_le_25000"])
    row["passes_large_loss_50k"] = bool(row["model_loss_count_le_50000"] <= row["baseline_loss_count_le_50000"])
    row["passes_large_loss_100k"] = bool(row["model_loss_count_le_100000"] <= row["baseline_loss_count_le_100000"])

    row["passes_primary_and_tail_common"] = bool(
        row["beats_trade_count"]
        and row["beats_frequency"]
        and row["beats_win_rate"]
        and row["passes_worst_trade"]
        and row["passes_2025_total_pnl"]
        and row["passes_2025_worst_trade"]
        and row["passes_large_loss_25k"]
        and row["passes_large_loss_50k"]
        and row["passes_large_loss_100k"]
    )

    common_comparison_rows.append(row)

    # Common year summaries.
    common_year_rows.append(
        summarize_year(
            common_df,
            "baseline_core_signal_recomputed",
            model_name=model_name,
            evaluated_model="baseline_on_common",
            universe="common",
        )
    )

    common_year_rows.append(
        summarize_year(
            common_df,
            "model_core_signal_same_thresholds",
            model_name=model_name,
            evaluated_model="model_on_common",
            universe="common",
        )
    )

    # Overlap.
    baseline_selected_dates = set(
        common_df.loc[common_df["baseline_core_signal_recomputed"], "trade_date"].astype(int).tolist()
    )

    model_selected_dates = set(
        common_df.loc[common_df["model_core_signal_same_thresholds"], "trade_date"].astype(int).tolist()
    )

    union_dates = baseline_selected_dates.union(model_selected_dates)

    overlap_rows.append({
        "model_name": model_name,
        "common_eligible_rows": int(len(common_df)),
        "baseline_selected_count": len(baseline_selected_dates),
        "model_selected_count": len(model_selected_dates),
        "both_selected_count": len(baseline_selected_dates.intersection(model_selected_dates)),
        "model_only_count": len(model_selected_dates.difference(baseline_selected_dates)),
        "baseline_only_count": len(baseline_selected_dates.difference(model_selected_dates)),
        "jaccard_overlap": (
            len(baseline_selected_dates.intersection(model_selected_dates)) / len(union_dates)
            if len(union_dates) else np.nan
        ),
    })

    # Worst trades on common universe.
    for evaluated_model, signal_col in [
        ("baseline_on_common", "baseline_core_signal_recomputed"),
        ("model_on_common", "model_core_signal_same_thresholds"),
    ]:
        selected = common_df.loc[common_df[signal_col]].sort_values("outcome_value", ascending=True).head(20).copy()
        selected["model_name"] = model_name
        selected["evaluated_model"] = evaluated_model
        selected["rank_worst"] = np.arange(1, len(selected) + 1)
        worst_20_rows.append(selected)

native_comparison = pd.DataFrame(native_comparison_rows)
common_comparison = pd.DataFrame(common_comparison_rows)
common_year_summary = pd.concat(common_year_rows, ignore_index=True) if common_year_rows else pd.DataFrame()
selected_overlap_common = pd.DataFrame(overlap_rows)
worst_20_common = pd.concat(worst_20_rows, ignore_index=True) if worst_20_rows else pd.DataFrame()


# =============================================================================
# 6. Validation
# =============================================================================

validation_rows = []

add_validation(
    validation_rows,
    "Baseline all-daily rows loaded",
    len(baseline_signal) > 0,
    f"rows={len(baseline_signal)}; first={baseline_signal['date'].min()}; last={baseline_signal['date'].max()}",
)

add_validation(
    validation_rows,
    "Saved forecast panel loaded",
    len(forecast_panel_long) > 0,
    f"rows={len(forecast_panel_long)}; models={model_names}",
)

add_validation(
    validation_rows,
    "Baseline Core recomputation matches stored Core signal",
    baseline_core_diff_count == 0,
    f"diff_count={baseline_core_diff_count}",
)

add_validation(
    validation_rows,
    "Full-series z-score repair restored 2021 eligibility",
    z_availability_by_year.loc[
        z_availability_by_year["year"].eq(2021),
        "model_signal_eligible_rows",
    ].min() > 200,
    z_availability_by_year.loc[
        z_availability_by_year["year"].eq(2021),
        ["model_name", "model_signal_eligible_rows", "common_signal_eligible_rows"],
    ].to_dict("records"),
)

add_validation(
    validation_rows,
    "Candidate common universes align with baseline eligible rows",
    common_comparison["common_eligible_rows"].min() >= 1300,
    common_comparison[["model_name", "common_eligible_rows"]].to_dict("records"),
)

add_validation(
    validation_rows,
    "No non-positive forecast variances in rescored signal panel",
    rescored_signal_panel_long["forecast_variance_30d"].dropna().gt(0).all(),
    f"min_forecast_variance={rescored_signal_panel_long['forecast_variance_30d'].dropna().min()}",
)

numeric_df = rescored_signal_panel_long.select_dtypes(include=[np.number])
inf_count = int(np.isinf(numeric_df.to_numpy(dtype=float, copy=True)).sum())

add_validation(
    validation_rows,
    "No infinite numeric values in rescored signal panel",
    inf_count == 0,
    f"inf_count={inf_count}",
)

validation = pd.DataFrame(validation_rows)

print()
print("Cell 3A validation checks:")
display(validation)

if validation["status"].eq("FAIL").any():
    print()
    print("FAILED VALIDATION CHECKS:")
    display(validation.loc[validation["status"].eq("FAIL")])
    raise ValueError("Cell 3A validation failed.")


# =============================================================================
# 7. Display outputs
# =============================================================================

print()
print("Native comparison after full-series z-score repair:")
display(native_comparison)

print()
print("Common-universe comparison after full-series z-score repair:")
main_cols = [
    "model_name",
    "common_eligible_rows",
    "baseline_trades",
    "model_trades",
    "delta_trades",
    "baseline_frequency",
    "model_frequency",
    "delta_frequency",
    "baseline_win_rate",
    "model_win_rate",
    "delta_win_rate",
    "baseline_total_pnl",
    "model_total_pnl",
    "delta_total_pnl",
    "baseline_avg_pnl_per_day",
    "model_avg_pnl_per_day",
    "delta_avg_pnl_per_day",
    "baseline_worst_trade",
    "model_worst_trade",
    "delta_worst_trade",
    "baseline_loss_count_le_25000",
    "model_loss_count_le_25000",
    "baseline_loss_count_le_50000",
    "model_loss_count_le_50000",
    "baseline_loss_count_le_100000",
    "model_loss_count_le_100000",
    "baseline_2025_trades",
    "model_2025_trades",
    "baseline_2025_win_rate",
    "model_2025_win_rate",
    "baseline_2025_total_pnl",
    "model_2025_total_pnl",
    "delta_2025_total_pnl",
    "baseline_2025_worst_trade",
    "model_2025_worst_trade",
    "delta_2025_worst_trade",
    "beats_trade_count",
    "beats_frequency",
    "beats_win_rate",
    "passes_worst_trade",
    "passes_2025_total_pnl",
    "passes_2025_worst_trade",
    "passes_large_loss_25k",
    "passes_large_loss_50k",
    "passes_large_loss_100k",
    "passes_primary_and_tail_common",
]

main_cols = [c for c in main_cols if c in common_comparison.columns]
display(common_comparison[main_cols])

print()
print("Models passing common-universe primary + tail criteria:")
display(common_comparison.loc[common_comparison["passes_primary_and_tail_common"]].copy())

print()
print("Common-universe year summary:")
display(common_year_summary)

print()
print("Common-universe selected-trade overlap:")
display(selected_overlap_common)

print()
print("Worst 20 selected trades on common universe:")
worst_cols = [
    "model_name",
    "evaluated_model",
    "rank_worst",
    "date",
    "trade_date",
    "vix_30d_vol_pct",
    "forecast_vol_30d_pct",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "rsi14",
    "outcome_value",
    "actual_dte",
    "pnl_per_day",
]
worst_cols = [c for c in worst_cols if c in worst_20_common.columns]
display(worst_20_common[worst_cols])


# =============================================================================
# 8. Save outputs
# =============================================================================

safe_start = baseline_signal["date"].min().strftime("%Y%m%d")
safe_end = baseline_signal["date"].max().strftime("%Y%m%d")

rescored_signal_panel_path = (
    CORSI_PROCESSED_DIR
    / f"03A_30d_corsi_full_series_rescored_signal_panel_long_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.parquet"
)

native_comparison_path = (
    CORSI_AUDIT_DIR
    / f"03A_30d_corsi_native_comparison_full_series_z_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

common_comparison_path = (
    CORSI_AUDIT_DIR
    / f"03A_30d_corsi_common_universe_comparison_full_series_z_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

common_year_summary_path = (
    CORSI_AUDIT_DIR
    / f"03A_30d_corsi_common_universe_year_summary_full_series_z_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

selected_overlap_path = (
    CORSI_AUDIT_DIR
    / f"03A_30d_corsi_common_universe_selected_overlap_full_series_z_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

worst_20_path = (
    CORSI_AUDIT_DIR
    / f"03A_30d_corsi_common_universe_worst_20_full_series_z_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

z_availability_path = (
    CORSI_AUDIT_DIR
    / f"03A_30d_corsi_z_availability_full_series_z_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

z_availability_by_year_path = (
    CORSI_AUDIT_DIR
    / f"03A_30d_corsi_z_availability_by_year_full_series_z_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

validation_path = (
    CORSI_AUDIT_DIR
    / f"03A_30d_corsi_rescore_validation_full_series_z_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

rescored_signal_panel_long.to_parquet(rescored_signal_panel_path, index=False)
native_comparison.to_csv(native_comparison_path, index=False)
common_comparison.to_csv(common_comparison_path, index=False)
common_year_summary.to_csv(common_year_summary_path, index=False)
selected_overlap_common.to_csv(selected_overlap_path, index=False)
worst_20_common.to_csv(worst_20_path, index=False)
z_availability.to_csv(z_availability_path, index=False)
z_availability_by_year.to_csv(z_availability_by_year_path, index=False)
validation.to_csv(validation_path, index=False)

print()
print("Cell 3A outputs saved:")
print(f"Full-series rescored signal panel:      {rescored_signal_panel_path}")
print(f"Native comparison:                      {native_comparison_path}")
print(f"Common-universe comparison:             {common_comparison_path}")
print(f"Common-universe year summary:           {common_year_summary_path}")
print(f"Common-universe selected overlap:       {selected_overlap_path}")
print(f"Common-universe worst 20:               {worst_20_path}")
print(f"Z availability:                         {z_availability_path}")
print(f"Z availability by year:                 {z_availability_by_year_path}")
print(f"Validation:                             {validation_path}")

print()
print("Cell 3A complete.")
print()
print("Next decision:")
print("Use the common-universe comparison, not the old Cell 3 comparison, to decide whether har_downside_v1 still passes.")

Cell 3A: Repair Corsi same-threshold evaluation
Project root:        C:\Users\patri\vrp_project
Baseline processed:  C:\Users\patri\vrp_project\data\processed\vrp_30d_baseline_v1
Corsi processed:     C:\Users\patri\vrp_project\data\processed\vrp_30d_corsi_v1
Corsi audit:         C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1
Run timestamp:       20260704_141017

Locked Cell 3A scope:


,model_refit,threshold_optimization,zscore_series,zscore_convention,z3m_window,z1y_window,comparison,forecast_vol_min,z3m_min,z1y_min,rsi14_max,vrp_log_min
0,NO,NO,full_daily_30d_signal_panel,prior_only_shift1,63,252,baseline_vs_model_on_common_eligible_universe,8.5,0.7,0.7,70,0.7



Inputs loaded:


,role,path,rows
0,baseline_signal_all_daily_rows,C:\Users\patri\vrp_project\data\processed\vrp_...,1612
1,saved_oos_forecast_panel_long,C:\Users\patri\vrp_project\data\processed\vrp_...,12096
2,old_cell3_model_comparison,C:\Users\patri\vrp_project\data\audit\vrp_30d_...,7



Models to rescore:


...


Common-universe selected-trade overlap:


,model_name,common_eligible_rows,baseline_selected_count,model_selected_count,both_selected_count,model_only_count,baseline_only_count,jaccard_overlap
0,har_downside_v1,1354,177,156,54,102,123,0.193548
1,har_full_defensible_v1,1354,177,173,48,125,129,0.158940
2,har_regime_v1,1354,177,153,48,105,129,0.170213
3,har_total_accel_v1,1354,177,187,70,117,107,0.238095
4,har_total_mean_v1,1354,177,181,73,108,104,0.256140
5,har_total_std_v1,1354,177,168,72,96,105,0.263736



Worst 20 selected trades on common universe:


,model_name,evaluated_model,rank_worst,date,trade_date,vix_30d_vol_pct,forecast_vol_30d_pct,model_vrp_log,model_vrp_z_3m,model_vrp_z_1y,rsi14,outcome_value,actual_dte,pnl_per_day
0,har_downside_v1,baseline_on_common,1,2025-03-03,20250303,22.808456,17.245166,0.559199,1.336892,0.822743,37.160997,-112316.486943,32.0,-3509.890217
1,har_downside_v1,baseline_on_common,2,2025-03-04,20250304,23.834459,17.496670,0.618244,1.655521,1.028975,33.223807,-100779.661310,31.0,-3250.956816
2,har_downside_v1,baseline_on_common,3,2025-03-06,20250306,25.083189,18.366674,0.623320,1.594187,1.036526,33.946232,-92588.332880,29.0,-3192.701134
3,har_downside_v1,baseline_on_common,4,2025-02-24,20250224,19.078121,15.121199,0.464889,0.822799,0.512896,43.422979,-51336.648143,32.0,-1604.270254
4,har_downside_v1,baseline_on_common,5,2025-02-26,20250226,19.083176,13.603360,0.676981,1.722068,1.266290,41.127776,-47172.123854,30.0,-1572.404128
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,har_total_std_v1,model_on_common,16,2026-03-05,20260305,23.309121,13.663333,1.068258,2.033460,1.702049,45.000430,-17217.829479,28.0,-614.922481
236,har_total_std_v1,model_on_common,17,2022-01-19,20220119,23.826348,15.658637,0.839539,0.873827,0.826791,34.918952,-16751.383263,30.0,-558.379442
237,har_total_std_v1,model_on_common,18,2026-03-03,20260303,23.459841,13.812975,1.059363,2.156190,1.694925,43.035216,-13409.852082,30.0,-446.995069
238,har_total_std_v1,model_on_common,19,2026-02-13,20260213,20.594557,13.852792,0.793080,0.887094,0.823469,43.580380,-12449.953702,28.0,-444.641204



Cell 3A outputs saved:
Full-series rescored signal panel:      C:\Users\patri\vrp_project\data\processed\vrp_30d_corsi_v1\03A_30d_corsi_full_series_rescored_signal_panel_long_20200102_20260602_20260704_141017.parquet
Native comparison:                      C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\03A_30d_corsi_native_comparison_full_series_z_20200102_20260602_20260704_141017.csv
Common-universe comparison:             C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\03A_30d_corsi_common_universe_comparison_full_series_z_20200102_20260602_20260704_141017.csv
Common-universe year summary:           C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\03A_30d_corsi_common_universe_year_summary_full_series_z_20200102_20260602_20260704_141017.csv
Common-universe selected overlap:       C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\03A_30d_corsi_common_universe_selected_overlap_full_series_z_20200102_20260602_20260704_141017.csv
Common-universe worst 20:      

In [7]:
# Cell 4: Lock-down package for har_downside_v1 as 30D Core VRP denominator
#
# Objective:
#   Decide whether har_downside_v1 can be locked as the new 30D Core VRP denominator.
#
# Scope:
#   - No model refitting
#   - No threshold optimization
#   - No new model families
#   - No cross-tenor extension
#   - No Secondary signal
#   - No sizing
#
# Lock-down standard:
#   Core frequency may be modestly lower than baseline if quality improves materially.
#
# Required to lock:
#   - Retain at least 85% of baseline Core trade count
#   - Win rate > baseline
#   - Profit factor > baseline
#   - Total P&L > baseline
#   - Worst trade no worse than baseline
#   - Selected-trade drawdown no worse than baseline
#   - Large-loss counts no worse than baseline
#   - 2025 total P&L materially improved
#   - 2025 worst trade no worse than baseline
#   - 2025 large-loss counts no worse than baseline
#
# If passed:
#   Lock har_downside_v1 as:
#       30D Core VRP denominator v1
#       forecast_variance_30d from har_downside_v1
#
# VRP formula:
#   model_vrp_log = log(implied_variance_30d / forecast_variance_30d)

from pathlib import Path
import json
import numpy as np
import pandas as pd


# =============================================================================
# 0. Paths and constants
# =============================================================================

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

CORSI_BRANCH = "vrp_30d_corsi_v1"
CORSI_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / CORSI_BRANCH
CORSI_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / CORSI_BRANCH

CORSI_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CORSI_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

LOCK_MODEL_NAME = "har_downside_v1"

MIN_TRADE_COUNT_RETENTION = 0.85
LARGE_LOSS_LEVELS = [-25000, -50000, -100000]

LOCKED_FEATURE_SET = [
    "log_rv_1d",
    "log_rv_std_5d",
    "log_rv_std_21d",
    "log_rv_std_63d",
    "log_downside_rv_5d",
    "log_downside_rv_21d",
    "log_downside_rv_63d",
    "downside_share_21d",
    "negative_return_count_21d",
]

LOCKED_SIGNAL_THRESHOLDS = {
    "forecast_vol_30d_pct_min": 8.5,
    "model_vrp_z_3m_min": 0.7,
    "model_vrp_z_1y_min": 0.7,
    "rsi14_max": 70,
    "model_vrp_log_min": 0.7,
}

print("Cell 4: Lock-down package for har_downside_v1")
print(f"Project root:       {PROJECT_ROOT}")
print(f"Corsi processed:    {CORSI_PROCESSED_DIR}")
print(f"Corsi audit:        {CORSI_AUDIT_DIR}")
print(f"Run timestamp:      {RUN_TIMESTAMP}")
print(f"Lock model:         {LOCK_MODEL_NAME}")


# =============================================================================
# 1. Helpers
# =============================================================================

def latest_file(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f"No files found in {directory} matching pattern: {pattern}")
    return matches[0]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def pass_fail(condition) -> str:
    return "PASS" if bool(condition) else "FAIL"


def add_lock_check(rows, check_name, passed, baseline_value, model_value, requirement, notes=""):
    rows.append({
        "check": check_name,
        "status": pass_fail(passed),
        "baseline_value": baseline_value,
        "model_value": model_value,
        "requirement": requirement,
        "notes": notes,
    })


def safe_get(row, col):
    if col not in row.index:
        return np.nan
    return row[col]


def classify_trade(row):
    baseline = bool(row.get("baseline_core_signal_recomputed", False))
    model = bool(row.get("model_core_signal_same_thresholds", False))

    if baseline and model:
        return "both"
    if baseline and not model:
        return "baseline_only"
    if model and not baseline:
        return "model_only"
    return "neither"


# =============================================================================
# 2. Load Cell 3A outputs
# =============================================================================

common_comparison_path = latest_file(
    CORSI_AUDIT_DIR,
    "03A_30d_corsi_common_universe_comparison_full_series_z_*.csv",
)

native_comparison_path = latest_file(
    CORSI_AUDIT_DIR,
    "03A_30d_corsi_native_comparison_full_series_z_*.csv",
)

common_year_summary_path = latest_file(
    CORSI_AUDIT_DIR,
    "03A_30d_corsi_common_universe_year_summary_full_series_z_*.csv",
)

worst_20_path = latest_file(
    CORSI_AUDIT_DIR,
    "03A_30d_corsi_common_universe_worst_20_full_series_z_*.csv",
)

selected_overlap_path = latest_file(
    CORSI_AUDIT_DIR,
    "03A_30d_corsi_common_universe_selected_overlap_full_series_z_*.csv",
)

z_availability_path = latest_file(
    CORSI_AUDIT_DIR,
    "03A_30d_corsi_z_availability_full_series_z_*.csv",
)

rescored_signal_panel_path = latest_file(
    CORSI_PROCESSED_DIR,
    "03A_30d_corsi_full_series_rescored_signal_panel_long_*.parquet",
)

common_comparison = normalize_columns(pd.read_csv(common_comparison_path))
native_comparison = normalize_columns(pd.read_csv(native_comparison_path))
common_year_summary = normalize_columns(pd.read_csv(common_year_summary_path))
worst_20 = normalize_columns(pd.read_csv(worst_20_path))
selected_overlap = normalize_columns(pd.read_csv(selected_overlap_path))
z_availability = normalize_columns(pd.read_csv(z_availability_path))
rescored_signal_panel = normalize_columns(pd.read_parquet(rescored_signal_panel_path))

for df in [common_year_summary, worst_20, rescored_signal_panel]:
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

if "trade_date" in rescored_signal_panel.columns:
    rescored_signal_panel["trade_date"] = pd.to_numeric(
        rescored_signal_panel["trade_date"],
        errors="coerce",
    ).astype("Int64")

print()
print("Cell 4 inputs loaded:")
display(pd.DataFrame([
    {"role": "common_comparison", "path": str(common_comparison_path), "rows": len(common_comparison)},
    {"role": "native_comparison", "path": str(native_comparison_path), "rows": len(native_comparison)},
    {"role": "common_year_summary", "path": str(common_year_summary_path), "rows": len(common_year_summary)},
    {"role": "worst_20", "path": str(worst_20_path), "rows": len(worst_20)},
    {"role": "selected_overlap", "path": str(selected_overlap_path), "rows": len(selected_overlap)},
    {"role": "z_availability", "path": str(z_availability_path), "rows": len(z_availability)},
    {"role": "rescored_signal_panel", "path": str(rescored_signal_panel_path), "rows": len(rescored_signal_panel)},
]))


# =============================================================================
# 3. Extract locked model row
# =============================================================================

lock_rows = common_comparison.loc[common_comparison["model_name"].eq(LOCK_MODEL_NAME)].copy()

if len(lock_rows) != 1:
    raise ValueError(f"Expected exactly one common-comparison row for {LOCK_MODEL_NAME}, found {len(lock_rows)}.")

lock_row = lock_rows.iloc[0]

baseline_trades = safe_get(lock_row, "baseline_trades")
model_trades = safe_get(lock_row, "model_trades")
trade_count_retention = model_trades / baseline_trades if baseline_trades else np.nan

baseline_win_rate = safe_get(lock_row, "baseline_win_rate")
model_win_rate = safe_get(lock_row, "model_win_rate")

baseline_profit_factor = safe_get(lock_row, "baseline_profit_factor")
model_profit_factor = safe_get(lock_row, "model_profit_factor")

baseline_total_pnl = safe_get(lock_row, "baseline_total_pnl")
model_total_pnl = safe_get(lock_row, "model_total_pnl")

baseline_worst_trade = safe_get(lock_row, "baseline_worst_trade")
model_worst_trade = safe_get(lock_row, "model_worst_trade")

baseline_dd = safe_get(lock_row, "baseline_selected_trade_drawdown")
model_dd = safe_get(lock_row, "model_selected_trade_drawdown")

baseline_2025_total_pnl = safe_get(lock_row, "baseline_2025_total_pnl")
model_2025_total_pnl = safe_get(lock_row, "model_2025_total_pnl")

baseline_2025_worst_trade = safe_get(lock_row, "baseline_2025_worst_trade")
model_2025_worst_trade = safe_get(lock_row, "model_2025_worst_trade")

baseline_2025_win_rate = safe_get(lock_row, "baseline_2025_win_rate")
model_2025_win_rate = safe_get(lock_row, "model_2025_win_rate")


# =============================================================================
# 4. Lock checks
# =============================================================================

lock_check_rows = []

add_lock_check(
    lock_check_rows,
    "Trade-count retention",
    trade_count_retention >= MIN_TRADE_COUNT_RETENTION,
    baseline_trades,
    model_trades,
    f"model_trades / baseline_trades >= {MIN_TRADE_COUNT_RETENTION:.0%}",
    f"retention={trade_count_retention:.2%}",
)

add_lock_check(
    lock_check_rows,
    "Win rate improves",
    model_win_rate > baseline_win_rate,
    baseline_win_rate,
    model_win_rate,
    "model_win_rate > baseline_win_rate",
)

add_lock_check(
    lock_check_rows,
    "Profit factor improves",
    model_profit_factor > baseline_profit_factor,
    baseline_profit_factor,
    model_profit_factor,
    "model_profit_factor > baseline_profit_factor",
)

add_lock_check(
    lock_check_rows,
    "Total P&L improves",
    model_total_pnl > baseline_total_pnl,
    baseline_total_pnl,
    model_total_pnl,
    "model_total_pnl > baseline_total_pnl",
)

add_lock_check(
    lock_check_rows,
    "Worst trade no worse",
    model_worst_trade >= baseline_worst_trade,
    baseline_worst_trade,
    model_worst_trade,
    "model_worst_trade >= baseline_worst_trade",
    "less negative is better",
)

add_lock_check(
    lock_check_rows,
    "Selected-trade drawdown no worse",
    model_dd >= baseline_dd,
    baseline_dd,
    model_dd,
    "model_selected_trade_drawdown >= baseline_selected_trade_drawdown",
    "less negative is better",
)

for level in LARGE_LOSS_LEVELS:
    suffix = f"{abs(level):.0f}"
    baseline_col = f"baseline_loss_count_le_{suffix}"
    model_col = f"model_loss_count_le_{suffix}"

    baseline_count = safe_get(lock_row, baseline_col)
    model_count = safe_get(lock_row, model_col)

    add_lock_check(
        lock_check_rows,
        f"Large-loss count <= -{suffix}",
        model_count <= baseline_count,
        baseline_count,
        model_count,
        f"{model_col} <= {baseline_col}",
    )

add_lock_check(
    lock_check_rows,
    "2025 total P&L improves",
    model_2025_total_pnl > baseline_2025_total_pnl,
    baseline_2025_total_pnl,
    model_2025_total_pnl,
    "model_2025_total_pnl > baseline_2025_total_pnl",
)

add_lock_check(
    lock_check_rows,
    "2025 win rate no worse",
    model_2025_win_rate >= baseline_2025_win_rate,
    baseline_2025_win_rate,
    model_2025_win_rate,
    "model_2025_win_rate >= baseline_2025_win_rate",
)

add_lock_check(
    lock_check_rows,
    "2025 worst trade no worse",
    model_2025_worst_trade >= baseline_2025_worst_trade,
    baseline_2025_worst_trade,
    model_2025_worst_trade,
    "model_2025_worst_trade >= baseline_2025_worst_trade",
    "less negative is better",
)

for level in LARGE_LOSS_LEVELS:
    suffix = f"{abs(level):.0f}"
    baseline_col = f"baseline_2025_loss_count_le_{suffix}"
    model_col = f"model_2025_loss_count_le_{suffix}"

    if baseline_col in common_comparison.columns and model_col in common_comparison.columns:
        baseline_count = safe_get(lock_row, baseline_col)
        model_count = safe_get(lock_row, model_col)

        add_lock_check(
            lock_check_rows,
            f"2025 large-loss count <= -{suffix}",
            model_count <= baseline_count,
            baseline_count,
            model_count,
            f"{model_col} <= {baseline_col}",
        )

lock_checks = pd.DataFrame(lock_check_rows)

lock_decision = "LOCK" if lock_checks["status"].eq("PASS").all() else "DO_NOT_LOCK"

lock_decision_summary = pd.DataFrame([{
    "model_name": LOCK_MODEL_NAME,
    "decision": lock_decision,
    "locked_use_case": "30D Core VRP denominator" if lock_decision == "LOCK" else "not_locked",
    "frequency_policy": "modestly lower Core frequency accepted if quality/tail materially improve",
    "trade_count_retention": trade_count_retention,
    "baseline_trades": baseline_trades,
    "model_trades": model_trades,
    "baseline_frequency": safe_get(lock_row, "baseline_frequency"),
    "model_frequency": safe_get(lock_row, "model_frequency"),
    "baseline_win_rate": baseline_win_rate,
    "model_win_rate": model_win_rate,
    "baseline_profit_factor": baseline_profit_factor,
    "model_profit_factor": model_profit_factor,
    "baseline_total_pnl": baseline_total_pnl,
    "model_total_pnl": model_total_pnl,
    "baseline_worst_trade": baseline_worst_trade,
    "model_worst_trade": model_worst_trade,
    "baseline_selected_trade_drawdown": baseline_dd,
    "model_selected_trade_drawdown": model_dd,
    "baseline_2025_total_pnl": baseline_2025_total_pnl,
    "model_2025_total_pnl": model_2025_total_pnl,
    "baseline_2025_worst_trade": baseline_2025_worst_trade,
    "model_2025_worst_trade": model_2025_worst_trade,
}])


# =============================================================================
# 5. Freeze model specification
# =============================================================================

locked_model_spec = {
    "model_name": LOCK_MODEL_NAME,
    "locked_use_case": "30D Core VRP denominator",
    "decision": lock_decision,
    "target": "log_forward_rv_21td_variance_decimal",
    "target_definition": "(STDEV.S(next 21 trading-day close-to-close log returns) * sqrt(252))^2, then log variance target for regression",
    "forecast_output": "forecast_variance_30d",
    "forecast_vol_output": "forecast_vol_30d_pct",
    "vrp_formula": "model_vrp_log = log(implied_variance_decimal / forecast_variance_30d)",
    "zscore_convention": {
        "type": "prior_only",
        "three_month_window": 63,
        "one_year_window": 252,
        "ddof": 1,
    },
    "features": LOCKED_FEATURE_SET,
    "model_fitting": {
        "estimator": "StandardScaler + Ridge",
        "ridge_alpha_selection": "time-series CV within each expanding training set",
        "walk_forward": "expanding yearly walk-forward",
        "embargo_rule": "training rows require target_end_date before test-year start",
        "forbidden_inputs": [
            "implied variance",
            "VIX-style volatility",
            "VRP",
            "RSI",
            "option outcome",
            "P&L",
            "actual DTE",
            "future returns beyond target construction",
        ],
    },
    "signal_thresholds": LOCKED_SIGNAL_THRESHOLDS,
    "acceptance_policy": {
        "frequency": f"model trade count must retain at least {MIN_TRADE_COUNT_RETENTION:.0%} of baseline Core trade count",
        "quality": "win rate, profit factor, total P&L, worst trade, drawdown, large-loss count, and 2025 behavior must improve or be no worse",
    },
}

locked_feature_set = pd.DataFrame([
    {
        "model_name": LOCK_MODEL_NAME,
        "feature_order": i + 1,
        "feature": feature,
    }
    for i, feature in enumerate(LOCKED_FEATURE_SET)
])

locked_thresholds = pd.DataFrame([
    {"threshold": k, "value": v}
    for k, v in LOCKED_SIGNAL_THRESHOLDS.items()
])


# =============================================================================
# 6. Build final selected-trade review files
# =============================================================================

lock_panel = rescored_signal_panel.loc[
    rescored_signal_panel["model_name"].eq(LOCK_MODEL_NAME)
].copy()

lock_panel = lock_panel.loc[lock_panel["common_signal_eligible"]].copy()
lock_panel["trade_classification"] = lock_panel.apply(classify_trade, axis=1)

selected_trade_review_cols = [
    "date",
    "trade_date",
    "trade_classification",
    "vix_30d_vol_pct",
    "rv21d_vol_pct",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "forecast_vol_30d_pct",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "rsi14",
    "baseline_core_signal_recomputed",
    "model_core_signal_same_thresholds",
    "outcome_value",
    "actual_dte",
    "pnl_per_day",
]

selected_trade_review_cols = [c for c in selected_trade_review_cols if c in lock_panel.columns]

selected_trade_review = lock_panel.loc[
    lock_panel["trade_classification"].isin(["baseline_only", "model_only", "both"]),
    selected_trade_review_cols,
].sort_values(["date", "trade_classification"]).copy()

trade_class_summary = (
    selected_trade_review
    .groupby("trade_classification", dropna=False)
    .agg(
        trades=("outcome_value", "size"),
        win_rate=("outcome_value", lambda s: float((s > 0).mean()) if len(s) else np.nan),
        total_pnl=("outcome_value", "sum"),
        avg_pnl=("outcome_value", "mean"),
        worst_trade=("outcome_value", "min"),
        best_trade=("outcome_value", "max"),
        avg_pnl_per_day=("pnl_per_day", "mean"),
    )
    .reset_index()
)

# 2025 cluster confirmation: baseline-selected 2025 losses, with model status.
cluster_2025 = lock_panel.loc[
    (lock_panel["date"].dt.year.eq(2025))
    & lock_panel["baseline_core_signal_recomputed"]
].copy()

cluster_2025 = cluster_2025[selected_trade_review_cols].sort_values("outcome_value").copy()

# Worst 20 for locked model and corresponding baseline-on-common.
locked_worst_20 = worst_20.loc[worst_20["model_name"].eq(LOCK_MODEL_NAME)].copy()


# =============================================================================
# 7. Display lock-down result
# =============================================================================

print()
print("=" * 100)
print("LOCK DECISION")
print("=" * 100)
display(lock_decision_summary.T)

print()
print("Lock checks:")
display(lock_checks)

print()
print("Locked denominator specification:")
display(pd.DataFrame([
    {
        "field": "model_name",
        "value": locked_model_spec["model_name"],
    },
    {
        "field": "decision",
        "value": locked_model_spec["decision"],
    },
    {
        "field": "use_case",
        "value": locked_model_spec["locked_use_case"],
    },
    {
        "field": "target",
        "value": locked_model_spec["target"],
    },
    {
        "field": "vrp_formula",
        "value": locked_model_spec["vrp_formula"],
    },
    {
        "field": "model_fitting",
        "value": locked_model_spec["model_fitting"]["estimator"],
    },
]))

print()
print("Locked feature set:")
display(locked_feature_set)

print()
print("Locked signal thresholds:")
display(locked_thresholds)

print()
print("Trade classification summary:")
display(trade_class_summary)

print()
print("2025 baseline-selected cluster under locked model:")
display(cluster_2025)

print()
print("Worst 20 selected trades for locked model / baseline-on-common:")
display(locked_worst_20)


# =============================================================================
# 8. Save outputs
# =============================================================================

safe_start = lock_panel["date"].min().strftime("%Y%m%d")
safe_end = lock_panel["date"].max().strftime("%Y%m%d")

lock_decision_path = (
    CORSI_AUDIT_DIR
    / f"04_30d_core_har_downside_lock_decision_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

lock_checks_path = (
    CORSI_AUDIT_DIR
    / f"04_30d_core_har_downside_lock_checks_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

locked_model_spec_json_path = (
    CORSI_AUDIT_DIR
    / f"04_30d_core_har_downside_locked_model_spec_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.json"
)

locked_feature_set_path = (
    CORSI_AUDIT_DIR
    / f"04_30d_core_har_downside_locked_feature_set_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

locked_thresholds_path = (
    CORSI_AUDIT_DIR
    / f"04_30d_core_har_downside_locked_thresholds_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

selected_trade_review_path = (
    CORSI_AUDIT_DIR
    / f"04_30d_core_har_downside_selected_trade_review_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

trade_class_summary_path = (
    CORSI_AUDIT_DIR
    / f"04_30d_core_har_downside_trade_class_summary_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

cluster_2025_path = (
    CORSI_AUDIT_DIR
    / f"04_30d_core_har_downside_2025_baseline_cluster_review_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

locked_worst_20_path = (
    CORSI_AUDIT_DIR
    / f"04_30d_core_har_downside_worst_20_review_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.csv"
)

locked_signal_panel_path = (
    CORSI_PROCESSED_DIR
    / f"04_30d_core_har_downside_locked_signal_panel_{safe_start}_{safe_end}_{RUN_TIMESTAMP}.parquet"
)

lock_decision_summary.to_csv(lock_decision_path, index=False)
lock_checks.to_csv(lock_checks_path, index=False)
locked_feature_set.to_csv(locked_feature_set_path, index=False)
locked_thresholds.to_csv(locked_thresholds_path, index=False)
selected_trade_review.to_csv(selected_trade_review_path, index=False)
trade_class_summary.to_csv(trade_class_summary_path, index=False)
cluster_2025.to_csv(cluster_2025_path, index=False)
locked_worst_20.to_csv(locked_worst_20_path, index=False)
lock_panel.to_parquet(locked_signal_panel_path, index=False)

with open(locked_model_spec_json_path, "w") as f:
    json.dump(locked_model_spec, f, indent=4, default=str)

print()
print("Cell 4 outputs saved:")
print(f"Lock decision:                    {lock_decision_path}")
print(f"Lock checks:                      {lock_checks_path}")
print(f"Locked model spec JSON:           {locked_model_spec_json_path}")
print(f"Locked feature set:               {locked_feature_set_path}")
print(f"Locked thresholds:                {locked_thresholds_path}")
print(f"Selected trade review:            {selected_trade_review_path}")
print(f"Trade class summary:              {trade_class_summary_path}")
print(f"2025 cluster review:              {cluster_2025_path}")
print(f"Worst 20 review:                  {locked_worst_20_path}")
print(f"Locked signal panel:              {locked_signal_panel_path}")

print()
print("Cell 4 complete.")

if lock_decision == "LOCK":
    print()
    print("FINAL DECISION: LOCK har_downside_v1 as the 30D Core VRP denominator v1.")
    print("Next phase: extend this locked denominator methodology across tenors, then test Core/Secondary by tenor bucket.")
else:
    print()
    print("FINAL DECISION: DO NOT LOCK yet. Review failed lock checks only; do not restart broad diagnostics.")

Cell 4: Lock-down package for har_downside_v1
Project root:       C:\Users\patri\vrp_project
Corsi processed:    C:\Users\patri\vrp_project\data\processed\vrp_30d_corsi_v1
Corsi audit:        C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1
Run timestamp:      20260704_141958
Lock model:         har_downside_v1

Cell 4 inputs loaded:


,role,path,rows
0,common_comparison,C:\Users\patri\vrp_project\data\audit\vrp_30d_...,6
1,native_comparison,C:\Users\patri\vrp_project\data\audit\vrp_30d_...,7
2,common_year_summary,C:\Users\patri\vrp_project\data\audit\vrp_30d_...,84
3,worst_20,C:\Users\patri\vrp_project\data\audit\vrp_30d_...,240
4,selected_overlap,C:\Users\patri\vrp_project\data\audit\vrp_30d_...,6
5,z_availability,C:\Users\patri\vrp_project\data\audit\vrp_30d_...,6
6,rescored_signal_panel,C:\Users\patri\vrp_project\data\processed\vrp_...,9672



LOCK DECISION


,0
model_name,har_downside_v1
decision,LOCK
locked_use_case,30D Core VRP denominator
frequency_policy,modestly lower Core frequency accepted if qual...
trade_count_retention,0.881356
baseline_trades,177
model_trades,156
baseline_frequency,0.130724
model_frequency,0.115214
baseline_win_rate,0.864407



Lock checks:


...


2025 baseline-selected cluster under locked model:


,date,trade_date,trade_classification,vix_30d_vol_pct,rv21d_vol_pct,vrp_log,vrp_z_3m,vrp_z_1y,forecast_vol_30d_pct,model_vrp_log,model_vrp_z_3m,model_vrp_z_1y,rsi14,baseline_core_signal_recomputed,model_core_signal_same_thresholds,outcome_value,actual_dte,pnl_per_day
1297,2025-03-03,20250303,baseline_only,22.808456,13.932921,0.985754,1.106951,0.810141,17.245166,0.559199,1.336892,0.822743,37.160997,True,False,-112316.486943,32.0,-3509.890217
1298,2025-03-04,20250304,baseline_only,23.834459,14.368250,1.012223,1.132385,0.853232,17.496670,0.618244,1.655521,1.028975,33.223807,True,False,-100779.661310,31.0,-3250.956816
1300,2025-03-06,20250306,baseline_only,25.083189,15.656730,0.942594,0.921200,0.711711,18.366674,0.623320,1.594187,1.036526,33.946232,True,False,-92588.332880,29.0,-3192.701134
1292,2025-02-24,20250224,baseline_only,19.078121,11.772753,0.965509,1.274300,0.806744,15.121199,0.464889,0.822799,0.512896,43.422979,True,False,-51336.648143,32.0,-1604.270254
1294,2025-02-26,20250226,baseline_only,19.083176,10.753467,1.147158,1.624308,1.136551,13.603360,0.676981,1.722068,1.266290,41.127776,True,False,-47172.123854,30.0,-1572.404128
1293,2025-02-25,20250225,baseline_only,19.578612,11.824882,1.008464,1.337134,0.883105,14.947504,0.539787,1.131516,0.779576,41.026556,True,False,-46389.320348,31.0,-1496.429689
1295,2025-02-27,20250227,baseline_only,21.086209,11.428641,1.224993,1.744878,1.273389,15.729443,0.586170,1.270211,0.929833,33.832673,True,False,-30036.321327,29.0,-1035.735218
1391,2025-07-17,20250717,both,16.625538,8.591696,1.320288,1.368210,1.355764,11.521558,0.733440,1.448942,1.163921,69.119757,True,True,12751.375179,29.0,439.702592
1387,2025-07-11,20250711,baseline_only,16.310672,10.001345,0.978200,1.134145,0.847242,13.045232,0.446794,0.102311,0.170149,68.087677,True,False,12875.913575,28.0,459.854056
1377,2025-06-26,20250626,baseline_only,16.428343,9.971038,0.998647,1.443501,0.856140,14.510695,0.248244,-0.738159,-0.515024,68.293518,True,False,13027.151841,29.0,449.212132



Worst 20 selected trades for locked model / baseline-on-common:


,date,trade_date,tenor,vix_30d_vol_pct,implied_variance_decimal,underlying_close,daily_log_return,rv21d_vol_decimal,rv21d_variance_decimal,rv21d_vol_pct,...,train_rows_used,selected_alpha,model_vrp_log,model_vrp_z_3m,model_vrp_z_1y,model_signal_eligible,model_core_signal_same_thresholds,common_signal_eligible,evaluated_model,rank_worst
0,2025-03-03,20250303,30,22.808456,0.052023,5849.72,-0.017753,0.139329,0.019413,13.932921,...,1557.0,100.0,0.559199,1.336892,0.822743,True,False,True,baseline_on_common,1
1,2025-03-04,20250304,30,23.834459,0.056808,5778.15,-0.012310,0.143682,0.020645,14.368250,...,1557.0,100.0,0.618244,1.655521,1.028975,True,False,True,baseline_on_common,2
2,2025-03-06,20250306,30,25.083189,0.062917,5738.52,-0.017980,0.156567,0.024513,15.656730,...,1557.0,100.0,0.623320,1.594187,1.036526,True,False,True,baseline_on_common,3
3,2025-02-24,20250224,30,19.078121,0.036397,5983.25,-0.004982,0.117728,0.013860,11.772753,...,1557.0,100.0,0.464889,0.822799,0.512896,True,False,True,baseline_on_common,4
4,2025-02-26,20250226,30,19.083176,0.036417,5956.06,0.000136,0.107535,0.011564,10.753467,...,1557.0,100.0,0.676981,1.722068,1.266290,True,False,True,baseline_on_common,5
5,2025-02-25,20250225,30,19.578612,0.038332,5955.25,-0.004691,0.118249,0.013983,11.824882,...,1557.0,100.0,0.539787,1.131516,0.779576,True,False,True,baseline_on_common,6
6,2025-02-27,20250227,30,21.086209,0.044463,5861.57,-0.015992,0.114286,0.013061,11.428641,...,1557.0,100.0,0.586170,1.270211,0.929833,True,False,True,baseline_on_common,7
7,2023-07-28,20230728,30,13.306566,0.017706,4582.23,0.009829,0.087086,0.007584,8.708607,...,1055.0,100.0,0.139139,-0.525629,-1.211552,True,False,True,baseline_on_common,8
8,2023-09-28,20230928,30,17.320099,0.029999,4299.70,0.005876,0.109190,0.011922,10.918979,...,1055.0,100.0,0.200316,0.019373,-0.689448,True,False,True,baseline_on_common,9
9,2023-07-21,20230721,30,13.620362,0.018551,4536.34,0.000324,0.093420,0.008727,9.342003,...,1055.0,100.0,0.274711,0.092244,-0.785180,True,False,True,baseline_on_common,10



Cell 4 outputs saved:
Lock decision:                    C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\04_30d_core_har_downside_lock_decision_20201231_20260522_20260704_141958.csv
Lock checks:                      C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\04_30d_core_har_downside_lock_checks_20201231_20260522_20260704_141958.csv
Locked model spec JSON:           C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\04_30d_core_har_downside_locked_model_spec_20201231_20260522_20260704_141958.json
Locked feature set:               C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\04_30d_core_har_downside_locked_feature_set_20201231_20260522_20260704_141958.csv
Locked thresholds:                C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\04_30d_core_har_downside_locked_thresholds_20201231_20260522_20260704_141958.csv
Selected trade review:            C:\Users\patri\vrp_project\data\audit\vrp_30d_corsi_v1\04_30d_core_har_downside_selected_trade_review